# Trabajo de Grado — Sistema inteligente de detección de malware con Machine Learning y XAI
### Notebook maestro (EMBER2024): Fase 1 (datos y selección de características) · Fase 2 (modelos) · Fase 3 (explicabilidad) · Fase 4 (interfaz y pruebas)

**Universidad Católica de Colombia — Programa de Ingeniería de Sistemas y Computación**
Jhon Alexander Parra Olarte · Jhon David González García — Docente: Stefany Tatiana Ospina — 2026
Repositorio del proyecto: https://github.com/jhgar059/repo_proyecto

**Dataset:** EMBER2024 (Joyce et al., KDD 2025) — subconjunto de ejecutables PE de Windows (Win32, Win64 y .NET), descargado desde Hugging Face (`joyce8/EMBER2024`) y vectorizado con el extractor oficial `thrember` (EMBER feature version 3, 2.568 características).

**Cómo usar este notebook (Google Colab):**
1. Correr las celdas **en orden**, de arriba hacia abajo, empezando SIEMPRE por la celda de configuración (0).
2. Cada celda revisa si su resultado ya existe en Google Drive (marcas `.ok` y archivos JSON) y **salta el trabajo ya hecho**; si la sesión se desconecta, basta con volver a correr desde la celda 0.
3. Los resultados para el documento (tablas JSON/CSV y figuras PNG) quedan en `Drive/trabajo_grado_malware/ember2024/figuras_documento/` y en `entregables_documento.zip`; la **evidencia completa de cada paso** (bases de datos reducidas, matrices de confusión, curvas, predicciones por archivo, valores SHAP, pruebas, etc.) queda organizada por fase en `evidencias/` con un manifiesto que describe cada archivo, y empaquetada en `evidencias_proyecto.zip`. La celda de exportación descarga ambos zip al computador.

**Tiempos estimados en Colab gratuito (CPU):** Fase 1a ≈ 1,5–2,5 h (descarga de ~23 GB + vectorización en streaming); Fase 1b ≈ 20–30 min; Fase 2 ≈ 45–90 min; Fase 3 ≈ 10 min; Fase 4 ≈ 5 min + prueba con usuarios.

**Memoria (Colab gratuito ≈ 12,7 GB de RAM):** el notebook está diseñado para no superar ~4 GB en su punto máximo: los zips se leen en streaming y se vectorizan por bloques de 5.000 archivos; la selección de características trabaja sobre una sola matriz float32 preasignada (≈1,5 GB) sin copias intermedias; las fases siguientes leen únicamente las K columnas seleccionadas; la búsqueda de hiperparámetros mantiene un modelo en memoria a la vez; cada fase libera sus variables pesadas al terminar (`liberar(...)`) y registra el consumo en `registro_memoria.csv` (`memoria_ram(...)`).

**Cambios frente a la versión anterior (EMBER2018):** se migró a EMBER2024 (dataset 2023-2024, con conjunto de prueba *temporal* y conjunto de desafío de malware evasivo); se unificó el ranking de características (la curva K vs. F1 y la selección final salen de la misma corrida); se guardan el `scaler` y el diccionario de nombres para la interfaz; se corrigió el manejo del formato de `shap_values` con Random Forest; se agregó la Fase 4 completa (aplicación Gradio, pruebas funcionales, de rendimiento y protocolo SUS); se eliminó el token de GitHub que estaba escrito en el código (debe revocarse) y se retiraron las celdas duplicadas de corridas antiguas.


## 0. Configuración inicial
Monta Google Drive, instala las dependencias que falten (incluido `thrember`, el extractor oficial de EMBER2024) y define la configuración global del experimento: rutas, semilla (42), fracciones de muestreo, valores de K, número de *folds* y las **metas del anteproyecto** (F1 ≥ 0,95; FPR ≤ 5 %; latencia < 10 s; SUS ≥ 70; reducción de dimensionalidad ≥ 90 %).

> Si Colab pide reiniciar el entorno tras la instalación, reinícialo y vuelve a correr esta celda.

In [ ]:
"""
0. CONFIGURACIÓN INICIAL
Monta Google Drive, instala dependencias y define la configuración global
(rutas, semilla, metas del proyecto) que usan todas las fases.
Correr SIEMPRE esta celda primero, en cualquier sesión nueva de Colab.
"""
import os
import sys
import gc
import json
import time
import subprocess
import importlib

# ----------------------------------------------------------------------------
# 0.1 Entorno: Colab (con Drive) o ejecución local (pruebas)
# ----------------------------------------------------------------------------
EN_COLAB = ("google.colab" in sys.modules) or os.path.exists("/content")
if EN_COLAB and not os.environ.get("TG_SIN_DRIVE"):
    from google.colab import drive
    drive.mount("/content/drive")

# Carpeta persistente (Drive): aquí quedan datasets procesados, modelos, figuras y resultados.
BASE_DIR = os.environ.get("TG_BASE_DIR", "/content/drive/MyDrive/trabajo_grado_malware/ember2024")
# Carpeta local de la VM (rápida, se borra al reiniciar la sesión): descargas y trabajo pesado.
LOCAL_DIR = os.environ.get("TG_LOCAL_DIR", "/content/ember2024_local")

RUTAS = {
    "datos":    os.path.join(BASE_DIR, "datos"),      # parquets vectorizados (Fase 1a)
    "fase1":    os.path.join(BASE_DIR, "fase1"),      # selección de características
    "fase2":    os.path.join(BASE_DIR, "fase2"),      # modelos y comparación
    "fase3":    os.path.join(BASE_DIR, "fase3"),      # explicabilidad
    "fase4":    os.path.join(BASE_DIR, "fase4"),      # interfaz y pruebas
    "modelos":  os.path.join(BASE_DIR, "modelos"),    # artefactos serializados
    "figuras":  os.path.join(BASE_DIR, "figuras"),    # figuras para el documento
    "evidencias": os.path.join(BASE_DIR, "evidencias"),  # paquete final de evidencias (celda de exportación)
}
for _r in [BASE_DIR, LOCAL_DIR] + list(RUTAS.values()):
    os.makedirs(_r, exist_ok=True)

# ----------------------------------------------------------------------------
# 0.2 Configuración global del experimento
# ----------------------------------------------------------------------------
CFG = {
    "RANDOM_STATE": 42,
    # --- Fase 1a: qué parte de EMBER2024 se usa ---
    "TIPOS_PE": ["Win32", "Win64", "Dot_Net"],   # el proyecto analiza ejecutables PE de Windows
    "FRACCION_TRAIN": 0.125,   # 1 de cada 8 registros del train PE; como los zip traen cada archivo 2 veces, quedan ~549.000 archivos únicos (23,5 %)
    "FRACCION_TEST": 1.0,      # el set de prueba temporal se usa completo (~540.000 archivos PE)
    "FRACCION_CHALLENGE": 1.0, # conjunto de desafío (malware evasivo) completo
    "BORRAR_ZIP_TRAS_PROCESAR": True,   # libera disco en la VM después de vectorizar cada zip
    "RUTA_ZIPS_LOCAL": os.environ.get("TG_RUTA_ZIPS_LOCAL"),  # solo para pruebas locales sin Hugging Face
    "HF_TOKEN": os.environ.get("HF_TOKEN"),  # opcional; el dataset es público
    # --- Fase 1b: selección de características ---
    "N_MUESTRA_SELECCION": 150_000,
    "TOP_PRELIMINAR": 200,
    "UMBRAL_CORRELACION": 0.90,
    "VALORES_K": [5, 10, 15, 20, 30, 40, 50, 75, 100, 150],
    "META_F1_CONSERVADO": 95.0,   # % del F1 de referencia que debe conservar el subconjunto
    # --- Fase 2: modelos ---
    "N_FOLDS": 3,
    "N_ITER_BUSQUEDA": 5,
    "N_JOBS": 2,
    # --- Metas del anteproyecto (compuertas KPI) ---
    "METAS": {
        "f1_minimo": 0.95,
        "fpr_maximo": 0.05,
        "latencia_maxima_s": 10.0,
        "sus_minimo": 70.0,
        "reduccion_dimensionalidad_minima_pct": 90.0,
    },
    # --- Fase 3 ---
    "N_MUESTRA_SHAP": 2000,
    "N_INSTANCIAS_ACUERDO": 100,
    # --- Exportación de evidencias ---
    "FILAS_MUESTRA_CSV": 5_000,          # filas de cada dataset reducido que se exportan también en CSV legible
    "EXPORTAR_CSV_COMPLETO": False,      # True: además exporta los datasets reducidos completos en .csv.gz (más pesado)
    "LIMITE_MB_MODELO_EN_ZIP": 150,      # los modelos serializados solo entran al zip si pesan menos que esto
    "LOTE_FILAS_LECTURA": 10_000,        # tamaño de bloque al leer parquets (control de memoria)
}
# Ajustes para pruebas rápidas (no usar en la corrida real)
if os.environ.get("TG_MODO_PRUEBA"):
    CFG.update(json.loads(os.environ["TG_MODO_PRUEBA"]))

# ----------------------------------------------------------------------------
# 0.3 Instalación de dependencias (solo las que falten)
# ----------------------------------------------------------------------------
def instalar(paquete_pip, modulo=None, forzar=False):
    """Instala un paquete con pip solo si el módulo no se puede importar."""
    modulo = modulo or paquete_pip.split("==")[0].split(">=")[0]
    try:
        if not forzar:
            importlib.import_module(modulo)
            return
    except ImportError:
        pass
    print(f"Instalando {paquete_pip} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete_pip], check=True)

if not os.environ.get("TG_SIN_INSTALAR"):
    # thrember = EMBER feature version 3 (extractor oficial de EMBER2024, basado en pefile)
    instalar("git+https://github.com/FutureComputing4AI/EMBER2024.git", modulo="thrember")
    # thrember importa 'SignedPEFile' de signify, que fue renombrado en signify >= 0.8:
    # se fija la versión 0.7.1 (última compatible con la API que usa thrember).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "signify==0.7.1"], check=True)
    for _p, _m in [("xgboost", "xgboost"), ("shap", "shap"), ("lime", "lime"), ("gradio", "gradio"),
                   ("pyarrow", "pyarrow"), ("huggingface_hub", "huggingface_hub"), ("joblib", "joblib")]:
        instalar(_p, _m)

# Parche: oscrypto (dependencia de signify) falla con OpenSSL >= 3.0.10 ("Error detecting the
# version of libcrypto") porque su expresión regular solo acepta versiones de un dígito.
def parchear_oscrypto():
    try:
        import oscrypto
    except ImportError:
        return
    carpeta = os.path.join(os.path.dirname(oscrypto.__file__), "_openssl")
    viejo = "re.search('\\\\b(\\\\d\\\\.\\\\d\\\\.\\\\d[a-z]*)\\\\b', version_string)"
    nuevo = "re.search('\\\\b(\\\\d+\\\\.\\\\d+\\\\.\\\\d+[a-z]*)\\\\b', version_string)"
    for archivo in ["_libcrypto_cffi.py", "_libcrypto_ctypes.py"]:
        ruta = os.path.join(carpeta, archivo)
        if os.path.exists(ruta):
            contenido = open(ruta, encoding="utf-8").read()
            if viejo in contenido:
                open(ruta, "w", encoding="utf-8").write(contenido.replace(viejo, nuevo))
                print(f"Parche oscrypto aplicado en {archivo}")

try:
    import thrember
except Exception as e:  # noqa: BLE001
    print("thrember no importó a la primera:", type(e).__name__, "-> aplicando parche y reintentando")
    parchear_oscrypto()
    for _m in list(sys.modules):
        if _m.startswith(("oscrypto", "signify", "thrember")):
            del sys.modules[_m]
    import thrember

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

EXTRACTOR = thrember.PEFeatureExtractor()
N_CARACTERISTICAS = EXTRACTOR.dim   # 2.568 en EMBER feature version 3
GRUPOS_CARACTERISTICAS = [(fe.name, fe.dim) for fe in EXTRACTOR.features]

np.random.seed(CFG["RANDOM_STATE"])

# ----------------------------------------------------------------------------
# 0.4 Utilidades comunes (checkpoints y JSON)
# ----------------------------------------------------------------------------
def esta_listo(ruta):
    """Un archivo se considera completo solo si existe su marca .ok."""
    return os.path.exists(ruta) and os.path.exists(ruta + ".ok")

def marcar_listo(ruta):
    with open(ruta + ".ok", "w") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S"))

def guardar_json(objeto, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(objeto, f, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o))

def cargar_json(ruta):
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)

def guardar_figura(fig, nombre):
    ruta = os.path.join(RUTAS["figuras"], nombre)
    fig.savefig(ruta, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("Figura guardada:", ruta)
    return ruta

def guardar_csv(df, ruta, index=False, **kwargs):
    """Guarda una tabla CSV (UTF-8) e informa dónde quedó: cada resultado queda identificado y descargable."""
    df.to_csv(ruta, index=index, **kwargs)
    print("Tabla guardada:", ruta)
    return ruta

# ----------------------------------------------------------------------------
# 0.5 Control de memoria (Colab gratuito ofrece ~12,7 GB de RAM)
# ----------------------------------------------------------------------------
try:
    import psutil
except ImportError:
    psutil = None
RUTA_REGISTRO_RAM = os.path.join(BASE_DIR, "registro_memoria.csv")

def memoria_ram(etiqueta=""):
    """Imprime la RAM del proceso y la disponible; la registra en registro_memoria.csv (evidencia de que la corrida cabe en Colab)."""
    if psutil is None:
        return None
    proceso_gb = psutil.Process(os.getpid()).memory_info().rss / 2 ** 30
    vm = psutil.virtual_memory()
    print(f"[RAM] {etiqueta}: proceso = {proceso_gb:.2f} GB | disponible = {vm.available / 2 ** 30:.2f} GB de {vm.total / 2 ** 30:.1f} GB")
    nuevo = not os.path.exists(RUTA_REGISTRO_RAM)
    with open(RUTA_REGISTRO_RAM, "a", encoding="utf-8") as f:
        if nuevo:
            f.write("fecha_hora,punto,ram_proceso_gb,ram_disponible_gb,ram_total_gb\n")
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')},{etiqueta},{proceso_gb:.3f},{vm.available / 2 ** 30:.3f},{vm.total / 2 ** 30:.3f}\n")
    return proceso_gb

def devolver_memoria_al_sistema():
    """Devuelve al sistema la memoria que pyarrow y el asignador de C conservan en caché tras liberar objetos."""
    try:
        import pyarrow as _pa
        _pa.default_memory_pool().release_unused()
    except Exception:  # noqa: BLE001
        pass
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:  # noqa: BLE001
        pass

def liberar(*nombres):
    """Elimina variables globales pesadas (si existen), fuerza la recolección de basura y devuelve memoria al sistema."""
    espacio = globals()
    for n in nombres:
        espacio.pop(n, None)
    gc.collect()
    devolver_memoria_al_sistema()

print(f"Entorno: {'Colab' if EN_COLAB else 'local'} | BASE_DIR = {BASE_DIR}")
memoria_ram("inicio")
print(f"thrember OK: {N_CARACTERISTICAS} características en {len(GRUPOS_CARACTERISTICAS)} grupos -> {GRUPOS_CARACTERISTICAS}")


### 0b. Diccionario de características
EMBER2024 representa cada archivo como un vector de **2.568** valores numéricos organizados en **12 grupos** (extraídos sin ejecutar el archivo). Esta celda construye un nombre legible para cada posición del vector (por ejemplo `f1060 → imports.lib_hash_066`) directamente a partir del código del extractor, para que las figuras de importancia, las explicaciones SHAP/LIME y la interfaz no muestren índices sin significado. También exporta la tabla del diccionario de datos usada en el documento.

In [ ]:
"""
0b. DICCIONARIO DE CARACTERÍSTICAS (EMBER feature version 3 / thrember)
Construye, a partir del propio extractor, un nombre legible para cada una de las
2.568 posiciones del vector y una descripción en español por grupo. Se usa para
las figuras de importancia (Fase 1), las explicaciones SHAP/LIME (Fase 3) y la
interfaz (Fase 4), de modo que el usuario no vea índices como "f1060".
"""
from pathlib import Path

# Descripción en español de cada grupo (orden = orden real del vector)
DESCRIPCION_GRUPOS = {
    "general":         ("Información general", "Tamaño, entropía global, si es PE y los 4 primeros bytes del archivo"),
    "histogram":       ("Histograma de bytes", "Frecuencia normalizada de cada valor de byte (0x00-0xFF)"),
    "byteentropy":     ("Histograma byte-entropía", "Distribución conjunta de valor de byte y entropía local en ventanas de 2 KB; detecta empaquetado/cifrado"),
    "strings":         ("Cadenas de texto", "Estadísticas de cadenas imprimibles y conteo de 77 patrones (URLs, IPs, rutas, PowerShell, base64, palabras clave)"),
    "header":          ("Encabezados PE", "Campos de los encabezados COFF, opcional y DOS (timestamp, máquina, subsistema, tamaños, banderas)"),
    "section":         ("Secciones", "Número, tamaños, entropía y permisos de las secciones (hashing) y datos del overlay"),
    "imports":         ("Importaciones", "Librerías y funciones del sistema importadas (hashing trick)"),
    "exports":         ("Exportaciones", "Funciones exportadas por el archivo (hashing trick)"),
    "datadirectories": ("Directorios de datos", "Tamaño y dirección virtual de los 16 directorios de datos del PE, reubicaciones"),
    "richheader":      ("Rich header", "Herramientas y versiones del compilador/enlazador registradas en el Rich header (hashing)"),
    "authenticode":    ("Firma Authenticode", "Número de certificados, autofirma, contrafirma, profundidad de la cadena y tiempos de firma"),
    "pefilewarnings":  ("Advertencias de formato", "Anomalías estructurales detectadas por pefile al analizar el archivo (87 advertencias + conteo)"),
}

def construir_diccionario_caracteristicas(extractor):
    """Devuelve (nombres, grupos): listas de longitud extractor.dim."""
    import thrember
    nombres, grupos = [], []

    def agregar(grupo, lista):
        for n in lista:
            nombres.append(f"{grupo}.{n}")
            grupos.append(grupo)

    for fe in extractor.features:
        g = fe.name
        if g == "general":
            agregar(g, ["size", "entropy", "is_pe", "start_byte_0", "start_byte_1", "start_byte_2", "start_byte_3"])
        elif g == "histogram":
            agregar(g, [f"byte_0x{i:02X}" for i in range(256)])
        elif g == "byteentropy":
            agregar(g, [f"entropia{i:02d}_byte{j:02d}" for i in range(16) for j in range(16)])
        elif g == "strings":
            claves = sorted(fe._regexes)  # mismo orden que usa thrember (regex_idxs)
            agregar(g, ["numstrings", "avlength", "printables"]
                       + [f"printabledist_{i:02d}_{repr(chr(0x20 + i))}" for i in range(96)]
                       + ["entropy"] + [f"count_{k}" for k in claves])
        elif g == "header":
            opt = ["major_image_version", "minor_image_version", "major_linker_version", "minor_linker_version",
                   "major_operating_system_version", "minor_operating_system_version", "major_subsystem_version",
                   "minor_subsystem_version", "sizeof_code", "sizeof_headers", "sizeof_image",
                   "sizeof_initialized_data", "sizeof_uninitialized_data", "sizeof_stack_reserve",
                   "sizeof_stack_commit", "sizeof_heap_reserve", "sizeof_heap_commit", "address_of_entrypoint",
                   "base_of_code", "image_base", "section_alignment", "checksum", "number_of_rvas_and_sizes"]
            agregar(g, ["coff.timestamp", "coff.number_of_sections", "coff.number_of_symbols",
                        "coff.sizeof_optional_header", "coff.pointer_to_symbol_table", "coff.machine", "optional.subsystem"]
                       + [f"optional.{o}" for o in opt]
                       + [f"characteristics.{c}" for c in fe._image_characteristics]
                       + [f"dll_characteristics.{c}" for c in fe._dll_characteristics]
                       + [f"dos.{m}" for m in fe._dos_members])
        elif g == "section":
            agregar(g, ["n_sections", "n_zero_size", "n_empty_name", "n_rx", "n_w", "max_entropy", "min_entropy",
                        "max_size_ratio", "min_size_ratio", "max_vsize_ratio", "min_vsize_ratio"]
                       + [f"size_hash_{i:02d}" for i in range(50)] + [f"vsize_hash_{i:02d}" for i in range(50)]
                       + [f"entropy_hash_{i:02d}" for i in range(50)] + [f"props_hash_{i:02d}" for i in range(50)]
                       + [f"entry_hash_{i:02d}" for i in range(10)]
                       + ["overlay.size", "overlay.size_ratio", "overlay.entropy"])
        elif g == "imports":
            agregar(g, ["n_imports", "n_libraries"] + [f"lib_hash_{i:03d}" for i in range(256)]
                       + [f"func_hash_{i:04d}" for i in range(1024)])
        elif g == "exports":
            agregar(g, ["has_exports"] + [f"hash_{i:03d}" for i in range(128)])
        elif g == "datadirectories":
            lst = []
            for n in fe._name_order:
                lst += [f"{n}.size", f"{n}.rva"]
            agregar(g, lst + ["has_relocs", "has_dynamic_relocs"])
        elif g == "richheader":
            agregar(g, ["n_pairs"] + [f"hash_{i:02d}" for i in range(32)])
        elif g == "authenticode":
            agregar(g, ["num_certs", "self_signed", "empty_program_name", "no_countersigner", "parse_error",
                        "chain_max_depth", "latest_signing_time", "signing_time_diff"])
        elif g == "pefilewarnings":
            ids = sorted(fe.warning_ids.items(), key=lambda kv: kv[1])
            lst = [f"w{i:02d}_{texto.replace('...', '').strip()[:40]}" for texto, i in ids]
            # posiciones sin advertencia asignada (el archivo trae 87 advertencias para 88 posiciones)
            while len(lst) < fe.dim - 1:
                lst.append(f"w{len(lst):02d}_sin_asignar")
            agregar(g, lst + ["n_warnings"])
        else:
            agregar(g, [f"x{i}" for i in range(fe.dim)])
        assert len(nombres) == sum(f.dim for f in extractor.features[: extractor.features.index(fe) + 1]), g
    assert len(nombres) == extractor.dim, (len(nombres), extractor.dim)
    return nombres, grupos

NOMBRES_CARACTERISTICAS, GRUPO_DE_CARACTERISTICA = construir_diccionario_caracteristicas(EXTRACTOR)

def nombre_legible(columna):
    """'f1060' -> 'imports.lib_hash_064' (nombre técnico legible)."""
    if isinstance(columna, str) and columna.startswith("f") and columna[1:].isdigit():
        return NOMBRES_CARACTERISTICAS[int(columna[1:])]
    return str(columna)

def grupo_de(columna):
    if isinstance(columna, str) and columna.startswith("f") and columna[1:].isdigit():
        return GRUPO_DE_CARACTERISTICA[int(columna[1:])]
    return "desconocido"

# Tabla resumen (para el documento): grupo, n.º de variables, descripción
tabla_diccionario = pd.DataFrame(
    [{"grupo": g, "nombre": DESCRIPCION_GRUPOS[g][0], "n_variables": d,
      "indice_inicial": sum(x[1] for x in GRUPOS_CARACTERISTICAS[:i]),
      "descripcion": DESCRIPCION_GRUPOS[g][1]}
     for i, (g, d) in enumerate(GRUPOS_CARACTERISTICAS)]
)
tabla_diccionario.to_csv(os.path.join(RUTAS["datos"], "diccionario_grupos.csv"), index=False)
guardar_json({"nombres": NOMBRES_CARACTERISTICAS, "grupos": GRUPO_DE_CARACTERISTICA},
             os.path.join(RUTAS["datos"], "diccionario_caracteristicas.json"))
# Diccionario completo (una fila por cada una de las 2.568 posiciones del vector): evidencia del diccionario de datos
guardar_csv(pd.DataFrame({"columna": [f"f{i}" for i in range(N_CARACTERISTICAS)], "indice": range(N_CARACTERISTICAS),
                          "grupo": GRUPO_DE_CARACTERISTICA, "nombre": NOMBRES_CARACTERISTICAS}),
            os.path.join(RUTAS["datos"], "diccionario_caracteristicas.csv"))
print(tabla_diccionario.to_string(index=False))
print(f"Total: {tabla_diccionario['n_variables'].sum()} características")


## Fase 1a — Adquisición y vectorización de EMBER2024 (OE1)
Descarga desde Hugging Face los archivos comprimidos de los tres formatos PE (Win32, Win64, .NET) para los conjuntos de **entrenamiento** (semanas 1-52), **prueba temporal** (semanas 53-64) y **desafío** (6.315 archivos maliciosos que ningún antivirus detectó en su primer análisis). Los `.jsonl` se leen en *streaming* desde el zip (nunca se descomprimen a disco) y cada archivo se vectoriza con `thrember.PEFeatureExtractor`.

Por las restricciones de Colab gratuito (≈12 GB de RAM y ≈70 GB de disco) el entrenamiento se **muestrea de forma aleatoria y determinista** (`FRACCION_TRAIN = 0.125`, 1 de cada 8 registros), lo que conserva las proporciones por tipo de archivo, semana y clase; la prueba temporal (≈540.000 archivos PE) y el conjunto de desafío se usan completos.

**Hallazgo de la corrida 1 (septiembre de 2026):** los `.zip` de entrenamiento y prueba de EMBER2024 contienen **cada registro dos veces** (la lectura completa de la prueba produjo 1.080.000 registros → 539.940 archivos únicos por SHA-256; la muestra de entrenamiento, 585.371 registros → 548.997 archivos únicos, es decir, una fracción efectiva del 23,5 %). La Fase 1b ya deduplicaba antes de seleccionar características y de construir los conjuntos reducidos, así que los resultados son válidos; desde esta versión la celda **omite los duplicados al leer cada zip** (ahorra la mitad del tiempo de vectorización) y el resumen del dataset cuenta archivos únicos.

⚠️ Primera ejecución: 1,5–2,5 horas. No cierres la pestaña. Si la sesión se reinicia, vuelve a correr desde la celda 0: los zips ya procesados se saltan.

In [ ]:
"""
FASE 1a - ADQUISICIÓN Y VECTORIZACIÓN DE EMBER2024 (subconjunto PE)

Qué hace:
  1. Descarga desde Hugging Face (joyce8/EMBER2024) los zips de los tres formatos
     PE de Windows (Win32, Win64, .NET) para train, test y el conjunto de desafío.
  2. Lee cada zip EN STREAMING (sin descomprimirlo a disco: los .jsonl extraídos
     ocuparían ~46 GB, más de lo que ofrece Colab gratuito).
  3. Aplica un muestreo aleatorio determinista (semilla fija) por línea:
     FRACCION_TRAIN para el entrenamiento y FRACCION_TEST para la prueba. Al ser
     uniforme, conserva las proporciones por tipo de archivo, semana y clase.
  4. Vectoriza cada archivo con el extractor oficial (thrember.PEFeatureExtractor
     .process_raw_features -> 2.568 valores float32) y guarda parquet por bloques.
  5. Deja marcas .ok por zip: si la sesión se cae, al volver a correr continúa
     donde quedó y no repite lo ya vectorizado.

Metadatos conservados por archivo: sha256, label (0 benigno / 1 malicioso),
file_type, first_submission_date, semana (0-63 desde 2023-09-24), family y behavior.

Duplicados en el origen: los .zip de entrenamiento y prueba de EMBER2024 contienen
cada registro dos veces (dos entradas con el mismo SHA-256); se detectó en la primera
corrida al deduplicar (1.080.000 registros de prueba -> 539.940 archivos únicos). Por
eso cada zip lleva un conjunto de SHA-256 ya guardados y omite las repeticiones sin
vectorizarlas (ahorra la mitad del tiempo); el resumen final cuenta archivos únicos.

Control de memoria: nunca se carga un zip ni un .jsonl completo en RAM. Las líneas
se leen una a una desde el zip, el pool de procesos recibe lotes de 64 líneas (la
tubería del pool limita cuántos lotes esperan en memoria) y el resultado se
escribe en parquet cada FILAS_POR_BLOQUE filas (5.000 x 2.568 float32 ≈ 50 MB).
"""
import io
import zlib
import shutil
import zipfile
import multiprocessing
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

REPO_HF = "joyce8/EMBER2024"
INICIO_EMBER2024 = int(pd.Timestamp("2023-09-24", tz="UTC").timestamp())  # semana 0 (igual que el notebook oficial)
SEGUNDOS_SEMANA = 7 * 24 * 3600
FILAS_POR_BLOQUE = 5_000
COLUMNAS_F = [f"f{i}" for i in range(N_CARACTERISTICAS)]

# Trabajo a realizar: (nombre del zip, split, fracción de muestreo, nombre de salida)
TRABAJOS = []
for _tipo in CFG["TIPOS_PE"]:
    TRABAJOS.append((f"{_tipo}_train.zip", "train", CFG["FRACCION_TRAIN"], f"train_{_tipo}.parquet"))
    TRABAJOS.append((f"{_tipo}_test.zip", "test", CFG["FRACCION_TEST"], f"test_{_tipo}.parquet"))
TRABAJOS.append(("challenge.zip", "challenge", CFG["FRACCION_CHALLENGE"], "challenge_PE.parquet"))


def ruta_zip(nombre):
    """Ruta local del zip: copia local (pruebas) o descarga desde Hugging Face (con caché)."""
    if CFG.get("RUTA_ZIPS_LOCAL"):
        candidato = os.path.join(CFG["RUTA_ZIPS_LOCAL"], nombre)
        if os.path.exists(candidato):
            return candidato
    return hf_hub_download(repo_id=REPO_HF, filename=nombre, repo_type="dataset",
                           cache_dir=os.path.join(LOCAL_DIR, "hf_cache"), token=CFG.get("HF_TOKEN"))


def borrar_zip(ruta):
    """Elimina el zip (y el blob de la caché de Hugging Face) para liberar disco."""
    try:
        real = os.path.realpath(ruta)
        if os.path.islink(ruta):
            os.remove(ruta)
        if os.path.exists(real):
            os.remove(real)
        print(f"  zip eliminado para liberar disco: {os.path.basename(ruta)}")
    except OSError as e:
        print("  no se pudo borrar el zip:", e)


# --- Trabajadores del pool: cada proceso crea su propio extractor una sola vez ---
_EXTRACTOR_WORKER = None

def _init_worker():
    global _EXTRACTOR_WORKER
    import thrember as _t
    _EXTRACTOR_WORKER = _t.PEFeatureExtractor()

def _vectorizar_linea(linea):
    """Convierte una línea JSON de EMBER2024 en (vector float32, metadatos)."""
    rec = json.loads(linea)
    vec = _EXTRACTOR_WORKER.process_raw_features(rec)
    comportamiento = rec.get("behavior")
    if isinstance(comportamiento, list):
        comportamiento = ";".join(str(b) for b in comportamiento)
    meta = (rec.get("sha256"), rec.get("label"), rec.get("file_type"),
            rec.get("first_submission_date"), rec.get("family"), comportamiento)
    return vec, meta


def normalizar_tipo(tipo):
    """Unifica la escritura del tipo de archivo ('Dot_Net', '.NET', 'win32', ...)."""
    t = str(tipo).strip().lower().replace(".", "").replace("_", "")
    return {"win32": "Win32", "win64": "Win64", "dotnet": "Dot_Net", "apk": "APK", "elf": "ELF", "pdf": "PDF"}.get(t, str(tipo))


def lineas_muestreadas(zf, miembro, fraccion, split):
    """Genera las líneas (bytes) que pasan el muestreo determinista por línea."""
    rng = np.random.default_rng([CFG["RANDOM_STATE"], zlib.crc32(miembro.encode())])
    patrones_pe = [b"win32", b"win64", b"dot_net", b"dotnet", b".net"]
    with zf.open(miembro) as f:
        for linea in f:
            if fraccion < 1.0 and rng.random() >= fraccion:
                continue
            if split == "challenge":
                # el challenge mezcla 6 formatos: filtro barato por tipo antes de parsear el JSON completo
                if not any(p in linea.lower() for p in patrones_pe):
                    continue
            yield linea


def procesar_zip(nombre_zip, split, fraccion, nombre_salida):
    ruta_salida = os.path.join(RUTAS["datos"], nombre_salida)
    if esta_listo(ruta_salida):
        print(f"[{nombre_zip}] ya vectorizado -> {nombre_salida} (se omite)")
        return
    if os.path.exists(ruta_salida):
        os.remove(ruta_salida)  # quedó incompleto en una corrida anterior

    t0 = time.time()
    ruta = ruta_zip(nombre_zip)
    print(f"[{nombre_zip}] procesando (split={split}, fracción={fraccion}) desde {ruta}")
    tipos = CFG["TIPOS_PE"]

    esquema = pa.schema([pa.field(c, pa.float32()) for c in COLUMNAS_F] + [
        pa.field("sha256", pa.string()), pa.field("label", pa.int8()), pa.field("file_type", pa.string()),
        pa.field("first_submission_date", pa.int64()), pa.field("semana", pa.int16()),
        pa.field("family", pa.string()), pa.field("behavior", pa.string()), pa.field("split", pa.string()),
    ])
    escritor = pq.ParquetWriter(ruta_salida, esquema, compression="snappy")

    bloque_X, bloque_meta = [], []
    n_guardadas = n_sin_etiqueta = n_leidas = n_duplicados = 0
    vistos = set()   # SHA-256 ya guardados en este zip (los zip de EMBER2024 repiten cada registro)

    def volcar():
        nonlocal bloque_X, bloque_meta, n_guardadas
        if not bloque_X:
            return
        X = np.vstack(bloque_X)
        arrays = [pa.array(X[:, j], type=pa.float32()) for j in range(X.shape[1])]
        sha, lab, tipo, fecha, fam, beh = zip(*bloque_meta)
        fecha = [int(x) if x is not None else -1 for x in fecha]
        semana = [int((x - INICIO_EMBER2024) // SEGUNDOS_SEMANA) if x >= 0 else -1 for x in fecha]
        arrays += [pa.array(sha, pa.string()), pa.array([int(l) for l in lab], pa.int8()), pa.array(tipo, pa.string()),
                   pa.array(fecha, pa.int64()), pa.array(semana, pa.int16()),
                   pa.array([f if f is None else str(f) for f in fam], pa.string()),
                   pa.array(beh, pa.string()), pa.array([split] * len(sha), pa.string())]
        escritor.write_table(pa.Table.from_arrays(arrays, schema=esquema))
        n_guardadas += len(sha)
        bloque_X, bloque_meta = [], []

    with zipfile.ZipFile(ruta) as zf:
        miembros = [m for m in zf.namelist() if m.endswith(".jsonl")]
        print(f"  {len(miembros)} archivos .jsonl dentro del zip")
        n_proc = max(1, min(CFG["N_JOBS"], multiprocessing.cpu_count()))
        with multiprocessing.get_context("fork").Pool(n_proc, initializer=_init_worker) as pool:
            for miembro in sorted(miembros):
                gen = lineas_muestreadas(zf, miembro, fraccion, split)
                for vec, meta in pool.imap(_vectorizar_linea, gen, chunksize=64):
                    n_leidas += 1
                    sha, lab, tipo, fecha, fam, beh = meta
                    tipo = normalizar_tipo(tipo)
                    meta = (sha, lab, tipo, fecha, fam, beh)
                    if split == "challenge" and tipo not in tipos:
                        continue
                    if split == "challenge" and lab is None:
                        lab = 1  # el conjunto de desafío es 100 % malicioso por construcción
                        meta = (sha, lab, tipo, fecha, fam, beh)
                    if lab is None or int(lab) < 0:
                        n_sin_etiqueta += 1
                        continue
                    if sha in vistos:
                        n_duplicados += 1
                        continue
                    vistos.add(sha)
                    if not np.all(np.isfinite(vec)):
                        vec = np.nan_to_num(vec, nan=0.0, posinf=0.0, neginf=0.0)
                    bloque_X.append(vec.astype(np.float32))
                    bloque_meta.append(meta)
                    if len(bloque_X) >= FILAS_POR_BLOQUE:
                        volcar()
                        if n_guardadas % 50_000 == 0:
                            print(f"  {n_guardadas:,} archivos guardados ({time.time() - t0:.0f} s)")
    volcar()
    escritor.close()
    del vistos
    marcar_listo(ruta_salida)
    guardar_json({"zip": nombre_zip, "split": split, "fraccion": fraccion, "registros_leidos": n_leidas, "archivos_guardados": n_guardadas,
                  "duplicados_omitidos": n_duplicados, "sin_etiqueta": n_sin_etiqueta, "segundos": round(time.time() - t0, 1)},
                 ruta_salida + ".resumen.json")
    print(f"[{nombre_zip}] listo: {n_guardadas:,} archivos guardados, {n_duplicados:,} duplicados omitidos, "
          f"{n_sin_etiqueta} sin etiqueta descartados, {time.time() - t0:.0f} s -> {nombre_salida}")
    if CFG["BORRAR_ZIP_TRAS_PROCESAR"] and not CFG.get("RUTA_ZIPS_LOCAL"):
        borrar_zip(ruta)


for _zip, _split, _frac, _salida in TRABAJOS:
    procesar_zip(_zip, _split, _frac, _salida)
    gc.collect()
    devolver_memoria_al_sistema()
    memoria_ram(f"fase1a tras {_zip}")

# --- Resumen del dataset construido (conteos de ARCHIVOS ÚNICOS por split, tipo y clase) ---
# Solo se leen las columnas de metadatos (no las 2.568 características) para no cargar los parquets completos.
# Un mismo sha256 puede aparecer en parquets distintos del mismo split (p. ej. un archivo etiquetado en dos formatos):
# se cuenta una sola vez, igual que hará la Fase 1b al construir los conjuntos reducidos.
filas, filas_semana, duplicados_entre_archivos = [], [], {}
_vistos_split = {}
for _zip, _split, _frac, _salida in TRABAJOS:
    ruta = os.path.join(RUTAS["datos"], _salida)
    meta = pd.read_parquet(ruta, columns=["sha256", "label", "file_type", "semana"])
    _v = _vistos_split.setdefault(_split, set())
    _repetido = meta["sha256"].isin(_v)
    duplicados_entre_archivos[_salida] = int(_repetido.sum())
    _v.update(meta.loc[~_repetido, "sha256"].tolist())
    meta = meta.loc[~_repetido, ["label", "file_type", "semana"]]
    for (tipo, lab), n in meta.groupby(["file_type", "label"]).size().items():
        filas.append({"split": _split, "file_type": tipo, "label": int(lab), "n": int(n),
                      "semana_min": int(meta.loc[meta.file_type == tipo, "semana"].min()),
                      "semana_max": int(meta.loc[meta.file_type == tipo, "semana"].max())})
    for (sem, lab), n in meta.groupby(["semana", "label"]).size().items():
        filas_semana.append({"split": _split, "archivo": _salida, "semana": int(sem), "label": int(lab), "n": int(n)})
    del meta
del _vistos_split
resumen_datos = pd.DataFrame(filas)
resumen_datos.to_csv(os.path.join(RUTAS["datos"], "resumen_datos.csv"), index=False)
# Evidencia adicional: distribución por semana y clase (muestra que el muestreo conserva la cobertura temporal)
guardar_csv(pd.DataFrame(filas_semana), os.path.join(RUTAS["datos"], "distribucion_por_semana.csv"))
tabla = resumen_datos.pivot_table(index=["split", "file_type"], columns="label", values="n", aggfunc="sum", fill_value=0)
tabla.columns = ["benignos", "maliciosos"][: len(tabla.columns)]
tabla["total"] = tabla.sum(axis=1)
print("\n=== Dataset construido (archivos PE de EMBER2024) ===")
print(tabla.to_string())
totales = tabla.groupby(level=0)["total"].sum().to_dict()
resumenes_zip = [cargar_json(os.path.join(RUTAS["datos"], _s + ".resumen.json")) for *_, _s in TRABAJOS
                 if os.path.exists(os.path.join(RUTAS["datos"], _s + ".resumen.json"))]
guardar_json({"por_split_y_tipo": tabla.reset_index().to_dict(orient="records"), "totales": totales,
              "nota": "conteos de archivos únicos por SHA-256 (los .zip de EMBER2024 traen cada registro dos veces)",
              "duplicados_omitidos_por_zip": {r["zip"]: r["duplicados_omitidos"] for r in resumenes_zip},
              "duplicados_entre_archivos_del_mismo_split": duplicados_entre_archivos,
              "fraccion_train": CFG["FRACCION_TRAIN"], "fraccion_test": CFG["FRACCION_TEST"],
              "n_caracteristicas": N_CARACTERISTICAS, "tipos_pe": CFG["TIPOS_PE"]},
             os.path.join(RUTAS["datos"], "resumen_datos.json"))
print("Totales (archivos únicos):", totales)
del filas, filas_semana, tabla
memoria_ram("fase1a fin")


## Fase 1b — Limpieza, selección de características y reducción de dimensionalidad (OE1)
Sobre una muestra estratificada de 150.000 archivos de entrenamiento: (1) limpieza (duplicados por SHA-256, valores no finitos, columnas constantes); (2) ranking de importancia con **Extra Trees** (impureza de Gini) y F1 de referencia con todas las variables; (3) filtro de redundancia por **correlación** (|r| > 0,90); (4) **curva K vs. F1 conservado** para K ∈ {5, 10, 15, 20, 30, 40, 50, 75, 100, 150}; (5) regla explícita de selección: **el menor K que conserva al menos el 95 % del F1 de referencia**. Genera los datasets reducidos y las figuras del capítulo 1.

**Evidencias (carpeta `fase1/`):** ranking completo de importancias, variables constantes y redundantes descartadas (con su |r|), curva K vs. F1, tabla de las K características seleccionadas (nombre, grupo, importancia), matriz de correlación del subconjunto, SHA-256 de la muestra usada y las **bases de datos reducidas** `train/test/challenge_reducido.parquet`.

**Memoria:** la muestra se lee por bloques directamente a una matriz float32 preasignada (≈1,5 GB) y la limpieza se hace en el sitio; los datasets reducidos se escriben por bloques (un bloque de 50.000 filas en RAM a la vez).

In [ ]:
"""
FASE 1b - LIMPIEZA, SELECCIÓN DE CARACTERÍSTICAS Y REDUCCIÓN DE DIMENSIONALIDAD

Qué hace (sobre una muestra estratificada del entrenamiento, leída por bloques):
  1. Limpieza: valores no finitos -> 0, duplicados por sha256, columnas constantes.
  2. Ranking de importancia con Extra Trees (criterio Gini) + F1 de referencia
     con TODAS las variables sobre una partición de validación (25 %).
  3. Filtro de redundancia: se descartan variables con |r| > UMBRAL_CORRELACION
     frente a otra mejor posicionada en el ranking.
  4. Curva K vs. F1 conservado: se entrena un clasificador por cada K candidato y
     se elige el MENOR K que conserva al menos META_F1_CONSERVADO % del F1 de
     referencia (regla explícita y reproducible, no un valor "a ojo").
  5. Construye los datasets reducidos (train / test / challenge) con esas K
     columnas y guarda el resumen de KPIs de la fase y las figuras.

Un mismo ranking se usa para la curva y para la selección final (en la versión
anterior del proyecto la curva y el ranking final venían de corridas distintas).

Control de memoria (Colab ≈ 12,7 GB): la muestra se lee por bloques y se escribe
directamente en UNA matriz float32 preasignada (150.000 x 2.568 ≈ 1,5 GB); nunca
se construye un DataFrame con todos los bloques ni copias intermedias. La limpieza
se hace en el sitio, la partición train/validación se copia una sola vez y la
matriz original se libera. Los datasets reducidos se escriben por bloques.

Evidencias que deja en fase1/: ranking completo de importancias, variables
constantes y redundantes descartadas, curva K vs. F1, características
seleccionadas (con nombre y grupo), matriz de correlación del subconjunto,
identificadores (sha256) de la muestra usada y los datasets reducidos.
"""
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

RUTA_RESUMEN_F1 = os.path.join(RUTAS["fase1"], "fase1_resumen.json")
RUTA_RANKING = os.path.join(RUTAS["fase1"], "ranking_caracteristicas.json")
RUTA_CURVA = os.path.join(RUTAS["fase1"], "curva_k_vs_f1.csv")
RUTA_SELECCION = os.path.join(RUTAS["fase1"], "seleccion_final.json")
PARQUETS_TRAIN = [os.path.join(RUTAS["datos"], f"train_{t}.parquet") for t in CFG["TIPOS_PE"]]
PARQUETS_TEST = [os.path.join(RUTAS["datos"], f"test_{t}.parquet") for t in CFG["TIPOS_PE"]]
PARQUET_CHALLENGE = os.path.join(RUTAS["datos"], "challenge_PE.parquet")
COLS_META = ["sha256", "label", "file_type", "first_submission_date", "semana", "family", "behavior", "split"]
LOTE = int(CFG["LOTE_FILAS_LECTURA"])


def cargar_muestra_matriz(parquets, n_total, semilla):
    """Muestra aleatoria de ~n_total filas leída por bloques directamente a una matriz float32 preasignada.

    Cada bloque de LOTE filas se muestrea con una semilla propia (reproducible), solo las filas elegidas se
    convierten a numpy y el bloque se descarta antes de leer el siguiente. Los duplicados por sha256 se filtran en
    la misma pasada. Devuelve X (n x 2568 float32), y (int8), lista de sha256, n_disponibles y n_duplicados."""
    n_disponibles = sum(pq.ParquetFile(p).metadata.num_rows for p in parquets)
    frac = min(1.0, n_total / max(n_disponibles, 1))
    capacidad = n_total + n_disponibles // LOTE + len(parquets) + 2   # holgura por el redondeo de cada bloque
    X = np.empty((capacidad, N_CARACTERISTICAS), dtype=np.float32)
    y = np.empty(capacidad, dtype=np.int8)
    shas, vistos, n, n_dup, s = [], set(), 0, 0, semilla
    for p in parquets:
        for lote in pq.ParquetFile(p).iter_batches(batch_size=LOTE, columns=COLUMNAS_F + ["sha256", "label"]):
            rng = np.random.default_rng(s)
            s += 1
            k = lote.num_rows if frac >= 1.0 else int(round(frac * lote.num_rows))
            if k == 0:
                continue
            if k < lote.num_rows:
                lote = lote.take(pa.array(np.sort(rng.choice(lote.num_rows, k, replace=False))))
            df_b = lote.to_pandas()
            del lote
            sha_b = df_b["sha256"].tolist()
            nuevos = []
            for i, h in enumerate(sha_b):
                if h in vistos:
                    n_dup += 1
                else:
                    vistos.add(h)
                    nuevos.append(i)
            m = len(nuevos)
            if m == 0:
                continue
            if n + m > capacidad:   # no debería ocurrir; protección por si el redondeo acumula más filas de las previstas
                m = capacidad - n
                nuevos = nuevos[:m]
            mat = df_b[COLUMNAS_F].to_numpy(dtype=np.float32)
            X[n:n + m] = mat if m == len(mat) else mat[nuevos]
            y[n:n + m] = df_b["label"].to_numpy()[nuevos]
            del mat
            shas.extend(sha_b[i] for i in nuevos)
            n += m
            del df_b
    del vistos
    gc.collect()
    devolver_memoria_al_sistema()   # pyarrow conserva en caché los bloques ya liberados; se devuelven al sistema
    return X[:n], y[:n], shas, n_disponibles, n_dup


# ----------------------------------------------------------------------------
# 1) Muestra de entrenamiento y limpieza (en el sitio, sin copias de la matriz)
# ----------------------------------------------------------------------------
t_fase = time.time()
memoria_ram("fase1b inicio")
X, y, sha_muestra, n_train_total, n_dup = cargar_muestra_matriz(PARQUETS_TRAIN, CFG["N_MUESTRA_SELECCION"], CFG["RANDOM_STATE"])
y = y.astype(int)
print(f"Filas en el entrenamiento (muestreado en Fase 1a): {n_train_total:,} | muestra para selección: {len(y):,} "
      f"({X.nbytes / 2 ** 30:.2f} GB en RAM) | duplicados por sha256 eliminados: {n_dup}")
memoria_ram("fase1b muestra cargada")

# Valores no finitos -> 0, por bloques de filas y en el sitio (evita crear una máscara booleana de toda la matriz)
def limpiar_no_finitos(matriz, filas_por_bloque=20_000):
    corregidos = 0
    for i in range(0, len(matriz), filas_por_bloque):
        bloque = matriz[i:i + filas_por_bloque]
        mascara = ~np.isfinite(bloque)
        c = int(mascara.sum())
        if c:
            bloque[mascara] = 0.0
            corregidos += c
    return corregidos

n_no_finitos = limpiar_no_finitos(X)

# Columnas constantes: min == max (equivale a nunique <= 1, sin pasar por pandas)
minimos, maximos = X.min(axis=0), X.max(axis=0)
idx_limpias = np.where(minimos != maximos)[0]
idx_constantes = np.where(minimos == maximos)[0]
cols_constantes = [COLUMNAS_F[i] for i in idx_constantes]
columnas_limpias = [COLUMNAS_F[i] for i in idx_limpias]
n_original, n_limpio = N_CARACTERISTICAS, len(columnas_limpias)
distribucion = {"benignos": int((y == 0).sum()), "maliciosos": int((y == 1).sum())}
print(f"Valores no finitos corregidos: {n_no_finitos}")
print(f"Columnas constantes eliminadas: {len(cols_constantes)} -> quedan {n_limpio} de {n_original}")
print(f"Distribución de clases en la muestra: {distribucion} "
      f"(proporción maliciosos = {distribucion['maliciosos'] / len(y):.3f})")
guardar_csv(pd.DataFrame({"columna": cols_constantes, "nombre": [nombre_legible(c) for c in cols_constantes],
                          "grupo": [grupo_de(c) for c in cols_constantes], "valor_constante": minimos[idx_constantes]}),
            os.path.join(RUTAS["fase1"], "columnas_constantes.csv"))

# Partición estratificada 75/25 por índices: cada matriz se copia UNA sola vez (solo columnas limpias) y X se libera
idx_tr, idx_va = train_test_split(np.arange(len(y)), test_size=0.25, random_state=CFG["RANDOM_STATE"], stratify=y)
idx_tr.sort(); idx_va.sort()
X_val = X[np.ix_(idx_va, idx_limpias)]
X_train = X[np.ix_(idx_tr, idx_limpias)]
y_train, y_val = y[idx_tr], y[idx_va]
particion = np.full(len(y), "entrenamiento", dtype=object); particion[idx_va] = "validacion"
guardar_csv(pd.DataFrame({"sha256": sha_muestra, "label": y, "particion": particion}),
            os.path.join(RUTAS["fase1"], "muestra_seleccion_sha256.csv.gz"), compression="gzip")
del X, particion, sha_muestra, idx_tr, idx_va
gc.collect()
devolver_memoria_al_sistema()
memoria_ram("fase1b tras partición (matriz original liberada)")

# ----------------------------------------------------------------------------
# 2) Ranking de importancia (Extra Trees) y F1 de referencia con todas las variables
# ----------------------------------------------------------------------------
if os.path.exists(RUTA_RANKING) and os.path.exists(RUTA_CURVA) and os.path.exists(RUTA_SELECCION):
    print("Fase 1b ya fue ejecutada: se cargan ranking, curva y selección guardados.")
    ranking_info = cargar_json(RUTA_RANKING)
    ranking_sin_redundancia = ranking_info["ranking_sin_redundancia"]
    f1_full = ranking_info["f1_referencia_todas"]
    df_curva = pd.read_csv(RUTA_CURVA)
    seleccion = cargar_json(RUTA_SELECCION)
    K_FINAL, columnas_seleccionadas = seleccion["K"], seleccion["columnas"]
    tiempo_ranking = ranking_info["tiempo_ranking_s"]
    importancias = pd.Series(ranking_info["importancia"])
else:
    print("Entrenando Extra Trees para el ranking de importancia...")
    t0 = time.time()
    etc = ExtraTreesClassifier(n_estimators=300, max_depth=25, min_samples_leaf=5, criterion="gini",
                               random_state=CFG["RANDOM_STATE"], n_jobs=CFG["N_JOBS"])
    etc.fit(X_train, y_train)
    tiempo_ranking = time.time() - t0
    f1_full = float(f1_score(y_val, etc.predict(X_val)))
    importancias = pd.Series(etc.feature_importances_, index=columnas_limpias).sort_values(ascending=False)
    del etc
    gc.collect()
    print(f"Ranking calculado en {tiempo_ranking:.0f} s | F1 de referencia ({n_limpio} variables): {f1_full:.4f}")
    # Evidencia: ranking completo (todas las variables limpias) con nombre y grupo
    guardar_csv(pd.DataFrame({"posicion": np.arange(1, len(importancias) + 1), "columna": importancias.index,
                              "nombre": [nombre_legible(c) for c in importancias.index],
                              "grupo": [grupo_de(c) for c in importancias.index], "importancia": importancias.values}),
                os.path.join(RUTAS["fase1"], "importancia_extratrees_completa.csv"))

    # 3) Filtro de redundancia por correlación sobre el top preliminar
    top_preliminar = importancias.head(CFG["TOP_PRELIMINAR"]).index.tolist()
    idx_top = [columnas_limpias.index(c) for c in top_preliminar]
    corr = pd.DataFrame(X_train[:, idx_top], columns=top_preliminar).corr().abs()
    descartadas, filas_redundantes = set(), []
    for i, ci in enumerate(top_preliminar):
        if ci in descartadas:
            continue
        for cj in top_preliminar[i + 1:]:
            if cj not in descartadas and corr.loc[ci, cj] > CFG["UMBRAL_CORRELACION"]:
                descartadas.add(cj)
                filas_redundantes.append({"columna_descartada": cj, "nombre": nombre_legible(cj), "grupo": grupo_de(cj),
                                          "redundante_con": ci, "nombre_conservada": nombre_legible(ci),
                                          "r_abs": round(float(corr.loc[ci, cj]), 4)})
    del corr
    ranking_sin_redundancia = [c for c in top_preliminar if c not in descartadas]
    print(f"Candidatos tras filtro de correlación (|r| > {CFG['UMBRAL_CORRELACION']}): "
          f"{len(ranking_sin_redundancia)} de {len(top_preliminar)} ({len(descartadas)} redundantes)")
    guardar_csv(pd.DataFrame(filas_redundantes, columns=["columna_descartada", "nombre", "grupo", "redundante_con",
                                                         "nombre_conservada", "r_abs"]),
                os.path.join(RUTAS["fase1"], "redundantes_descartadas.csv"))
    guardar_json({"f1_referencia_todas": f1_full, "n_caracteristicas_originales": n_original,
                  "n_caracteristicas_limpias": n_limpio, "columnas_constantes": cols_constantes,
                  "ranking_sin_redundancia": ranking_sin_redundancia,
                  "importancia": importancias.head(CFG["TOP_PRELIMINAR"]).round(6).to_dict(),
                  "redundantes_descartadas": sorted(descartadas), "tiempo_ranking_s": round(tiempo_ranking, 1)},
                 RUTA_RANKING)

    # 4) Curva K vs. F1 conservado (mismo ranking, mismos datos de validación)
    filas = []
    for k in CFG["VALORES_K"]:
        if k > len(ranking_sin_redundancia):
            print(f"K={k} supera los candidatos disponibles ({len(ranking_sin_redundancia)}); se omite.")
            continue
        sel = [columnas_limpias.index(c) for c in ranking_sin_redundancia[:k]]
        t0 = time.time()
        clf = ExtraTreesClassifier(n_estimators=200, max_depth=25, min_samples_leaf=5,
                                   random_state=CFG["RANDOM_STATE"], n_jobs=CFG["N_JOBS"])
        clf.fit(X_train[:, sel], y_train)
        f1_k = float(f1_score(y_val, clf.predict(X_val[:, sel])))
        filas.append({"K": k, "f1": round(f1_k, 4), "pct_reduccion": round(100 * (1 - k / n_original), 2),
                      "pct_f1_conservado": round(100 * f1_k / f1_full, 2), "tiempo_s": round(time.time() - t0, 1)})
        print(f"K={k:4d} | F1={f1_k:.4f} | reducción={filas[-1]['pct_reduccion']:.1f}% | "
              f"F1 conservado={filas[-1]['pct_f1_conservado']:.1f}%")
        del clf
        gc.collect()
    df_curva = pd.DataFrame(filas)
    df_curva.to_csv(RUTA_CURVA, index=False)

    # Regla de selección: menor K con F1 conservado >= meta; si ninguno cumple, el K con mayor F1
    cumplen = df_curva[df_curva["pct_f1_conservado"] >= CFG["META_F1_CONSERVADO"]]
    K_FINAL = int(cumplen["K"].min()) if len(cumplen) else int(df_curva.loc[df_curva["f1"].idxmax(), "K"])
    columnas_seleccionadas = ranking_sin_redundancia[:K_FINAL]
    guardar_json({"K": K_FINAL, "columnas": columnas_seleccionadas,
                  "nombres": [nombre_legible(c) for c in columnas_seleccionadas],
                  "grupos": [grupo_de(c) for c in columnas_seleccionadas],
                  "regla": f"menor K con >= {CFG['META_F1_CONSERVADO']}% del F1 de referencia" if len(cumplen)
                           else "ningún K alcanzó la meta; se tomó el K de mayor F1"}, RUTA_SELECCION)

fila_k = df_curva[df_curva["K"] == K_FINAL].iloc[0]
print(f"\nK seleccionado = {K_FINAL}: F1={fila_k['f1']:.4f} ({fila_k['pct_f1_conservado']:.1f}% del F1 de referencia "
      f"{f1_full:.4f}), reducción de dimensionalidad = {fila_k['pct_reduccion']:.1f}%")
print("Características seleccionadas:")
for c in columnas_seleccionadas:
    print(f"  {c:>6} -> {nombre_legible(c)}")

# Evidencia: tabla de las K características seleccionadas (posición, nombre, grupo, importancia, ranking Extra Trees)
tabla_seleccion = pd.DataFrame({
    "posicion": np.arange(1, K_FINAL + 1), "columna": columnas_seleccionadas,
    "nombre": [nombre_legible(c) for c in columnas_seleccionadas], "grupo": [grupo_de(c) for c in columnas_seleccionadas],
    "importancia_extratrees": [float(importancias.get(c, np.nan)) for c in columnas_seleccionadas],
    "posicion_ranking_extratrees": [int(list(importancias.index).index(c)) + 1 if c in importancias.index else -1
                                    for c in columnas_seleccionadas]})
guardar_csv(tabla_seleccion, os.path.join(RUTAS["fase1"], "caracteristicas_seleccionadas.csv"))

# Evidencia: matriz de correlación del subconjunto seleccionado (verifica que quedó sin redundancia fuerte)
idx_sel = [columnas_limpias.index(c) for c in columnas_seleccionadas]
corr_sel = np.nan_to_num(np.corrcoef(X_train[:, idx_sel].astype(np.float64).T))  # nan solo si una columna fuera constante en el 75 %
etiquetas_sel = [f"{c} · {nombre_legible(c)[:28]}" for c in columnas_seleccionadas]
guardar_csv(pd.DataFrame(corr_sel, index=etiquetas_sel, columns=columnas_seleccionadas).round(4),
            os.path.join(RUTAS["fase1"], "correlacion_seleccionadas.csv"), index=True)
fig, ax = plt.subplots(figsize=(max(6, 0.32 * K_FINAL), max(5, 0.3 * K_FINAL)))
im = ax.imshow(corr_sel, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(K_FINAL)); ax.set_xticklabels(columnas_seleccionadas, rotation=90, fontsize=6)
ax.set_yticks(range(K_FINAL)); ax.set_yticklabels(etiquetas_sel, fontsize=6)
fig.colorbar(im, ax=ax, fraction=0.03, label="Correlación de Pearson")
ax.set_title(f"Correlación entre las {K_FINAL} características seleccionadas (máx. |r| fuera de la diagonal = "
             f"{np.abs(corr_sel - np.eye(K_FINAL)).max():.2f})", fontsize=9)
guardar_figura(fig, "fig_correlacion_seleccionadas.png")
max_corr_sel = float(np.abs(corr_sel - np.eye(K_FINAL)).max())

# ----------------------------------------------------------------------------
# 5) Datasets reducidos (K columnas + metadatos) para las fases siguientes: escritos por bloques
# ----------------------------------------------------------------------------
def reducir(parquets, salida):
    """Escribe salida.parquet con las K columnas + metadatos, leyendo por bloques y sin duplicados por sha256.
    En RAM solo vive un bloque a la vez (LOTE*5 filas x (K + 8) columnas)."""
    if esta_listo(salida):
        n = pq.ParquetFile(salida).metadata.num_rows
        print(f"ya existe: {os.path.basename(salida)} ({n:,} filas)")
        return {"filas": int(n), "duplicados_eliminados": None, "no_finitos_corregidos": None}
    escritor, vistos, n_filas, n_dup, n_nf = None, set(), 0, 0, 0
    for p in parquets:
        for lote in pq.ParquetFile(p).iter_batches(batch_size=LOTE * 5, columns=columnas_seleccionadas + COLS_META):
            tabla = pa.Table.from_batches([lote])
            sha = tabla.column("sha256").to_pylist()
            nuevos = []
            for i, h in enumerate(sha):
                if h in vistos:
                    n_dup += 1
                else:
                    vistos.add(h)
                    nuevos.append(i)
            if not nuevos:
                continue
            if len(nuevos) < len(sha):
                tabla = tabla.take(pa.array(nuevos, pa.int64()))
            for c in columnas_seleccionadas:
                v = tabla.column(c).to_numpy(zero_copy_only=False)
                no_fin = ~np.isfinite(v)
                if no_fin.any():
                    n_nf += int(no_fin.sum())
                    v = np.where(no_fin, 0.0, v).astype(np.float32)
                    tabla = tabla.set_column(tabla.schema.get_field_index(c), c, pa.array(v, pa.float32()))
            if escritor is None:
                escritor = pq.ParquetWriter(salida, tabla.schema, compression="snappy")
            escritor.write_table(tabla)
            n_filas += tabla.num_rows
            del tabla, lote
    if escritor is None:   # sin filas (p. ej. conjunto de desafío vacío): parquet vacío con el mismo esquema
        esquema_vacio = pq.ParquetFile(parquets[0]).schema_arrow
        esquema_vacio = pa.schema([esquema_vacio.field(c) for c in columnas_seleccionadas + COLS_META])
        pq.write_table(esquema_vacio.empty_table(), salida)
    else:
        escritor.close()
    del vistos
    gc.collect()
    devolver_memoria_al_sistema()
    marcar_listo(salida)
    print(f"guardado {os.path.basename(salida)}: {n_filas:,} filas x {len(columnas_seleccionadas)} características "
          f"(+{len(COLS_META)} metadatos) | duplicados: {n_dup} | no finitos corregidos: {n_nf}")
    return {"filas": int(n_filas), "duplicados_eliminados": int(n_dup), "no_finitos_corregidos": int(n_nf)}

info_reducidos = {
    "train": reducir(PARQUETS_TRAIN, os.path.join(RUTAS["fase1"], "train_reducido.parquet")),
    "test": reducir(PARQUETS_TEST, os.path.join(RUTAS["fase1"], "test_reducido.parquet")),
    "challenge": reducir([PARQUET_CHALLENGE], os.path.join(RUTAS["fase1"], "challenge_reducido.parquet")),
}
memoria_ram("fase1b tras datasets reducidos")

# ----------------------------------------------------------------------------
# 6) Figuras de la fase
# ----------------------------------------------------------------------------
resumen_datos = cargar_json(os.path.join(RUTAS["datos"], "resumen_datos.json"))
comp = pd.DataFrame(resumen_datos["por_split_y_tipo"])
fig, ax = plt.subplots(figsize=(9, 4.8))
splits = ["train", "test", "challenge"]
tipos = CFG["TIPOS_PE"]
ancho = 0.25
for i, s in enumerate(splits):
    sub = comp[comp["split"] == s].set_index("file_type").reindex(tipos).fillna(0)
    ben = sub.get("benignos", pd.Series(0, index=tipos)).to_numpy()
    mal = sub.get("maliciosos", pd.Series(0, index=tipos)).to_numpy()
    x = np.arange(len(tipos)) + (i - 1) * ancho
    ax.bar(x, ben, ancho, color="#4C78A8", label="Benignos" if i == 0 else None)
    ax.bar(x, mal, ancho, bottom=ben, color="#E45756", label="Maliciosos" if i == 0 else None)
    for xi, tot in zip(x, ben + mal):
        ax.text(xi, tot, f"{int(tot):,}\n{s}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(np.arange(len(tipos)))
ax.set_xticklabels([t.replace("Dot_Net", ".NET") for t in tipos])
ax.set_ylabel("Número de archivos únicos")
ax.set_title("Composición del subconjunto PE de EMBER2024 usado en el proyecto (archivos únicos por SHA-256)", fontsize=10)
ax.legend()
ax.grid(axis="y", alpha=0.3)
guardar_figura(fig, "fig_composicion_dataset.png")

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(df_curva["K"], df_curva["pct_f1_conservado"], marker="o", color="#1F3864", label="% del F1 de referencia conservado")
ax.axhline(CFG["META_F1_CONSERVADO"], color="#C00000", linestyle="--", linewidth=1, label=f"Meta: {CFG['META_F1_CONSERVADO']:.0f} %")
ax.axvline(K_FINAL, color="#2E7D32", linestyle=":", linewidth=1.2, label=f"K seleccionado = {K_FINAL}")
ax.set_xlabel("Número de características conservadas (K)")
ax.set_ylabel("% del F1 de referencia")
ax.set_title(f"Curva de reducción de dimensionalidad vs. desempeño (F1 referencia = {f1_full:.4f})")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
guardar_figura(fig, "fig_curva_k_vs_f1.png")

ranking_info = cargar_json(RUTA_RANKING)
imp = pd.Series(ranking_info["importancia"])
imp_sel = imp.reindex(columnas_seleccionadas).sort_values()
fig, ax = plt.subplots(figsize=(8, max(4, 0.28 * len(imp_sel))))
colores = pd.Series(GRUPO_DE_CARACTERISTICA).drop_duplicates().tolist()
paleta = dict(zip(colores, plt.cm.tab20(np.linspace(0, 1, len(colores)))))
ax.barh([f"{c} · {nombre_legible(c)[:38]}" for c in imp_sel.index], imp_sel.values,
        color=[paleta[grupo_de(c)] for c in imp_sel.index])
ax.set_xlabel("Importancia (Extra Trees, impureza de Gini)")
ax.set_title(f"Las {K_FINAL} características seleccionadas y su grupo de origen")
from matplotlib.patches import Patch
grupos_presentes = sorted(set(grupo_de(c) for c in imp_sel.index), key=colores.index)
ax.legend(handles=[Patch(color=paleta[g], label=DESCRIPCION_GRUPOS[g][0]) for g in grupos_presentes], fontsize=8, loc="lower right")
ax.grid(axis="x", alpha=0.3)
guardar_figura(fig, "fig_importancia_topk.png")

# Aporte de cada grupo al subconjunto seleccionado (para el análisis del capítulo)
grupos_sel = pd.Series([grupo_de(c) for c in columnas_seleccionadas]).value_counts()
guardar_csv(pd.DataFrame({"grupo": grupos_sel.index, "nombre_grupo": [DESCRIPCION_GRUPOS[g][0] for g in grupos_sel.index],
                          "n_seleccionadas": grupos_sel.values,
                          "n_en_el_vector": [dict(GRUPOS_CARACTERISTICAS)[g] for g in grupos_sel.index]}),
            os.path.join(RUTAS["fase1"], "grupos_en_seleccion.csv"))
resumen_fase1 = {
    "n_train_muestreado_fase1a": int(n_train_total), "n_muestra_seleccion": int(len(y)),
    "n_muestra_entrenamiento": int(len(y_train)), "n_muestra_validacion": int(len(y_val)),
    "distribucion_muestra": distribucion, "duplicados_eliminados": int(n_dup), "no_finitos_corregidos": int(n_no_finitos),
    "n_caracteristicas_originales": n_original, "columnas_constantes_eliminadas": len(cols_constantes),
    "n_caracteristicas_limpias": n_limpio, "candidatos_tras_correlacion": len(ranking_sin_redundancia),
    "f1_referencia_todas": round(f1_full, 4), "K": K_FINAL, "f1_K": float(fila_k["f1"]),
    "pct_f1_conservado": float(fila_k["pct_f1_conservado"]), "pct_reduccion": float(fila_k["pct_reduccion"]),
    "cumple_meta_f1_conservado": bool(fila_k["pct_f1_conservado"] >= CFG["META_F1_CONSERVADO"]),
    "cumple_meta_reduccion": bool(fila_k["pct_reduccion"] >= CFG["METAS"]["reduccion_dimensionalidad_minima_pct"]),
    "max_correlacion_abs_entre_seleccionadas": round(max_corr_sel, 4),
    "columnas_seleccionadas": columnas_seleccionadas,
    "nombres_seleccionados": [nombre_legible(c) for c in columnas_seleccionadas],
    "grupos_en_seleccion": grupos_sel.to_dict(), "curva": df_curva.to_dict(orient="records"),
    "datasets_reducidos": info_reducidos,
    "tiempo_ranking_s": round(tiempo_ranking, 1), "tiempo_fase_s": round(time.time() - t_fase, 1),
}
guardar_json(resumen_fase1, RUTA_RESUMEN_F1)
print("\nResumen Fase 1 guardado en", RUTA_RESUMEN_F1)
print(json.dumps({k: v for k, v in resumen_fase1.items() if k not in ("columnas_seleccionadas", "nombres_seleccionados", "curva")},
                 indent=2, ensure_ascii=False))

# Liberar la memoria de la fase (todo lo necesario quedó en Drive)
liberar("X_train", "X_val", "y_train", "y_val", "y", "minimos", "maximos", "importancias", "corr_sel", "imp", "imp_sel", "comp")
memoria_ram("fase1b fin")


## Fase 2 — Entrenamiento, comparación y selección del modelo (OE2)
Entrena **Random Forest**, **XGBoost** y una **red neuronal densa** con búsqueda aleatoria de hiperparámetros (`RandomizedSearchCV`, 5 combinaciones) y validación cruzada estratificada de 3 particiones. La evaluación final se hace sobre la **prueba temporal** (archivos posteriores al entrenamiento, nunca vistos) y sobre el **conjunto de desafío** (malware evasivo), reportando accuracy, precision, recall, F1, AUC-ROC, PR-AUC, FPR, FNR, TPR al 1 % de FPR, desempeño por tipo de archivo y por semana, y tiempos de entrenamiento e inferencia. La selección del modelo sigue una regla explícita: cumplir la meta de FPR (≤ 5 %) y, entre los que cumplen, el mayor recall (desempate por F1 y latencia).

**Evidencias (carpeta `fase2/`):** resultados de la búsqueda de hiperparámetros de cada modelo, historia de entrenamiento de la red, tabla comparativa completa, **matrices de confusión**, reportes de clasificación por clase, puntos de las curvas ROC y precisión-recall, métricas por umbral, por tipo de archivo y por semana, y las **predicciones por archivo** (sha256 + probabilidad de cada modelo) sobre la prueba temporal y el desafío.

⚠️ Entre 45 y 90 minutos (corrida 1: 55 min, de los cuales 46 fueron del Random Forest). Cada modelo se guarda (comprimido) en Drive apenas termina. Cambio posterior a la corrida 1: la partición de validación de la red neuronal ahora es estratificada y barajada (antes Keras tomaba las últimas filas, ordenadas por formato; solo afectaba a la parada temprana de la red). **Memoria:** solo se cargan las K columnas necesarias, la búsqueda mantiene un modelo en memoria a la vez (el paralelismo va por hilos dentro de cada modelo) y los tres modelos se liberan al final de la celda.

In [ ]:
"""
FASE 2 - ENTRENAMIENTO, COMPARACIÓN Y SELECCIÓN DEL MODELO

Qué hace:
  1. Carga los datasets reducidos (K características) de la Fase 1 y ajusta un
     StandardScaler SOLO con el entrenamiento (se guarda para la interfaz).
  2. Entrena tres familias de modelos con búsqueda aleatoria de hiperparámetros y
     validación cruzada estratificada (N_FOLDS): Random Forest, XGBoost y una red
     neuronal densa (Keras). Cada modelo se guarda apenas termina (checkpoint).
  3. Evalúa sobre el conjunto de PRUEBA TEMPORAL de EMBER2024 (archivos 12 semanas
     posteriores a los de entrenamiento, nunca vistos) y sobre el CONJUNTO DE
     DESAFÍO (malware evasivo: muestras que ningún antivirus detectó al inicio),
     replicando el protocolo del artículo de EMBER2024 (challenge + benignos de prueba).
  4. Métricas: accuracy, precision, recall, F1, AUC-ROC, PR-AUC, FPR, FNR, TPR al
     1 % de FPR, F1 por tipo de archivo y por semana (deriva temporal), tiempos.
  5. Selecciona el modelo con una regla explícita (ver seleccionar_modelo) y guarda
     todo en fase2_resumen.json y en figuras para el documento.

Control de memoria: solo se leen las K columnas + los metadatos necesarios (no las
cadenas family/behavior de 830.000 filas); los DataFrames se liberan apenas se
construyen las matrices float32; la búsqueda de hiperparámetros corre con UN
modelo en memoria a la vez (el paralelismo va dentro del modelo, por hilos, y no
en procesos que duplicarían los datos); los modelos se guardan comprimidos y se
liberan al final de la fase (las fases siguientes los cargan desde Drive).

Evidencias que deja en fase2/: resultados de la búsqueda de hiperparámetros por
modelo, historia de entrenamiento de la red, tabla comparativa completa, matrices
de confusión, reportes de clasificación, puntos de las curvas ROC y PR, métricas
por umbral, por tipo de archivo y por semana, y las predicciones por archivo
(sha256 + probabilidad de cada modelo) sobre la prueba temporal y el desafío.
"""
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
                             classification_report)
import xgboost as xgb

seleccion = cargar_json(os.path.join(RUTAS["fase1"], "seleccion_final.json"))
COLS = seleccion["columnas"]
NOMBRES_COLS = [nombre_legible(c) for c in COLS]
K_FINAL = len(COLS)
t_fase = time.time()
memoria_ram("fase2 inicio")

# Solo las columnas necesarias (las K características + metadatos de evaluación); nada de cadenas pesadas
df_train = pd.read_parquet(os.path.join(RUTAS["fase1"], "train_reducido.parquet"), columns=COLS + ["label"])
df_test = pd.read_parquet(os.path.join(RUTAS["fase1"], "test_reducido.parquet"), columns=COLS + ["sha256", "label", "file_type", "semana"])
df_chal = pd.read_parquet(os.path.join(RUTAS["fase1"], "challenge_reducido.parquet"), columns=COLS + ["sha256", "label", "file_type"])
y_train = df_train["label"].to_numpy().astype(int)
y_test = df_test["label"].to_numpy().astype(int)
meta_test = df_test[["sha256", "file_type", "semana"]].reset_index(drop=True)
meta_chal = df_chal[["sha256", "file_type"]].reset_index(drop=True)
print(f"Train: {df_train.shape[0]:,} filas | Test temporal: {df_test.shape[0]:,} | Challenge: {df_chal.shape[0]:,} | K = {K_FINAL}")
print(f"Balance de clases (train): maliciosos = {y_train.mean():.3f} | (test): {y_test.mean():.3f}")

scaler = StandardScaler().fit(df_train[COLS].to_numpy(dtype=np.float32))
joblib.dump(scaler, os.path.join(RUTAS["modelos"], "scaler.joblib"))
X_train = scaler.transform(df_train[COLS].to_numpy(dtype=np.float32)).astype(np.float32)
X_test = scaler.transform(df_test[COLS].to_numpy(dtype=np.float32)).astype(np.float32)
X_chal = scaler.transform(df_chal[COLS].to_numpy(dtype=np.float32)).astype(np.float32)
y_chal = np.ones(len(X_chal), dtype=int)
peso_positivo = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))  # para XGBoost si hay desbalance
del df_train, df_test, df_chal
gc.collect()
tipo_test, semana_test, tipo_chal = meta_test["file_type"].to_numpy(), meta_test["semana"].to_numpy(), meta_chal["file_type"].to_numpy()
memoria_ram("fase2 matrices listas (DataFrames liberados)")

cv = StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["RANDOM_STATE"])
RUTA_MODELOS = RUTAS["modelos"]


def tamano_mb(ruta):
    return round(os.path.getsize(ruta) / 2 ** 20, 2) if os.path.exists(ruta) else None


def con_checkpoint(nombre, entrenar):
    """Carga el modelo si ya existe en Drive; si no, lo entrena y lo guarda junto con su información y la
    tabla de resultados de la búsqueda de hiperparámetros (evidencia fase2/busqueda_<modelo>.csv)."""
    ruta_info = os.path.join(RUTA_MODELOS, f"{nombre}_info.json")
    ruta_modelo = os.path.join(RUTA_MODELOS, f"{nombre}.joblib")
    ruta_keras = ruta_modelo.replace(".joblib", ".keras")
    if os.path.exists(ruta_info) and (os.path.exists(ruta_modelo) or os.path.exists(ruta_keras)):
        info = cargar_json(ruta_info)
        print(f"[{nombre}] cargado desde checkpoint: {info['mejores_hiperparametros']}")
        if nombre == "dnn":
            from tensorflow import keras
            return keras.models.load_model(ruta_keras), info
        return joblib.load(ruta_modelo), info
    t0 = time.time()
    modelo, params, f1_cv, tabla_busqueda = entrenar()
    info = {"mejores_hiperparametros": params, "f1_cv": None if f1_cv is None else round(float(f1_cv), 4),
            "tiempo_entrenamiento_s": round(time.time() - t0, 1)}
    if nombre == "dnn":
        modelo.save(ruta_keras)
        info["archivo_modelo"] = os.path.basename(ruta_keras)
        info["tamano_modelo_mb"] = tamano_mb(ruta_keras)
    else:
        if isinstance(modelo, RandomForestClassifier):
            modelo.set_params(n_jobs=1)   # el paralelismo por hilos solo se usó para entrenar; la interfaz predice de a un archivo
        joblib.dump(modelo, ruta_modelo, compress=3)   # comprimido: un Random Forest sin comprimir puede pesar varios GB
        info["archivo_modelo"] = os.path.basename(ruta_modelo)
        info["tamano_modelo_mb"] = tamano_mb(ruta_modelo)
        if hasattr(modelo, "estimators_"):
            info["n_arboles"] = len(modelo.estimators_)
            info["n_nodos_total"] = int(sum(e.tree_.node_count for e in modelo.estimators_))
    if tabla_busqueda is not None:
        guardar_csv(tabla_busqueda, os.path.join(RUTAS["fase2"], f"busqueda_hiperparametros_{nombre}.csv"))
    guardar_json(info, ruta_info)
    print(f"[{nombre}] entrenado en {info['tiempo_entrenamiento_s']:.0f} s | mejores hiperparámetros: {params} | "
          f"F1 CV: {info['f1_cv']} | tamaño en disco: {info['tamano_modelo_mb']} MB")
    return modelo, info


def tabla_cv(busqueda):
    """cv_results_ de RandomizedSearchCV en una tabla ordenada por F1 medio (evidencia de la búsqueda)."""
    r = pd.DataFrame(busqueda.cv_results_)
    cols = [c for c in r.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score", "rank_test_score", "mean_fit_time"]
    return r[cols].sort_values("rank_test_score").reset_index(drop=True)


# --- Random Forest ---
def entrenar_rf():
    # n_jobs de la búsqueda = 1 (un modelo en memoria a la vez); el paralelismo va dentro del bosque (hilos, sin copiar datos)
    busqueda = RandomizedSearchCV(
        RandomForestClassifier(random_state=CFG["RANDOM_STATE"], n_jobs=CFG["N_JOBS"], class_weight="balanced"),
        param_distributions={"n_estimators": [150, 250], "max_depth": [10, 15, 20, 25],
                             "min_samples_leaf": [2, 4, 6], "max_features": ["sqrt", 0.5]},
        n_iter=CFG["N_ITER_BUSQUEDA"], scoring="f1", cv=cv, random_state=CFG["RANDOM_STATE"], n_jobs=1, verbose=1)
    busqueda.fit(X_train, y_train)
    return busqueda.best_estimator_, busqueda.best_params_, busqueda.best_score_, tabla_cv(busqueda)

# --- XGBoost ---
def entrenar_xgb():
    busqueda = RandomizedSearchCV(
        xgb.XGBClassifier(random_state=CFG["RANDOM_STATE"], n_jobs=CFG["N_JOBS"], eval_metric="logloss",
                          scale_pos_weight=peso_positivo, tree_method="hist"),
        param_distributions={"n_estimators": [150, 250, 400], "max_depth": [4, 6, 8], "learning_rate": [0.05, 0.1, 0.2],
                             "subsample": [0.8, 1.0], "colsample_bytree": [0.7, 1.0]},
        n_iter=CFG["N_ITER_BUSQUEDA"], scoring="f1", cv=cv, random_state=CFG["RANDOM_STATE"], n_jobs=1, verbose=1)
    busqueda.fit(X_train, y_train)
    return busqueda.best_estimator_, busqueda.best_params_, busqueda.best_score_, tabla_cv(busqueda)

# --- Red neuronal densa ---
def entrenar_dnn():
    import tensorflow as tf
    from tensorflow import keras
    tf.random.set_seed(CFG["RANDOM_STATE"])
    modelo = keras.Sequential([
        keras.layers.Input(shape=(X_train.shape[1],)),
        keras.layers.Dense(128, activation="relu"), keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation="relu"), keras.layers.Dropout(0.2),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy",
                   metrics=[keras.metrics.AUC(name="auc"), "accuracy"])
    parada = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True)
    pesos = {0: 1.0, 1: peso_positivo}
    epocas = int(CFG.get("EPOCAS_DNN", 100))
    # Partición de validación (20 %) estratificada y barajada. (validation_split de Keras toma las ÚLTIMAS filas sin
    # barajar y, como el conjunto reducido está ordenado por formato, en la corrida 1 la validación quedó sesgada hacia
    # Win64/.NET; solo afectó al criterio de parada temprana de la red, no a la evaluación sobre la prueba temporal.)
    from sklearn.model_selection import train_test_split
    idx_tr, idx_va = train_test_split(np.arange(len(y_train)), test_size=0.2, random_state=CFG["RANDOM_STATE"], stratify=y_train)
    historia = modelo.fit(X_train[idx_tr], y_train[idx_tr], validation_data=(X_train[idx_va], y_train[idx_va]),
                          epochs=epocas, batch_size=1024, callbacks=[parada], class_weight=pesos, verbose=0)
    del idx_tr, idx_va
    params = {"arquitectura": "128-64-32-1 (ReLU, dropout 0.3/0.2)", "optimizador": "Adam lr=1e-3",
              "batch_size": 1024, "epocas_efectivas": int(len(historia.history["loss"])),
              "early_stopping": "val_auc, paciencia 8", "validacion": "20 % estratificada y barajada (semilla 42)"}
    # Evidencia: historia de entrenamiento por época (pérdida y AUC en entrenamiento y validación)
    tabla_hist = pd.DataFrame(historia.history)
    tabla_hist.insert(0, "epoca", np.arange(1, len(tabla_hist) + 1))
    guardar_csv(tabla_hist, os.path.join(RUTAS["fase2"], "historia_entrenamiento_dnn.csv"))
    return modelo, params, None, None

print("\n=== Random Forest ===")
rf, info_rf = con_checkpoint("random_forest", entrenar_rf)
memoria_ram("fase2 tras Random Forest")
print("\n=== XGBoost ===")
xg, info_xgb = con_checkpoint("xgboost", entrenar_xgb)
memoria_ram("fase2 tras XGBoost")
print("\n=== Red neuronal densa ===")
dnn, info_dnn = con_checkpoint("dnn", entrenar_dnn)
memoria_ram("fase2 tras red neuronal")

MODELOS = {"Random Forest": (rf, info_rf), "XGBoost": (xg, info_xgb), "Red Neuronal Densa": (dnn, info_dnn)}
CLAVE_MODELO = {"Random Forest": "random_forest", "XGBoost": "xgboost", "Red Neuronal Densa": "dnn"}


def probabilidades(modelo, X):
    if hasattr(modelo, "predict_proba"):
        return modelo.predict_proba(X)[:, 1]
    return modelo.predict(X, batch_size=4096, verbose=0).ravel()


def tpr_a_fpr_objetivo(y_true, proba, fpr_objetivo=0.01):
    fpr, tpr, umbrales = roc_curve(y_true, proba)
    ok = np.where(fpr <= fpr_objetivo)[0]
    i = ok[-1] if len(ok) else 0
    return float(tpr[i]), float(umbrales[i])


def submuestrear_curva(*arreglos, maximo=3000):
    """Reduce los puntos de una curva (ROC/PR) a lo sumo `maximo`, conservando extremos (evidencia compacta)."""
    n = len(arreglos[0])
    if n <= maximo:
        return arreglos
    idx = np.unique(np.concatenate([[0, n - 1], np.linspace(0, n - 1, maximo).astype(int)]))
    return tuple(a[idx] for a in arreglos)


def evaluar(nombre, modelo, info):
    t0 = time.time()
    p_test = probabilidades(modelo, X_test)
    ms_lote = 1000 * (time.time() - t0) / len(X_test)
    # latencia de un solo archivo (lo que verá el usuario en la interfaz): mediana de 50 predicciones unitarias
    lat = []
    for i in range(50):
        t1 = time.time()
        probabilidades(modelo, X_test[i:i + 1])
        lat.append(1000 * (time.time() - t1))
    pred = (p_test >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    tpr1, umbral1 = tpr_a_fpr_objetivo(y_test, p_test, 0.01)
    # Conjunto de desafío: malware evasivo + benignos del set de prueba (protocolo EMBER2024)
    p_chal = probabilidades(modelo, X_chal)
    p_ben = p_test[y_test == 0]
    y_mix = np.concatenate([np.zeros(len(p_ben), int), np.ones(len(p_chal), int)])
    p_mix = np.concatenate([p_ben, p_chal])
    res = {
        "modelo": nombre, "accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred), "auc_roc": roc_auc_score(y_test, p_test),
        "pr_auc": average_precision_score(y_test, p_test), "fpr": fp / (fp + tn), "fnr": fn / (fn + tp),
        "tpr_al_1pct_fpr": tpr1, "umbral_1pct_fpr": umbral1, "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "recall_challenge": float((p_chal >= 0.5).mean()), "auc_challenge": roc_auc_score(y_mix, p_mix),
        "pr_auc_challenge": average_precision_score(y_mix, p_mix),
        "tiempo_entrenamiento_s": info["tiempo_entrenamiento_s"], "inferencia_ms_por_archivo_lote": ms_lote,
        "latencia_ms_un_archivo_mediana": float(np.median(lat)), "latencia_ms_un_archivo_p95": float(np.percentile(lat, 95)),
        "f1_cv": info.get("f1_cv"), "hiperparametros": info["mejores_hiperparametros"],
        "tamano_modelo_mb": info.get("tamano_modelo_mb"),
    }
    # Desempeño por tipo de archivo y por semana (deriva temporal)
    res["f1_por_tipo"] = {t: float(f1_score(y_test[m], pred[m])) for t in CFG["TIPOS_PE"] if (m := (tipo_test == t)).sum() > 0}
    res["f1_por_semana"] = {int(s): float(f1_score(y_test[semana_test == s], pred[semana_test == s]))
                            for s in sorted(set(semana_test)) if (semana_test == s).sum() >= 30}
    res["recall_challenge_por_tipo"] = {t: float((p_chal[m] >= 0.5).mean()) for t in CFG["TIPOS_PE"] if (m := (tipo_chal == t)).sum() > 0}
    return res, p_test, p_chal


resultados, probas_test, probas_chal = {}, {}, {}
for nombre, (modelo, info) in MODELOS.items():
    resultados[nombre], probas_test[nombre], probas_chal[nombre] = evaluar(nombre, modelo, info)

columnas_tabla = ["modelo", "accuracy", "precision", "recall", "f1", "auc_roc", "pr_auc", "fpr", "fnr", "tpr_al_1pct_fpr",
                  "recall_challenge", "auc_challenge", "tiempo_entrenamiento_s", "inferencia_ms_por_archivo_lote",
                  "latencia_ms_un_archivo_mediana"]
df_res = pd.DataFrame([{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items() if k in columnas_tabla}
                       for r in resultados.values()])[columnas_tabla]
df_res.to_csv(os.path.join(RUTAS["fase2"], "fase2_comparacion_modelos.csv"), index=False)
print("\n=== COMPARACIÓN (prueba temporal de EMBER2024, umbral 0.5) ===")
print(df_res.to_string(index=False))

# ----------------------------------------------------------------------------
# Evidencias de la evaluación (todas identificadas por modelo y listas para descargar)
# ----------------------------------------------------------------------------
# a) Tabla comparativa COMPLETA (todas las métricas escalares, incluida la matriz de confusión y el desafío)
escalares = [k for k in resultados["Random Forest"] if not isinstance(resultados["Random Forest"][k], dict)]
guardar_csv(pd.DataFrame([{k: (round(v, 6) if isinstance(v, float) else v) for k, v in r.items() if k in escalares}
                          for r in resultados.values()]), os.path.join(RUTAS["fase2"], "fase2_comparacion_modelos_completa.csv"))

# b) Matrices de confusión (conteos y porcentajes) sobre la prueba temporal
guardar_csv(pd.DataFrame([{"modelo": n, "verdaderos_negativos": r["tn"], "falsos_positivos": r["fp"], "falsos_negativos": r["fn"],
                           "verdaderos_positivos": r["tp"], "n_test": r["tn"] + r["fp"] + r["fn"] + r["tp"],
                           "fpr_pct": round(100 * r["fpr"], 4), "fnr_pct": round(100 * r["fnr"], 4),
                           "accuracy_pct": round(100 * r["accuracy"], 4)} for n, r in resultados.items()]),
            os.path.join(RUTAS["fase2"], "matrices_confusion.csv"))

# c) Reporte de clasificación por clase (precision / recall / F1 / soporte) por modelo
filas_rep = []
for n in resultados:
    rep = classification_report(y_test, (probas_test[n] >= 0.5).astype(int), target_names=["benigno", "malicioso"], output_dict=True, zero_division=0)
    for clase, vals in rep.items():
        if isinstance(vals, dict):
            filas_rep.append({"modelo": n, "clase": clase, **{k: round(float(v), 6) for k, v in vals.items()}})
        else:
            filas_rep.append({"modelo": n, "clase": clase, "precision": None, "recall": None, "f1-score": round(float(vals), 6), "support": len(y_test)})
guardar_csv(pd.DataFrame(filas_rep), os.path.join(RUTAS["fase2"], "reportes_clasificacion.csv"))

# d) Puntos de las curvas ROC y precisión-recall (submuestreados a 3.000 puntos por modelo)
filas_roc, filas_pr = [], []
for n in resultados:
    fpr_c, tpr_c, umb_c = roc_curve(y_test, probas_test[n])
    fpr_c, tpr_c, umb_c = submuestrear_curva(fpr_c, tpr_c, umb_c)
    filas_roc.append(pd.DataFrame({"modelo": n, "fpr": fpr_c, "tpr": tpr_c, "umbral": umb_c}))
    pr_c, rc_c, umb_p = precision_recall_curve(y_test, probas_test[n])
    pr_c, rc_c = submuestrear_curva(pr_c, rc_c)
    filas_pr.append(pd.DataFrame({"modelo": n, "recall": rc_c, "precision": pr_c}))
guardar_csv(pd.concat(filas_roc, ignore_index=True), os.path.join(RUTAS["fase2"], "curvas_roc.csv"))
guardar_csv(pd.concat(filas_pr, ignore_index=True), os.path.join(RUTAS["fase2"], "curvas_precision_recall.csv"))
del filas_roc, filas_pr

# e) Métricas según el umbral de decisión (sustenta la elección del umbral 0,5 y muestra el compromiso FPR/recall)
filas_umb = []
for n in resultados:
    for u in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        pr_u = (probas_test[n] >= u).astype(int)
        tn_u, fp_u, fn_u, tp_u = confusion_matrix(y_test, pr_u).ravel()
        filas_umb.append({"modelo": n, "umbral": u, "precision": round(precision_score(y_test, pr_u, zero_division=0), 6),
                          "recall": round(recall_score(y_test, pr_u), 6), "f1": round(f1_score(y_test, pr_u), 6),
                          "fpr": round(fp_u / max(fp_u + tn_u, 1), 6), "fnr": round(fn_u / max(fn_u + tp_u, 1), 6),
                          "recall_challenge": round(float((probas_chal[n] >= u).mean()), 6)})
guardar_csv(pd.DataFrame(filas_umb), os.path.join(RUTAS["fase2"], "metricas_vs_umbral.csv"))

# f) Métricas por tipo de archivo y por semana (formato largo, con el número de archivos de cada grupo)
filas_tipo, filas_sem = [], []
for n in resultados:
    pr = (probas_test[n] >= 0.5).astype(int)
    for t in CFG["TIPOS_PE"]:
        m = tipo_test == t
        if m.sum() == 0:
            continue
        tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test[m], pr[m], labels=[0, 1]).ravel()
        filas_tipo.append({"modelo": n, "file_type": t, "n": int(m.sum()), "n_maliciosos": int(y_test[m].sum()),
                           "accuracy": round(accuracy_score(y_test[m], pr[m]), 6), "precision": round(precision_score(y_test[m], pr[m], zero_division=0), 6),
                           "recall": round(recall_score(y_test[m], pr[m], zero_division=0), 6), "f1": round(f1_score(y_test[m], pr[m], zero_division=0), 6),
                           "fpr": round(fp_t / max(fp_t + tn_t, 1), 6), "auc_roc": round(roc_auc_score(y_test[m], probas_test[n][m]), 6) if len(set(y_test[m])) > 1 else None,
                           "recall_challenge": resultados[n]["recall_challenge_por_tipo"].get(t)})
    for s in sorted(set(semana_test)):
        m = semana_test == s
        filas_sem.append({"modelo": n, "semana": int(s), "n": int(m.sum()), "n_maliciosos": int(y_test[m].sum()),
                          "f1": round(f1_score(y_test[m], pr[m], zero_division=0), 6), "recall": round(recall_score(y_test[m], pr[m], zero_division=0), 6),
                          "fpr": round(float(((pr[m] == 1) & (y_test[m] == 0)).sum() / max((y_test[m] == 0).sum(), 1)), 6),
                          "incluida_en_figura": bool(m.sum() >= 30)})
guardar_csv(pd.DataFrame(filas_tipo), os.path.join(RUTAS["fase2"], "metricas_por_tipo.csv"))
guardar_csv(pd.DataFrame(filas_sem), os.path.join(RUTAS["fase2"], "metricas_por_semana.csv"))

# g) Predicciones por archivo (sha256 + probabilidad de cada modelo): permite auditar cualquier resultado
pred_test = meta_test.copy()
pred_test["label"] = y_test
for n in resultados:
    pred_test[f"proba_{CLAVE_MODELO[n]}"] = probas_test[n].astype(np.float32)
pred_test.to_parquet(os.path.join(RUTAS["fase2"], "predicciones_test.parquet"), index=False)
print("Tabla guardada:", os.path.join(RUTAS["fase2"], "predicciones_test.parquet"), f"({len(pred_test):,} archivos)")
pred_chal = meta_chal.copy()
for n in resultados:
    pred_chal[f"proba_{CLAVE_MODELO[n]}"] = probas_chal[n].astype(np.float32)
guardar_csv(pred_chal, os.path.join(RUTAS["fase2"], "predicciones_challenge.csv"))
del pred_test, pred_chal


def seleccionar_modelo(resultados, metas):
    """Regla de selección explícita:
    1) Se priorizan los modelos que cumplen la meta de falsos positivos (FPR <= fpr_maximo).
    2) Entre ellos, el de mayor recall (el falso negativo es el error más costoso en detección de malware);
       empates (< 0.005) se resuelven por F1 y luego por latencia.
    3) Si ninguno cumple la meta de FPR, se elige el de mayor F1 y se documenta que la meta no se alcanzó."""
    filas = list(resultados.values())
    cumplen = [r for r in filas if r["fpr"] <= metas["fpr_maximo"]]
    candidatos = cumplen if cumplen else filas
    criterio = "fpr<=meta -> recall -> f1 -> latencia" if cumplen else "ninguno cumple FPR -> mayor F1"
    if cumplen:
        mejor = max(candidatos, key=lambda r: (round(r["recall"], 2), round(r["f1"], 3), -r["latencia_ms_un_archivo_mediana"]))
    else:
        mejor = max(candidatos, key=lambda r: r["f1"])
    return mejor["modelo"], criterio

modelo_seleccionado, criterio = seleccionar_modelo(resultados, CFG["METAS"])
print(f"\nModelo seleccionado: {modelo_seleccionado} (criterio: {criterio})")
guardar_json({"modelo": modelo_seleccionado, "criterio": criterio, "umbral": 0.5, "K": K_FINAL, "columnas": COLS},
             os.path.join(RUTAS["modelos"], "modelo_seleccionado.json"))

# ----------------------------------------------------------------------------
# Figuras
# ----------------------------------------------------------------------------
colores = {"Random Forest": "#1F77B4", "XGBoost": "#FF7F0E", "Red Neuronal Densa": "#2CA02C"}
metricas = [("f1", "F1-score"), ("recall", "Recall"), ("precision", "Precision"), ("auc_roc", "AUC-ROC"), ("recall_challenge", "Recall\nchallenge")]
fig, ax = plt.subplots(figsize=(10, 4.8))
x = np.arange(len(metricas))
for i, (nombre, r) in enumerate(resultados.items()):
    vals = [r[m] for m, _ in metricas]
    b = ax.bar(x + (i - 1) * 0.26, vals, 0.26, color=colores[nombre], label=nombre)
    for xi, v in zip(x + (i - 1) * 0.26, vals):
        ax.text(xi, v + 0.005, f"{v:.3f}", ha="center", fontsize=7)
ax.axhline(CFG["METAS"]["f1_minimo"], color="#C00000", linestyle="--", linewidth=1, label=f"Meta F1 = {CFG['METAS']['f1_minimo']}")
ax.set_xticks(x); ax.set_xticklabels([n for _, n in metricas]); ax.set_ylim(0, 1.08); ax.set_ylabel("Valor")
ax.set_title("Comparación de modelos sobre la prueba temporal y el conjunto de desafío de EMBER2024")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4); ax.grid(axis="y", alpha=0.3)
guardar_figura(fig, "fig_comparacion_modelos.png")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for nombre in resultados:
    fpr, tpr, _ = roc_curve(y_test, probas_test[nombre])
    for ax in axes:
        ax.plot(fpr, tpr, color=colores[nombre], label=f"{nombre} (AUC = {resultados[nombre]['auc_roc']:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=0.8); axes[0].set_title("Curva ROC (escala lineal)")
axes[1].set_xscale("log"); axes[1].set_xlim(1e-4, 1); axes[1].set_ylim(0.5, 1.01); axes[1].set_title("Curva ROC (FPR en escala logarítmica)")
axes[1].axvline(0.01, color="gray", linestyle=":", label="FPR = 1 %")
for ax in axes:
    ax.set_xlabel("Tasa de falsos positivos (FPR)"); ax.set_ylabel("Tasa de verdaderos positivos (recall)"); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower right")
guardar_figura(fig, "fig_roc_modelos.png")

# Curvas precisión-recall (complementan la ROC cuando interesa la clase positiva)
fig, ax = plt.subplots(figsize=(6.5, 4.6))
for nombre in resultados:
    pr_c, rc_c, _ = precision_recall_curve(y_test, probas_test[nombre])
    ax.plot(rc_c, pr_c, color=colores[nombre], label=f"{nombre} (PR-AUC = {resultados[nombre]['pr_auc']:.4f})")
ax.axhline(y_test.mean(), color="gray", linestyle=":", linewidth=0.8, label=f"Proporción de maliciosos = {y_test.mean():.3f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precisión"); ax.set_title("Curvas precisión-recall sobre la prueba temporal"); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower left")
guardar_figura(fig, "fig_pr_modelos.png")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
for ax, (nombre, r) in zip(axes, resultados.items()):
    m = np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]])
    ax.imshow(m, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{m[i, j]:,}\n({100 * m[i, j] / m.sum():.1f} %)", ha="center", va="center",
                    color="white" if m[i, j] > m.max() / 2 else "black", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Benigno", "Malicioso"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["Benigno", "Malicioso"])
    ax.set_xlabel("Predicción"); ax.set_ylabel("Clase real"); ax.set_title(f"{nombre}\nFPR = {100 * r['fpr']:.2f} %, FNR = {100 * r['fnr']:.2f} %", fontsize=9)
fig.suptitle("Matrices de confusión sobre la prueba temporal (umbral 0,5)")
guardar_figura(fig, "fig_matrices_confusion.png")

fig, ax = plt.subplots(figsize=(9, 4.2))
for nombre, r in resultados.items():
    s = sorted(r["f1_por_semana"].items())
    ax.plot([k for k, _ in s], [v for _, v in s], marker="o", color=colores[nombre], label=nombre)
ax.set_xlabel("Semana del conjunto de prueba (semanas desde el 24/09/2023)"); ax.set_ylabel("F1-score")
ax.set_title("Deriva temporal: F1 por semana sobre archivos posteriores al entrenamiento"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
guardar_figura(fig, "fig_f1_por_semana.png")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
tipos = CFG["TIPOS_PE"]; x = np.arange(len(tipos))
for i, (nombre, r) in enumerate(resultados.items()):
    axes[0].bar(x + (i - 1) * 0.26, [r["f1_por_tipo"].get(t, 0) for t in tipos], 0.26, color=colores[nombre], label=nombre)
    axes[1].bar(x + (i - 1) * 0.26, [r["recall_challenge_por_tipo"].get(t, 0) for t in tipos], 0.26, color=colores[nombre], label=nombre)
for ax, titulo in zip(axes, ["F1 por tipo de archivo (prueba temporal)", "Recall por tipo en el conjunto de desafío (malware evasivo)"]):
    ax.set_xticks(x); ax.set_xticklabels([t.replace("Dot_Net", ".NET") for t in tipos]); ax.set_ylim(0, 1.05); ax.set_title(titulo, fontsize=10); ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8)
guardar_figura(fig, "fig_desempeno_por_tipo.png")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
nombres = list(resultados)
axes[0].bar(nombres, [resultados[n]["tiempo_entrenamiento_s"] / 60 for n in nombres], color=[colores[n] for n in nombres])
axes[0].set_ylabel("Minutos"); axes[0].set_title("Tiempo de entrenamiento (incluida la búsqueda de hiperparámetros)", fontsize=9)
axes[1].bar(nombres, [resultados[n]["latencia_ms_un_archivo_mediana"] for n in nombres], color=[colores[n] for n in nombres])
axes[1].set_ylabel("ms"); axes[1].set_title("Latencia de predicción de un archivo (mediana)", fontsize=9)
for ax in axes:
    ax.grid(axis="y", alpha=0.3); ax.tick_params(axis="x", labelsize=8)
guardar_figura(fig, "fig_tiempos_modelos.png")

# Búsqueda de hiperparámetros: F1 medio de validación cruzada de cada combinación probada (evidencia del ajuste)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, clave, nombre in zip(axes, ["random_forest", "xgboost"], ["Random Forest", "XGBoost"]):
    ruta_b = os.path.join(RUTAS["fase2"], f"busqueda_hiperparametros_{clave}.csv")
    if os.path.exists(ruta_b):
        tb = pd.read_csv(ruta_b)
        def _fmt(v):
            return int(v) if isinstance(v, float) and v.is_integer() else v
        etiquetas = [", ".join(f"{c.replace('param_', '')}={_fmt(v)}" for c, v in fila.items() if c.startswith("param_")) for _, fila in tb.iterrows()]
        ax.barh(range(len(tb)), tb["mean_test_score"], xerr=tb["std_test_score"], color=colores[nombre], alpha=0.85)
        ax.set_yticks(range(len(tb))); ax.set_yticklabels(etiquetas, fontsize=6); ax.invert_yaxis()
        ax.set_xlim(max(0, tb["mean_test_score"].min() - 0.02), min(1.0, tb["mean_test_score"].max() + 0.01))
    ax.set_xlabel("F1 medio en validación cruzada (± desv. est.)"); ax.set_title(f"Búsqueda aleatoria de hiperparámetros – {nombre}", fontsize=9); ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
guardar_figura(fig, "fig_busqueda_hiperparametros.png")

ruta_hist = os.path.join(RUTAS["fase2"], "historia_entrenamiento_dnn.csv")
if os.path.exists(ruta_hist):
    th = pd.read_csv(ruta_hist)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].plot(th["epoca"], th["loss"], label="entrenamiento"); axes[0].plot(th["epoca"], th["val_loss"], label="validación (20 %)")
    axes[0].set_title("Pérdida (entropía cruzada binaria)", fontsize=9)
    axes[1].plot(th["epoca"], th["auc"], label="entrenamiento"); axes[1].plot(th["epoca"], th["val_auc"], label="validación (20 %)")
    axes[1].set_title("AUC-ROC", fontsize=9)
    for ax in axes:
        ax.set_xlabel("Época"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.suptitle("Historia de entrenamiento de la red neuronal densa (early stopping sobre val_auc)", fontsize=10)
    fig.tight_layout()
    guardar_figura(fig, "fig_dnn_historia.png")

resumen_fase2 = {
    "K": K_FINAL, "columnas": COLS, "n_train": int(len(y_train)), "n_test": int(len(y_test)), "n_challenge": int(len(y_chal)),
    "balance_train_pct_maliciosos": round(100 * float(y_train.mean()), 2), "balance_test_pct_maliciosos": round(100 * float(y_test.mean()), 2),
    "n_folds": CFG["N_FOLDS"], "n_iter_busqueda": CFG["N_ITER_BUSQUEDA"], "resultados": resultados,
    "modelo_seleccionado": modelo_seleccionado, "criterio_seleccion": criterio,
    "metas": CFG["METAS"],
    "cumplimiento_metas_modelo_seleccionado": {
        "f1": bool(resultados[modelo_seleccionado]["f1"] >= CFG["METAS"]["f1_minimo"]),
        "fpr": bool(resultados[modelo_seleccionado]["fpr"] <= CFG["METAS"]["fpr_maximo"]),
        "latencia": bool(resultados[modelo_seleccionado]["latencia_ms_un_archivo_mediana"] / 1000 <= CFG["METAS"]["latencia_maxima_s"]),
    },
    "tiempo_fase_s": round(time.time() - t_fase, 1),
}
guardar_json(resumen_fase2, os.path.join(RUTAS["fase2"], "fase2_resumen.json"))
print("\nResumen Fase 2 guardado. Cumplimiento de metas del modelo seleccionado:", resumen_fase2["cumplimiento_metas_modelo_seleccionado"])

# Liberar los modelos y matrices de la fase (quedaron guardados en Drive; la Fase 3 carga solo el modelo seleccionado)
liberar("rf", "xg", "dnn", "MODELOS", "X_train", "X_test", "X_chal", "probas_test", "probas_chal", "meta_test", "meta_chal",
        "tipo_test", "semana_test", "tipo_chal", "filas_umb", "filas_tipo", "filas_sem", "filas_rep")
memoria_ram("fase2 fin (modelos liberados)")


## Fase 3 — Integración de explicabilidad (XAI) con SHAP y LIME (OE3)
Explica el modelo seleccionado con **SHAP** (global y local) y contrasta con **LIME**. Además de las figuras, se mide la explicabilidad con indicadores verificables: cobertura de explicaciones, fidelidad (aditividad) de SHAP, acuerdo SHAP–LIME (índice de Jaccard entre los cinco factores principales) y tiempo por explicación. Cada factor se traduce a una **frase en español** según el grupo de la característica y el signo de su aporte; esas frases son las que muestra la interfaz.

**Evidencias (carpeta `fase3/`):** importancia global por característica y por grupo, **valores SHAP de cada archivo de la muestra** (`shap_valores_muestra.parquet`), la muestra explicada archivo por archivo (clase real, probabilidad, riesgo, factores y frase), el acuerdo SHAP–LIME por instancia y las explicaciones de los ejemplos. El modelo y los explicadores se liberan al terminar (la Fase 4 los carga una sola vez dentro del analizador).

In [ ]:
"""
FASE 3 - INTEGRACIÓN DE EXPLICABILIDAD (XAI): SHAP + LIME + LENGUAJE NATURAL

Qué hace, sobre el modelo seleccionado en la Fase 2:
  1. Explicación GLOBAL con SHAP (TreeExplainer para modelos de árboles; KernelExplainer
     si el seleccionado fuera la red neuronal): importancia media |SHAP| por
     característica y por grupo, con nombres legibles.
  2. Explicación LOCAL: valores SHAP por archivo (waterfall) para un malicioso y un
     benigno del set de prueba; LIME para los mismos casos como técnica de contraste.
  3. Evaluación cuantitativa de la explicabilidad: (a) cobertura (fracción de
     predicciones con explicación válida), (b) acuerdo SHAP-LIME (índice de Jaccard
     entre los top-5 factores) sobre N instancias, (c) tiempo por explicación,
     (d) fidelidad de SHAP (la suma de contribuciones + valor base reproduce la salida).
  4. Traducción a lenguaje natural: cada factor se convierte en una frase en español
     comprensible (según el grupo de la característica y el signo de su contribución),
     que es lo que muestra la interfaz de la Fase 4.

Control de memoria: solo se carga el modelo seleccionado (una vez), se leen las K
columnas necesarias, la matriz completa de prueba se libera apenas se extrae la
muestra y todo (modelo, explicadores, matrices) se libera al final de la fase.

Evidencias que deja en fase3/: valores SHAP de cada archivo de la muestra
(shap_valores_muestra.parquet), tabla de la muestra explicada (sha256, clase,
probabilidad, riesgo y factores principales), importancia global por
característica y por grupo, acuerdo SHAP-LIME por instancia y las explicaciones
en lenguaje natural de los ejemplos.
"""
import joblib
import shap
import lime
import lime.lime_tabular

sel_modelo = cargar_json(os.path.join(RUTAS["modelos"], "modelo_seleccionado.json"))
NOMBRE_MODELO = sel_modelo["modelo"]
COLS = sel_modelo["columnas"]
NOMBRES_COLS = [nombre_legible(c) for c in COLS]
scaler = joblib.load(os.path.join(RUTAS["modelos"], "scaler.joblib"))
archivo_modelo = {"Random Forest": "random_forest.joblib", "XGBoost": "xgboost.joblib", "Red Neuronal Densa": "dnn.keras"}[NOMBRE_MODELO]
if NOMBRE_MODELO == "Red Neuronal Densa":
    from tensorflow import keras
    modelo = keras.models.load_model(os.path.join(RUTAS["modelos"], archivo_modelo))
    predecir_proba = lambda X: modelo.predict(np.asarray(X, dtype=np.float32), batch_size=4096, verbose=0).ravel()
else:
    modelo = joblib.load(os.path.join(RUTAS["modelos"], archivo_modelo))
    predecir_proba = lambda X: modelo.predict_proba(np.asarray(X, dtype=np.float32))[:, 1]
print(f"Modelo explicado: {NOMBRE_MODELO} | K = {len(COLS)}")
memoria_ram("fase3 modelo cargado")

df_test = pd.read_parquet(os.path.join(RUTAS["fase1"], "test_reducido.parquet"), columns=COLS + ["sha256", "label", "file_type"])
df_train = pd.read_parquet(os.path.join(RUTAS["fase1"], "train_reducido.parquet"), columns=COLS)
y_test = df_test["label"].to_numpy().astype(int)
X_train_s = scaler.transform(df_train.sample(n=min(20_000, len(df_train)), random_state=CFG["RANDOM_STATE"]).to_numpy(dtype=np.float32)).astype(np.float32)
del df_train

# Muestra estratificada del set de prueba para las explicaciones (solo la muestra se conserva en memoria)
rng = np.random.default_rng(CFG["RANDOM_STATE"])
n_m = min(CFG["N_MUESTRA_SHAP"], len(y_test))
idx_mal = rng.choice(np.where(y_test == 1)[0], n_m // 2, replace=False)
idx_ben = rng.choice(np.where(y_test == 0)[0], n_m - n_m // 2, replace=False)
idx_muestra = np.concatenate([idx_mal, idx_ben])
X_m = scaler.transform(df_test.iloc[idx_muestra][COLS].to_numpy(dtype=np.float32)).astype(np.float32)
y_m = y_test[idx_muestra]
sha_m = df_test["sha256"].to_numpy()[idx_muestra]
tipo_m = df_test["file_type"].to_numpy()[idx_muestra]
del df_test
gc.collect()
p_m = predecir_proba(X_m)

# ----------------------------------------------------------------------------
# 1) Explicador SHAP y valores para la muestra
# ----------------------------------------------------------------------------
t0 = time.time()
if NOMBRE_MODELO == "Red Neuronal Densa":
    fondo = shap.kmeans(X_train_s, 50)
    explainer = shap.KernelExplainer(predecir_proba, fondo)
    n_m = min(300, n_m)  # KernelExplainer es costoso: se acota la muestra
    X_m, y_m, p_m, idx_muestra, sha_m, tipo_m = X_m[:n_m], y_m[:n_m], p_m[:n_m], idx_muestra[:n_m], sha_m[:n_m], tipo_m[:n_m]
    valores_shap = np.asarray(explainer.shap_values(X_m, nsamples=200))
    valor_base = float(np.ravel(explainer.expected_value)[-1])
    escala_shap = "probabilidad"
else:
    explainer = shap.TreeExplainer(modelo)
    # Random Forest: los árboles son muy profundos (decenas de miles de hojas) y el algoritmo exacto de Tree SHAP
    # (O(T·L·D²) por archivo) tardaría horas; se usa la variante aproximada (Saabas), documentada, y se mide su
    # fidelidad más abajo. XGBoost (árboles pequeños) usa el algoritmo exacto.
    APROXIMAR = NOMBRE_MODELO == "Random Forest"
    sv = explainer.shap_values(X_m, approximate=APROXIMAR)
    if isinstance(sv, list):           # versiones antiguas de shap: lista [clase0, clase1]
        valores_shap = np.asarray(sv[1])
    elif np.ndim(sv) == 3:             # versiones recientes con RandomForest: (n, K, 2)
        valores_shap = np.asarray(sv)[:, :, 1]
    else:                              # XGBoost: (n, K) en log-odds
        valores_shap = np.asarray(sv)
    ev = np.ravel(explainer.expected_value)
    valor_base = float(ev[1] if len(ev) > 1 else ev[0])
    escala_shap = "probabilidad" if NOMBRE_MODELO == "Random Forest" else "log-odds"
t_shap_total = time.time() - t0
ms_por_explicacion_shap = 1000 * t_shap_total / len(X_m)
print(f"SHAP calculado para {len(X_m)} archivos en {t_shap_total:.1f} s ({ms_por_explicacion_shap:.2f} ms por archivo); escala: {escala_shap}")

# Fidelidad (aditividad): base + suma de contribuciones debe reproducir la salida del modelo
if escala_shap == "probabilidad":
    reconstruccion = valor_base + valores_shap.sum(axis=1)
    error_fidelidad = float(np.mean(np.abs(reconstruccion - p_m)))
else:
    logit = 1 / (1 + np.exp(-(valor_base + valores_shap.sum(axis=1))))
    error_fidelidad = float(np.mean(np.abs(logit - p_m)))
cobertura = float(np.mean(np.isfinite(valores_shap).all(axis=1) & (np.abs(valores_shap).sum(axis=1) > 0)))
print(f"Cobertura de explicación: {100 * cobertura:.1f} % | error medio de fidelidad SHAP: {error_fidelidad:.5f}")

# ----------------------------------------------------------------------------
# 2) Explicación global (por característica y por grupo)
# ----------------------------------------------------------------------------
imp_global = pd.DataFrame({"columna": COLS, "nombre": NOMBRES_COLS, "grupo": [grupo_de(c) for c in COLS],
                           "shap_medio_abs": np.abs(valores_shap).mean(axis=0),
                           "shap_medio": valores_shap.mean(axis=0)}).sort_values("shap_medio_abs", ascending=False)
imp_global.to_csv(os.path.join(RUTAS["fase3"], "shap_importancia_global.csv"), index=False)
imp_grupo = imp_global.groupby("grupo")["shap_medio_abs"].sum().sort_values(ascending=False)
guardar_csv(pd.DataFrame({"grupo": imp_grupo.index, "nombre_grupo": [DESCRIPCION_GRUPOS[g][0] for g in imp_grupo.index],
                          "n_caracteristicas_en_seleccion": [int((imp_global["grupo"] == g).sum()) for g in imp_grupo.index],
                          "suma_shap_medio_abs": imp_grupo.values,
                          "pct_del_total": np.round(100 * imp_grupo.values / max(imp_grupo.sum(), 1e-12), 2)}),
            os.path.join(RUTAS["fase3"], "shap_importancia_por_grupo.csv"))
print("\nTop-10 características por |SHAP| medio:")
print(imp_global.head(10)[["columna", "nombre", "grupo", "shap_medio_abs"]].to_string(index=False))

# Evidencia: valores SHAP de cada archivo de la muestra (una columna por característica) + tabla legible por archivo
tabla_shap = pd.DataFrame(valores_shap.astype(np.float32), columns=[f"shap_{c}" for c in COLS])
tabla_shap.insert(0, "probabilidad_malware", p_m.astype(np.float32)); tabla_shap.insert(0, "label", y_m)
tabla_shap.insert(0, "file_type", tipo_m); tabla_shap.insert(0, "sha256", sha_m)
tabla_shap["valor_base"] = np.float32(valor_base); tabla_shap["escala"] = escala_shap
tabla_shap.to_parquet(os.path.join(RUTAS["fase3"], "shap_valores_muestra.parquet"), index=False)
print("Tabla guardada:", os.path.join(RUTAS["fase3"], "shap_valores_muestra.parquet"), f"({len(tabla_shap)} archivos x {len(COLS)} características)")
del tabla_shap

fig, ax = plt.subplots(figsize=(8, 6))
top = imp_global.head(15).iloc[::-1]
ax.barh([f"{c} · {n[:40]}" for c, n in zip(top["columna"], top["nombre"])], top["shap_medio_abs"], color="#1F3864")
ax.set_xlabel(f"Importancia global: media de |valor SHAP| ({escala_shap})"); ax.set_title(f"Explicación global SHAP – {NOMBRE_MODELO} (n = {len(X_m)} archivos de prueba)")
ax.grid(axis="x", alpha=0.3)
guardar_figura(fig, "fig_shap_global_barras.png")

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp_grupo.index[::-1].map(lambda g: DESCRIPCION_GRUPOS[g][0]), imp_grupo.values[::-1], color="#2E75B6")
ax.set_xlabel("Suma de |SHAP| medio de las características del grupo"); ax.set_title("Aporte de cada grupo de características a las decisiones del modelo")
ax.grid(axis="x", alpha=0.3)
guardar_figura(fig, "fig_shap_por_grupo.png")

fig = plt.figure(figsize=(9, 7))
shap.summary_plot(valores_shap, X_m, feature_names=[f"{c} · {n[:32]}" for c, n in zip(COLS, NOMBRES_COLS)], max_display=15, show=False)
plt.title(f"Resumen SHAP (beeswarm) – {NOMBRE_MODELO}", fontsize=10)
guardar_figura(plt.gcf(), "fig_shap_beeswarm.png")

# ----------------------------------------------------------------------------
# 3) Traducción a lenguaje natural
# ----------------------------------------------------------------------------
PLANTILLAS_GRUPO = {
    "general":         "propiedades generales del archivo (tamaño/entropía global: {nombre})",
    "histogram":       "la distribución de valores de byte del archivo ({nombre})",
    "byteentropy":     "la entropía local del contenido ({nombre}), asociada a código empaquetado o cifrado",
    "strings":         "las cadenas de texto que contiene ({nombre})",
    "header":          "campos del encabezado del ejecutable ({nombre})",
    "section":         "la estructura de sus secciones de código y datos ({nombre})",
    "imports":         "las librerías y funciones del sistema que importa ({nombre})",
    "exports":         "las funciones que exporta ({nombre})",
    "datadirectories": "los directorios de datos del formato PE ({nombre})",
    "richheader":      "las herramientas de compilación registradas en el Rich header ({nombre})",
    "authenticode":    "su firma digital Authenticode ({nombre})",
    "pefilewarnings":  "anomalías estructurales del formato PE ({nombre})",
}

def explicar_en_lenguaje_natural(contribuciones, columnas, top_n=5):
    """contribuciones: vector SHAP de UN archivo. Devuelve lista de frases (las top_n por |SHAP|)."""
    orden = [i for i in np.argsort(-np.abs(contribuciones)) if abs(contribuciones[i]) > 1e-6][:top_n]
    frases = []
    if not orden:
        return ["Ninguna característica individual domina la decisión: el modelo combina pequeñas evidencias de muchas variables."]
    for i in orden:
        col = columnas[i]
        g = grupo_de(col)
        nombre = nombre_legible(col).split(".", 1)[-1]
        efecto = "AUMENTA la sospecha de malware" if contribuciones[i] > 0 else "REDUCE la sospecha (comportamiento típico de software legítimo)"
        frases.append(f"{PLANTILLAS_GRUPO.get(g, '{nombre}').format(nombre=nombre)} → {efecto} (aporte SHAP {contribuciones[i]:+.3f}).")
    return frases

def nivel_riesgo(p):
    if p >= 0.85: return "ALTO"
    if p >= 0.5:  return "MEDIO"
    if p >= 0.2:  return "BAJO"
    return "MUY BAJO"

# Evidencia: la muestra explicada, archivo por archivo (clase real, probabilidad, riesgo, veredicto y los 5 factores en texto)
filas_expl = []
for i in range(len(X_m)):
    orden_i = [j for j in np.argsort(-np.abs(valores_shap[i])) if abs(valores_shap[i][j]) > 1e-6][:5]
    filas_expl.append({"sha256": sha_m[i], "file_type": tipo_m[i], "label": int(y_m[i]), "probabilidad_malware": round(float(p_m[i]), 4),
                       "veredicto": "MALICIOSO" if p_m[i] >= 0.5 else "BENIGNO", "acierto": bool((p_m[i] >= 0.5) == (y_m[i] == 1)),
                       "nivel_riesgo": nivel_riesgo(p_m[i]),
                       "top5_caracteristicas": "; ".join(f"{COLS[j]} ({valores_shap[i][j]:+.3f})" for j in orden_i),
                       "explicacion": " | ".join(explicar_en_lenguaje_natural(valores_shap[i], COLS))})
guardar_csv(pd.DataFrame(filas_expl), os.path.join(RUTAS["fase3"], "muestra_explicada.csv"))
del filas_expl

ejemplos = {}
with open(os.path.join(RUTAS["fase3"], "explicaciones_ejemplos.txt"), "w", encoding="utf-8") as f_txt:
    for etiqueta, cond in [("malicioso", (y_m == 1) & (p_m >= 0.5)), ("benigno", (y_m == 0) & (p_m < 0.5))]:
        cand = np.where(cond)[0]
        i = int(cand[0]) if len(cand) else 0
        ejemplos[etiqueta] = i
        encabezado = (f"--- Ejemplo {etiqueta} (sha256 {sha_m[i]}, {tipo_m[i]}, índice de prueba {idx_muestra[i]}): "
                      f"probabilidad de malware = {p_m[i]:.3f}, riesgo {nivel_riesgo(p_m[i])} ---")
        print("\n" + encabezado)
        f_txt.write(encabezado + "\n")
        for fr in explicar_en_lenguaje_natural(valores_shap[i], COLS):
            print("  •", fr)
            f_txt.write("  • " + fr + "\n")
        f_txt.write("\n")
        exp = shap.Explanation(values=valores_shap[i], base_values=valor_base, data=X_m[i],
                               feature_names=[f"{c} · {n[:30]}" for c, n in zip(COLS, NOMBRES_COLS)])
        plt.figure(figsize=(9, 6))
        shap.plots.waterfall(exp, max_display=10, show=False)
        plt.title(f"Explicación local SHAP – archivo {etiqueta} (p = {p_m[i]:.3f})", fontsize=10)
        guardar_figura(plt.gcf(), f"fig_shap_local_{etiqueta}.png")

# ----------------------------------------------------------------------------
# 4) LIME como técnica de contraste + acuerdo SHAP-LIME + tiempos
# ----------------------------------------------------------------------------
explicador_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_s, feature_names=NOMBRES_COLS, class_names=["Benigno", "Malicioso"],
    mode="classification", discretize_continuous=True, random_state=CFG["RANDOM_STATE"])
proba_lime = lambda X: np.column_stack([1 - predecir_proba(X), predecir_proba(X)])

def top_lime(i, n=5, num_samples=1000):
    exp = explicador_lime.explain_instance(X_m[i], proba_lime, num_features=n, num_samples=num_samples, labels=(1,))
    return exp, [int(j) for j, _ in exp.as_map()[1][:n]]   # índices de característica (mismo orden que COLS)

n_ac = min(CFG["N_INSTANCIAS_ACUERDO"], len(X_m))
jaccards, tiempos_lime, filas_acuerdo = [], [], []
for i in range(n_ac):
    t0 = time.time()
    _, top_l = top_lime(i)
    tiempos_lime.append(1000 * (time.time() - t0))
    top_s = set(np.argsort(-np.abs(valores_shap[i]))[:5].tolist())
    top_l = set(top_l)
    jaccards.append(len(top_s & top_l) / len(top_s | top_l))
    filas_acuerdo.append({"sha256": sha_m[i], "label": int(y_m[i]), "probabilidad_malware": round(float(p_m[i]), 4),
                          "top5_shap": ";".join(COLS[j] for j in sorted(top_s)), "top5_lime": ";".join(COLS[j] for j in sorted(top_l)),
                          "coincidencias": len(top_s & top_l), "jaccard": round(jaccards[-1], 4), "ms_lime": round(tiempos_lime[-1], 1)})
acuerdo_medio = float(np.mean(jaccards))
guardar_csv(pd.DataFrame(filas_acuerdo), os.path.join(RUTAS["fase3"], "acuerdo_shap_lime_por_instancia.csv"))
print(f"\nAcuerdo SHAP-LIME (Jaccard top-5, n={n_ac}): media = {acuerdo_medio:.3f} | LIME: {np.median(tiempos_lime):.0f} ms/archivo vs SHAP: {ms_por_explicacion_shap:.2f} ms/archivo")

exp_lime, _ = top_lime(ejemplos["malicioso"], n=8)
fig = exp_lime.as_pyplot_figure(label=1)
fig.set_size_inches(9, 5)
plt.title("Explicación local LIME del mismo archivo malicioso (contraste con SHAP)", fontsize=10)
guardar_figura(fig, "fig_lime_ejemplo.png")
exp_lime.save_to_file(os.path.join(RUTAS["fase3"], "lime_ejemplo_malicioso.html"))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(jaccards, bins=np.linspace(0, 1, 11), color="#2E75B6", edgecolor="white")
ax.axvline(acuerdo_medio, color="#C00000", linestyle="--", label=f"media = {acuerdo_medio:.2f}")
ax.set_xlabel("Índice de Jaccard entre los 5 factores principales de SHAP y de LIME"); ax.set_ylabel("Archivos"); ax.legend()
ax.set_title(f"Acuerdo entre técnicas de explicación (n = {n_ac} archivos de prueba)")
guardar_figura(fig, "fig_acuerdo_shap_lime.png")

resumen_fase3 = {
    "modelo_explicado": NOMBRE_MODELO, "tecnica_principal": "SHAP (" + ("TreeExplainer" if NOMBRE_MODELO != "Red Neuronal Densa" else "KernelExplainer") + ")",
    "tecnica_contraste": "LIME (LimeTabularExplainer)", "escala_shap": escala_shap, "n_archivos_explicados": int(len(X_m)),
    "shap_aproximado": bool(NOMBRE_MODELO == "Random Forest"),
    "cobertura_explicacion_pct": round(100 * cobertura, 2), "error_fidelidad_shap": round(error_fidelidad, 6),
    "ms_por_explicacion_shap": round(ms_por_explicacion_shap, 3), "ms_por_explicacion_lime_mediana": round(float(np.median(tiempos_lime)), 1),
    "acuerdo_shap_lime_jaccard_top5": round(acuerdo_medio, 4), "n_instancias_acuerdo": n_ac,
    "top_10_global": imp_global.head(10)[["columna", "nombre", "grupo", "shap_medio_abs"]].round(5).to_dict(orient="records"),
    "aporte_por_grupo": imp_grupo.round(5).to_dict(), "valor_base": valor_base,
    "ejemplo_malicioso": {"indice_prueba": int(idx_muestra[ejemplos["malicioso"]]), "probabilidad": float(p_m[ejemplos["malicioso"]]),
                          "explicacion": explicar_en_lenguaje_natural(valores_shap[ejemplos["malicioso"]], COLS)},
    "ejemplo_benigno": {"indice_prueba": int(idx_muestra[ejemplos["benigno"]]), "probabilidad": float(p_m[ejemplos["benigno"]]),
                        "explicacion": explicar_en_lenguaje_natural(valores_shap[ejemplos["benigno"]], COLS)},
    "acierto_en_muestra_explicada": round(float(np.mean((p_m >= 0.5) == (y_m == 1))), 4),
}
guardar_json(resumen_fase3, os.path.join(RUTAS["fase3"], "fase3_resumen.json"))
joblib.dump({"plantillas": PLANTILLAS_GRUPO, "valor_base": valor_base, "escala": escala_shap}, os.path.join(RUTAS["modelos"], "xai_config.joblib"))
print("\nResumen Fase 3 guardado en", os.path.join(RUTAS["fase3"], "fase3_resumen.json"))

# Liberar modelo, explicadores y matrices (la Fase 4 carga el modelo una sola vez dentro del analizador)
liberar("modelo", "explainer", "explicador_lime", "predecir_proba", "proba_lime", "X_train_s", "X_m", "valores_shap",
        "exp", "exp_lime", "imp_global", "sv", "y_test")
memoria_ram("fase3 fin (modelo y explicadores liberados)")


## Fase 4a — Interfaz interactiva (Gradio) y motor de análisis (OE3)
Escribe `app_malware.py` (también se sube al repositorio) con la clase `AnalizadorMalware` y la interfaz web. El usuario sube un archivo, el sistema extrae las 2.568 características con el mismo extractor de EMBER2024, aplica la selección de características, el escalador y el modelo, y muestra el veredicto, la probabilidad, el nivel de riesgo, una recomendación y los cinco factores que más pesaron, en lenguaje comprensible. El archivo **nunca se ejecuta**.

In [ ]:
"""
FASE 4a - INTERFAZ INTERACTIVA (Gradio) + MOTOR DE ANÁLISIS

Genera el archivo app_malware.py (módulo autocontenido que también va al repositorio)
con la clase AnalizadorMalware y la interfaz web. El flujo para un archivo subido por
el usuario es:
   bytes del archivo -> thrember.PEFeatureExtractor.feature_vector (2.568 características,
   mismo extractor con el que se construyó EMBER2024) -> selección de las K columnas
   -> StandardScaler -> modelo seleccionado -> probabilidad -> SHAP local -> frases en
   español + nivel de riesgo + recomendación.
El análisis es ESTÁTICO: el archivo nunca se ejecuta.
"""
CODIGO_APP = r'''
"""app_malware.py - Sistema inteligente de detección de malware con explicaciones (Trabajo de Grado, UCC 2026)."""
import os, io, json, time, hashlib
import numpy as np
import pandas as pd
import joblib

LIMITE_BYTES = 50 * 1024 * 1024  # 50 MB

PLANTILLAS_GRUPO = {
    "general":         "propiedades generales del archivo ({nombre})",
    "histogram":       "la distribución de valores de byte ({nombre})",
    "byteentropy":     "la entropía local del contenido ({nombre}), típica de código empaquetado o cifrado",
    "strings":         "las cadenas de texto que contiene ({nombre})",
    "header":          "campos del encabezado del ejecutable ({nombre})",
    "section":         "la estructura de sus secciones de código y datos ({nombre})",
    "imports":         "las librerías y funciones del sistema que importa ({nombre})",
    "exports":         "las funciones que exporta ({nombre})",
    "datadirectories": "los directorios de datos del formato PE ({nombre})",
    "richheader":      "las herramientas de compilación registradas en el Rich header ({nombre})",
    "authenticode":    "su firma digital Authenticode ({nombre})",
    "pefilewarnings":  "anomalías estructurales del formato PE ({nombre})",
}
RECOMENDACIONES = {
    "ALTO":     "No abra ni ejecute este archivo. Elimínelo y, si lo recibió por correo o mensajería, reporte al remitente como sospechoso.",
    "MEDIO":    "Evite ejecutarlo. Verifíquelo con un segundo antivirus o en un servicio como VirusTotal antes de usarlo.",
    "BAJO":     "El archivo parece legítimo, pero mantenga su antivirus activo y descárguelo solo de fuentes oficiales.",
    "MUY BAJO": "No se encontraron indicios de malware. Aun así, use siempre software de fuentes confiables.",
}

def nivel_riesgo(p):
    if p >= 0.85: return "ALTO"
    if p >= 0.5:  return "MEDIO"
    if p >= 0.2:  return "BAJO"
    return "MUY BAJO"


class AnalizadorMalware:
    """Carga los artefactos entrenados y analiza archivos (estáticamente) con explicación."""

    def __init__(self, base_dir):
        import thrember
        self.base_dir = base_dir
        m = os.path.join(base_dir, "modelos")
        sel = json.load(open(os.path.join(m, "modelo_seleccionado.json"), encoding="utf-8"))
        self.nombre_modelo = sel["modelo"]
        self.columnas = sel["columnas"]
        self.indices = [int(c[1:]) for c in self.columnas]
        self.umbral = float(sel.get("umbral", 0.5))
        dic = json.load(open(os.path.join(base_dir, "datos", "diccionario_caracteristicas.json"), encoding="utf-8"))
        self.nombres_todos, self.grupos_todos = dic["nombres"], dic["grupos"]
        self.nombres = [self.nombres_todos[i] for i in self.indices]
        self.grupos = [self.grupos_todos[i] for i in self.indices]
        self.scaler = joblib.load(os.path.join(m, "scaler.joblib"))
        self.extractor = thrember.PEFeatureExtractor()
        if self.nombre_modelo == "Red Neuronal Densa":
            from tensorflow import keras
            self.modelo = keras.models.load_model(os.path.join(m, "dnn.keras"))
            self._proba = lambda X: self.modelo.predict(np.asarray(X, dtype=np.float32), verbose=0).ravel()
        else:
            archivo = "random_forest.joblib" if self.nombre_modelo == "Random Forest" else "xgboost.joblib"
            self.modelo = joblib.load(os.path.join(m, archivo))
            self._proba = lambda X: self.modelo.predict_proba(np.asarray(X, dtype=np.float32))[:, 1]
        self._preparar_explicador()

    def _preparar_explicador(self):
        import shap
        if self.nombre_modelo == "Red Neuronal Densa":
            fondo = np.zeros((1, len(self.columnas)), dtype=np.float32)  # referencia: valores medios (escalados = 0)
            self.explainer = shap.KernelExplainer(self._proba, fondo)
            self._shap = lambda X: np.asarray(self.explainer.shap_values(X, nsamples=200)).reshape(len(X), -1)
        else:
            self.explainer = shap.TreeExplainer(self.modelo)
            aproximar = self.nombre_modelo == "Random Forest"  # árboles profundos: Tree SHAP exacto sería demasiado lento
            def _shap(X):
                sv = self.explainer.shap_values(X, approximate=aproximar)
                if isinstance(sv, list):
                    return np.asarray(sv[1])
                sv = np.asarray(sv)
                return sv[:, :, 1] if sv.ndim == 3 else sv
            self._shap = _shap

    # ---------------- núcleo ----------------
    def vector_desde_bytes(self, bytez):
        completo = np.asarray(self.extractor.feature_vector(bytez), dtype=np.float32)
        return completo, completo[self.indices]

    def analizar_vector_reducido(self, x_reducido, extra=None):
        t0 = time.time()
        x = np.nan_to_num(np.asarray(x_reducido, dtype=np.float32).reshape(1, -1))
        xs = self.scaler.transform(x).astype(np.float32)
        p = float(self._proba(xs)[0])
        contrib = self._shap(xs)[0]
        orden = [i for i in np.argsort(-np.abs(contrib)) if abs(contrib[i]) > 1e-6][:5]
        factores = []
        for i in orden:
            nombre = self.nombres[i].split(".", 1)[-1]
            efecto = "aumenta la sospecha" if contrib[i] > 0 else "reduce la sospecha"
            frase = PLANTILLAS_GRUPO.get(self.grupos[i], "{nombre}").format(nombre=nombre)
            factores.append({"caracteristica": self.columnas[i], "nombre": self.nombres[i], "grupo": self.grupos[i],
                             "aporte_shap": round(float(contrib[i]), 4), "efecto": efecto,
                             "explicacion": f"{frase[0].upper() + frase[1:]} {efecto} de malware."})
        riesgo = nivel_riesgo(p)
        res = {"probabilidad_malware": round(p, 4), "veredicto": "MALICIOSO" if p >= self.umbral else "BENIGNO",
               "nivel_riesgo": riesgo, "recomendacion": RECOMENDACIONES[riesgo], "factores": factores,
               "modelo": self.nombre_modelo, "latencia_ms": round(1000 * (time.time() - t0), 1)}
        if extra:
            res.update(extra)
        return res

    def analizar_bytes(self, bytez, nombre_archivo=""):
        t0 = time.time()
        if bytez is None or len(bytez) == 0:
            return {"error": "El archivo está vacío. Seleccione un archivo válido."}
        if len(bytez) > LIMITE_BYTES:
            return {"error": f"El archivo supera el límite de {LIMITE_BYTES // (1024 * 1024)} MB permitido para el análisis."}
        completo, reducido = self.vector_desde_bytes(bytez)
        es_pe = bool(completo[2] >= 1.0)  # general.is_pe
        advertencias = []
        if not es_pe:
            advertencias.append("El archivo no es un ejecutable de Windows (PE); el modelo fue entrenado con ejecutables PE, "
                                "por lo que el resultado para este tipo de archivo es solo orientativo.")
        res = self.analizar_vector_reducido(reducido, extra={
            "nombre_archivo": nombre_archivo, "tamano_bytes": len(bytez), "es_pe": es_pe,
            "sha256": hashlib.sha256(bytez).hexdigest(), "advertencias": advertencias})
        res["latencia_total_ms"] = round(1000 * (time.time() - t0), 1)
        return res

    def analizar_ruta(self, ruta):
        with open(ruta, "rb") as f:
            return self.analizar_bytes(f.read(), os.path.basename(ruta))


def formatear_markdown(res):
    if "error" in res:
        return f"### ⚠️ {res['error']}"
    color = {"ALTO": "🔴", "MEDIO": "🟠", "BAJO": "🟡", "MUY BAJO": "🟢"}[res["nivel_riesgo"]]
    lineas = [f"## {color} Veredicto: **{res['veredicto']}** — probabilidad de malware: **{100 * res['probabilidad_malware']:.1f} %**",
              f"**Nivel de riesgo:** {res['nivel_riesgo']}  |  **Modelo:** {res['modelo']}  |  **Tiempo de análisis:** {res.get('latencia_total_ms', res['latencia_ms'])} ms",
              f"**Recomendación:** {res['recomendacion']}", "", "### ¿Por qué? Factores que más pesaron en la decisión"]
    if not res["factores"]:
        lineas.append("Ninguna característica individual domina la decisión: el modelo combina pequeñas evidencias de muchas variables.")
    for k, f in enumerate(res["factores"], 1):
        lineas.append(f"{k}. {f['explicacion']} *(aporte {f['aporte_shap']:+.3f})*")
    for a in res.get("advertencias", []):
        lineas.append(f"\n> ⚠️ {a}")
    if res.get("sha256"):
        lineas.append(f"\n<sub>SHA-256: {res['sha256']} · tamaño: {res.get('tamano_bytes', 0):,} bytes · el archivo NO fue ejecutado (análisis estático).</sub>")
    return "\n".join(lineas)


def figura_factores(res):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 3.2))
    if "error" not in res and res["factores"]:
        f = res["factores"][::-1]
        ax.barh([x["nombre"][:38] for x in f], [x["aporte_shap"] for x in f],
                color=["#C00000" if x["aporte_shap"] > 0 else "#2E7D32" for x in f])
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_xlabel("Aporte SHAP (rojo: hacia malware · verde: hacia benigno)")
        ax.set_title("Principales factores de la decisión", fontsize=10)
    fig.tight_layout()
    return fig


def construir_interfaz(analizador, ejemplos=None):
    import gradio as gr
    ejemplos = ejemplos if ejemplos is not None else pd.DataFrame()

    def analizar_archivo(ruta):
        if ruta is None:
            return "Seleccione un archivo para analizar.", None, None
        res = analizador.analizar_ruta(ruta if isinstance(ruta, str) else ruta.name)
        tabla = pd.DataFrame(res["factores"])[["nombre", "aporte_shap", "efecto"]] if res.get("factores") else None
        return formatear_markdown(res), tabla, figura_factores(res)

    def analizar_ejemplo(etiqueta):
        if ejemplos.empty or etiqueta is None:
            return "No hay ejemplos cargados.", None, None
        fila = ejemplos[ejemplos["etiqueta"] == etiqueta].iloc[0]
        res = analizador.analizar_vector_reducido(fila[analizador.columnas].to_numpy(dtype=np.float32),
                                                  extra={"nombre_archivo": etiqueta, "sha256": fila.get("sha256", "")})
        tabla = pd.DataFrame(res["factores"])[["nombre", "aporte_shap", "efecto"]] if res.get("factores") else None
        return formatear_markdown(res) + f"\n\n<sub>Etiqueta real en EMBER2024: **{'MALICIOSO' if int(fila['label']) == 1 else 'BENIGNO'}**</sub>", tabla, figura_factores(res)

    with gr.Blocks(title="Detector de malware con explicaciones") as demo:
        gr.Markdown("# 🛡️ Detector de malware con explicaciones comprensibles\n"
                    "Suba un archivo ejecutable de Windows (.exe, .dll) y el sistema le dirá si parece **malicioso o benigno** y **por qué**. "
                    "El archivo **no se ejecuta**: se analiza su estructura de forma estática con un modelo de machine learning entrenado con EMBER2024.")
        with gr.Tab("Analizar un archivo"):
            entrada = gr.File(label="Archivo a analizar (máx. 50 MB)", type="filepath")
            boton = gr.Button("Analizar", variant="primary")
            salida_md = gr.Markdown()
            salida_tabla = gr.Dataframe(label="Factores principales (SHAP)", interactive=False)
            salida_fig = gr.Plot(label="Aporte de cada factor")
            boton.click(analizar_archivo, inputs=entrada, outputs=[salida_md, salida_tabla, salida_fig])
        with gr.Tab("Probar con ejemplos de EMBER2024"):
            gr.Markdown("Ejemplos reales del conjunto de prueba y del conjunto de desafío (malware evasivo). Solo se guardan sus características, no los archivos.")
            lista = gr.Dropdown(choices=ejemplos["etiqueta"].tolist() if not ejemplos.empty else [], label="Ejemplo")
            boton2 = gr.Button("Analizar ejemplo")
            md2, tabla2, fig2 = gr.Markdown(), gr.Dataframe(interactive=False), gr.Plot()
            boton2.click(analizar_ejemplo, inputs=lista, outputs=[md2, tabla2, fig2])
        gr.Markdown("<sub>Trabajo de grado – Universidad Católica de Colombia, 2026. Uso académico: no sustituye un antivirus.</sub>")
    return demo


if __name__ == "__main__":
    base = os.environ.get("TG_BASE_DIR", ".")
    analizador = AnalizadorMalware(base)
    ruta_ej = os.path.join(base, "fase4", "ejemplos_demo.parquet")
    ejemplos = pd.read_parquet(ruta_ej) if os.path.exists(ruta_ej) else None
    construir_interfaz(analizador, ejemplos).launch(share=True)
'''

RUTA_APP = os.path.join(RUTAS["fase4"], "app_malware.py")
with open(RUTA_APP, "w", encoding="utf-8") as f:
    f.write(CODIGO_APP)
print("Aplicación escrita en", RUTA_APP)

# Ejemplos de demostración: 3 del set de prueba (2 maliciosos, 1 benigno) y 2 del conjunto de desafío
sel_modelo = cargar_json(os.path.join(RUTAS["modelos"], "modelo_seleccionado.json"))
COLS = sel_modelo["columnas"]
# Solo las columnas necesarias (K + 3 metadatos): evita cargar las cadenas de 540.000 filas
df_test = pd.read_parquet(os.path.join(RUTAS["fase1"], "test_reducido.parquet"), columns=COLS + ["label", "file_type", "sha256"])
df_chal = pd.read_parquet(os.path.join(RUTAS["fase1"], "challenge_reducido.parquet"), columns=COLS + ["label", "file_type", "sha256"])
partes = []
for i, fila in enumerate(df_test[df_test.label == 1].sample(2, random_state=1).itertuples(index=False)):
    partes.append(pd.Series(fila._asdict()).to_frame().T.assign(etiqueta=f"Prueba – malicioso {i + 1} ({fila.file_type})"))
for i, fila in enumerate(df_test[df_test.label == 0].sample(2, random_state=1).itertuples(index=False)):
    partes.append(pd.Series(fila._asdict()).to_frame().T.assign(etiqueta=f"Prueba – benigno {i + 1} ({fila.file_type})"))
for i, fila in enumerate(df_chal.sample(min(2, len(df_chal)), random_state=1).itertuples(index=False)):
    partes.append(pd.Series(fila._asdict()).to_frame().T.assign(etiqueta=f"Desafío – malware evasivo {i + 1} ({fila.file_type})"))
ejemplos_demo = pd.concat(partes, ignore_index=True)
ejemplos_demo[COLS] = ejemplos_demo[COLS].astype(np.float32)
ejemplos_demo["label"] = ejemplos_demo["label"].astype(int)
ejemplos_demo = ejemplos_demo[COLS + ["label", "file_type", "sha256", "etiqueta"]]
ejemplos_demo.to_parquet(os.path.join(RUTAS["fase4"], "ejemplos_demo.parquet"), index=False)
del df_test, df_chal

# Importar el módulo recién escrito y crear el analizador
import importlib.util
spec = importlib.util.spec_from_file_location("app_malware", RUTA_APP)
app_malware = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app_malware)
analizador = app_malware.AnalizadorMalware(BASE_DIR)
print(f"Analizador listo: modelo {analizador.nombre_modelo}, {len(analizador.columnas)} características.")
res_demo = analizador.analizar_vector_reducido(ejemplos_demo.iloc[0][COLS].to_numpy(dtype=np.float32))
print(app_malware.formatear_markdown(res_demo))
# Evidencia: resultado completo de un análisis de ejemplo (formato JSON que devuelve el motor de la interfaz)
guardar_json({"ejemplo": ejemplos_demo.iloc[0]["etiqueta"], "sha256": ejemplos_demo.iloc[0]["sha256"],
              "label_real": int(ejemplos_demo.iloc[0]["label"]), "resultado": res_demo}, os.path.join(RUTAS["fase4"], "ejemplo_resultado_analisis.json"))
gc.collect()
memoria_ram("fase4a analizador listo")


## Fase 4b — Pruebas funcionales y de rendimiento
Nueve pruebas funcionales automatizadas (PF-01 a PF-09) con resultado PASA/FALLA y pruebas de rendimiento (latencia p50/p95 y desglose extracción vs. predicción+explicación) para archivos de 0,1 / 1 / 5 / 20 MB. Los archivos de prueba son ejecutables PE mínimos construidos en la celda (válidos para el analizador, sin código funcional).

**Evidencias (carpeta `fase4/`):** tabla de pruebas funcionales, respuesta completa del analizador en cada prueba, identificación (SHA-256) de los archivos de prueba, resumen y **mediciones crudas** de rendimiento (una fila por repetición) y la figura de latencia.

In [ ]:
"""
FASE 4b - PRUEBAS FUNCIONALES Y DE RENDIMIENTO DEL SISTEMA

Pruebas funcionales (automatizadas, con resultado PASA/FALLA guardado en JSON):
  PF-01 archivo PE válido -> veredicto, probabilidad en [0,1], explicación con factores
  PF-02 archivo PE con indicadores sospechosos (URLs, PowerShell) -> se analiza sin error
  PF-03 archivo que no es PE (texto) -> se analiza y se advierte al usuario
  PF-04 archivo vacío -> mensaje de error controlado
  PF-05 archivo mayor al límite (50 MB) -> rechazo controlado
  PF-06 determinismo: el mismo archivo produce el mismo resultado
  PF-07 latencia de extremo a extremo por debajo de la meta (10 s)
  PF-08 coherencia con ejemplos etiquetados de EMBER2024 (informativa)
  PF-09 la interfaz Gradio se construye sin errores
Pruebas de rendimiento: latencia (p50/p95) y desglose extracción vs. predicción+explicación
para archivos de 0,1 / 1 / 5 / 20 MB, 20 repeticiones cada uno.
"""
import struct
import hashlib

def construir_pe_minimo(relleno_texto=b"\xC3" * 64, cadenas_extra=b"", overlay=b""):
    """Construye un PE32 mínimo y válido (nunca se ejecuta) para probar el extractor y la interfaz."""
    e_lfanew = 0x80
    dos = bytearray(b"MZ" + b"\x00" * 58 + struct.pack("<I", e_lfanew))
    dos += b"\x0e\x1f\xba\x0e\x00\xb4\x09\xcd\x21\xb8\x01\x4c\xcd\x21This program cannot be run in DOS mode.\r\r\n$"
    dos += b"\x00" * (e_lfanew - len(dos))
    file_align, sect_align, size_headers = 0x200, 0x1000, 0x400
    text_rva, text_raw, text_raw_size = 0x1000, 0x400, 0x200
    idata_rva, idata_raw, idata_raw_size = 0x2000, 0x600, 0x200
    idata = bytearray(); desc_size = 40; ilt_off = desc_size; iat_off = ilt_off + 12; names_off = iat_off + 12
    names = bytearray(); hint_rvas = []
    for fn in [b"ExitProcess", b"CreateFileA"]:
        hint_rvas.append(idata_rva + names_off + len(names)); names += b"\x00\x00" + fn + b"\x00"
        if len(names) % 2: names += b"\x00"
    dll_name_rva = idata_rva + names_off + len(names); names += b"KERNEL32.dll\x00"
    ilt = b"".join(struct.pack("<I", r) for r in hint_rvas) + struct.pack("<I", 0)
    desc = struct.pack("<IIIII", idata_rva + ilt_off, 0, 0, dll_name_rva, idata_rva + iat_off) + b"\x00" * 20
    idata += desc + ilt + ilt + names; idata += b"\x00" * (idata_raw_size - len(idata))
    coff = struct.pack("<HHIIIHH", 0x14c, 2, 0x60000000, 0, 0, 0xE0, 0x0102)
    opt = struct.pack("<HBBIIIIIIIIIHHHHHHIIIIHHIIIIII", 0x10b, 14, 0, text_raw_size, idata_raw_size, 0, text_rva, text_rva,
                      idata_rva, 0x400000, sect_align, file_align, 6, 0, 0, 0, 6, 0, 0, 0x3000, size_headers, 0, 2, 0x8140,
                      0x100000, 0x1000, 0x100000, 0x1000, 0, 16)
    dd = bytearray(8 * 16); struct.pack_into("<II", dd, 8, idata_rva, desc_size); struct.pack_into("<II", dd, 8 * 12, idata_rva + iat_off, 12)
    def sect(name, vsize, vaddr, rawsize, rawptr, chars):
        return name.ljust(8, b"\x00") + struct.pack("<IIIIIIHHI", vsize, vaddr, rawsize, rawptr, 0, 0, 0, 0, chars)
    hdr = bytes(dos) + b"PE\x00\x00" + coff + opt + bytes(dd) + sect(b".text", text_raw_size, text_rva, text_raw_size, text_raw, 0x60000020) \
          + sect(b".idata", idata_raw_size, idata_rva, idata_raw_size, idata_raw, 0xC0000040)
    hdr += b"\x00" * (size_headers - len(hdr))
    text = (relleno_texto + cadenas_extra)[:text_raw_size].ljust(text_raw_size, b"\x00")
    return hdr + text + bytes(idata) + overlay

pruebas = []
def registrar(codigo, descripcion, condicion, detalle=""):
    pruebas.append({"codigo": codigo, "descripcion": descripcion, "resultado": "PASA" if condicion else "FALLA", "detalle": str(detalle)[:300]})
    print(f"{codigo} {'PASA' if condicion else 'FALLA'} - {descripcion} {('| ' + str(detalle)[:120]) if detalle else ''}")

meta_lat = CFG["METAS"]["latencia_maxima_s"]
pe_basico = construir_pe_minimo()
pe_sospechoso = construir_pe_minimo(cadenas_extra=b" http://malicioso.example/dl.exe powershell -enc Invoke-Expression CreateRemoteThread ")
rng = np.random.default_rng(CFG["RANDOM_STATE"])

r1 = analizador.analizar_bytes(pe_basico, "pe_minimo.exe")
registrar("PF-01", "PE válido: veredicto, probabilidad y factores", "error" not in r1 and 0 <= r1["probabilidad_malware"] <= 1 and len(r1["factores"]) >= 1 and r1["es_pe"],
          f"veredicto={r1.get('veredicto')} p={r1.get('probabilidad_malware')} factores={len(r1.get('factores', []))}")
r2 = analizador.analizar_bytes(pe_sospechoso, "pe_sospechoso.exe")
registrar("PF-02", "PE con indicadores sospechosos se analiza sin error", "error" not in r2, f"p={r2.get('probabilidad_malware')} riesgo={r2.get('nivel_riesgo')}")
r3 = analizador.analizar_bytes(b"Hola, este es un archivo de texto plano sin estructura PE.\n" * 100, "notas.txt")
registrar("PF-03", "Archivo no PE: se analiza y se advierte", "error" not in r3 and r3["es_pe"] is False and len(r3["advertencias"]) == 1, r3.get("advertencias", [""])[0][:80])
r4 = analizador.analizar_bytes(b"", "vacio.exe")
registrar("PF-04", "Archivo vacío: error controlado", "error" in r4, r4.get("error"))
r5 = analizador.analizar_bytes(b"\x00" * (app_malware.LIMITE_BYTES + 1), "gigante.exe")
registrar("PF-05", "Archivo mayor al límite: rechazo controlado", "error" in r5, r5.get("error"))
r6 = analizador.analizar_bytes(pe_sospechoso, "pe_sospechoso.exe")
registrar("PF-06", "Determinismo: mismo archivo, mismo resultado", r6["probabilidad_malware"] == r2["probabilidad_malware"] and r6["sha256"] == r2["sha256"])
pe_grande = construir_pe_minimo(overlay=rng.integers(0, 256, 5 * 1024 * 1024, dtype=np.uint8).tobytes())
r7 = analizador.analizar_bytes(pe_grande, "pe_5mb.exe")
registrar("PF-07", f"Latencia extremo a extremo < {meta_lat:.0f} s (archivo de 5 MB)", "error" not in r7 and r7["latencia_total_ms"] / 1000 < meta_lat, f"{r7.get('latencia_total_ms')} ms")
aciertos = 0
for _, fila in ejemplos_demo.iterrows():
    r = analizador.analizar_vector_reducido(fila[COLS].to_numpy(dtype=np.float32))
    aciertos += int((r["veredicto"] == "MALICIOSO") == (int(fila["label"]) == 1))
registrar("PF-08", "Coherencia con ejemplos etiquetados de EMBER2024 (informativa)", aciertos >= len(ejemplos_demo) // 2, f"{aciertos}/{len(ejemplos_demo)} coinciden con la etiqueta real")
try:
    demo = app_malware.construir_interfaz(analizador, ejemplos_demo)
    registrar("PF-09", "La interfaz Gradio se construye sin errores", True, type(demo).__name__)
except Exception as e:  # noqa: BLE001
    registrar("PF-09", "La interfaz Gradio se construye sin errores", False, repr(e))

df_pruebas = pd.DataFrame(pruebas)
df_pruebas.to_csv(os.path.join(RUTAS["fase4"], "pruebas_funcionales.csv"), index=False)
print(f"\nPruebas funcionales: {(df_pruebas.resultado == 'PASA').sum()}/{len(df_pruebas)} superadas")
# Evidencia: respuesta completa del analizador en cada prueba (sin los bytes de los archivos) e identificación de los
# archivos de prueba por SHA-256 (no se guardan los ejecutables sintéticos para no distribuir binarios)
guardar_json({"PF-01": r1, "PF-02": r2, "PF-03": r3, "PF-04": r4, "PF-05": r5, "PF-06": r6, "PF-07": r7},
             os.path.join(RUTAS["fase4"], "pruebas_funcionales_detalle.json"))
guardar_csv(pd.DataFrame([{"archivo": "pe_minimo.exe", "bytes": len(pe_basico), "sha256": hashlib.sha256(pe_basico).hexdigest()},
                          {"archivo": "pe_sospechoso.exe", "bytes": len(pe_sospechoso), "sha256": hashlib.sha256(pe_sospechoso).hexdigest()},
                          {"archivo": "pe_5mb.exe", "bytes": len(pe_grande), "sha256": hashlib.sha256(pe_grande).hexdigest()}]),
            os.path.join(RUTAS["fase4"], "archivos_de_prueba.csv"))
del pe_grande

# ----------------------------------------------------------------------------
# Pruebas de rendimiento
# ----------------------------------------------------------------------------
REPETICIONES = int(CFG.get("REPETICIONES_RENDIMIENTO", 20))
tamanos = {"0.1 MB": 100 * 1024, "1 MB": 1024 * 1024, "5 MB": 5 * 1024 * 1024, "20 MB": 20 * 1024 * 1024}
filas = []
for etiqueta, n in tamanos.items():
    archivo = construir_pe_minimo(overlay=rng.integers(0, 256, n, dtype=np.uint8).tobytes())
    for rep in range(1, REPETICIONES + 1):
        t0 = time.time(); _, red = analizador.vector_desde_bytes(archivo); t1 = time.time()
        analizador.analizar_vector_reducido(red); t2 = time.time()
        filas.append({"tamano": etiqueta, "bytes": len(archivo), "repeticion": rep, "extraccion_ms": 1000 * (t1 - t0),
                      "prediccion_y_explicacion_ms": 1000 * (t2 - t1), "total_ms": 1000 * (t2 - t0)})
    del archivo   # el archivo de 20 MB se libera antes de generar el siguiente
df_rend = pd.DataFrame(filas)
guardar_csv(df_rend.round(2), os.path.join(RUTAS["fase4"], "pruebas_rendimiento_detalle.csv"))  # todas las mediciones crudas
resumen_rend = df_rend.groupby("tamano", sort=False).agg(
    extraccion_ms_p50=("extraccion_ms", "median"), prediccion_ms_p50=("prediccion_y_explicacion_ms", "median"),
    total_ms_p50=("total_ms", "median"), total_ms_p95=("total_ms", lambda s: float(np.percentile(s, 95))), total_ms_max=("total_ms", "max")).round(1)
resumen_rend["cumple_meta"] = resumen_rend["total_ms_p95"] / 1000 <= meta_lat
resumen_rend.to_csv(os.path.join(RUTAS["fase4"], "pruebas_rendimiento.csv"))
print("\n=== Rendimiento (ms) ===")
print(resumen_rend.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
g = df_rend.groupby("tamano", sort=False)
ax.bar(resumen_rend.index, resumen_rend["extraccion_ms_p50"], label="Extracción de características (thrember)", color="#4C78A8")
ax.bar(resumen_rend.index, resumen_rend["prediccion_ms_p50"], bottom=resumen_rend["extraccion_ms_p50"], label="Predicción + explicación SHAP", color="#F58518")
ax.plot(resumen_rend.index, resumen_rend["total_ms_p95"], "k^--", label="p95 total")
ax.axhline(1000 * meta_lat, color="#C00000", linestyle=":", label=f"Meta: {meta_lat:.0f} s")
ax.set_ylabel("Milisegundos (mediana)"); ax.set_xlabel("Tamaño del archivo analizado"); ax.set_yscale("log")
ax.set_title("Latencia de análisis de extremo a extremo por tamaño de archivo"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
guardar_figura(fig, "fig_latencia_por_tamano.png")

guardar_json({"pruebas_funcionales": pruebas, "n_pasan": int((df_pruebas.resultado == "PASA").sum()), "n_total": int(len(df_pruebas)),
              "rendimiento": resumen_rend.reset_index().to_dict(orient="records"), "repeticiones": REPETICIONES,
              "meta_latencia_s": meta_lat, "entorno": "Google Colab (CPU)" if EN_COLAB else "local",
              "modelo_evaluado": analizador.nombre_modelo, "n_caracteristicas": len(analizador.columnas)},
             os.path.join(RUTAS["fase4"], "fase4_pruebas.json"))
print("\nResultados de pruebas guardados en", os.path.join(RUTAS["fase4"], "fase4_pruebas.json"))
memoria_ram("fase4b pruebas fin")


## Fase 4c — Lanzar la interfaz web
Genera un enlace público temporal para las sesiones de prueba con usuarios. Mientras esta celda corre, la aplicación está disponible. Toma la captura de pantalla para el documento (`figuras/fig_interfaz_gradio.png`).

In [ ]:
"""
FASE 4c - LANZAR LA INTERFAZ WEB (Gradio)

Genera un enlace público temporal (share=True) para que los participantes de la
prueba de usabilidad usen el sistema desde cualquier navegador. Mientras esta
celda esté corriendo, la aplicación permanece disponible; detén la celda para
cerrarla. Toma una captura de pantalla de la interfaz para el documento
(Figura del capítulo 3) y guárdala como figuras/fig_interfaz_gradio.png.
"""
demo = app_malware.construir_interfaz(analizador, ejemplos_demo)
demo.launch(share=True, debug=False)


## Fase 4d — Prueba de usabilidad (SUS)
Crea la plantilla de respuestas y el cuestionario **System Usability Scale** (10 ítems, escala 1-5). Con las respuestas reales de los participantes calcula el puntaje SUS (0-100), su intervalo de confianza y la figura. **Los puntajes no se simulan**: sin respuestas reales la celda solo deja la plantilla y el estado queda como pendiente.

In [ ]:
"""
FASE 4d - PRUEBA DE USABILIDAD: SYSTEM USABILITY SCALE (SUS)

El SUS (Brooke, 1996) consta de 10 afirmaciones que cada participante califica de
1 (totalmente en desacuerdo) a 5 (totalmente de acuerdo). Puntaje por participante:
   ítems impares -> (respuesta - 1); ítems pares -> (5 - respuesta); suma x 2,5  => 0-100.
Meta del proyecto: promedio >= 70 (por encima del promedio de referencia de 68).

Cómo usarlo:
  1. Esta celda crea la plantilla fase4/sus_respuestas.csv (una fila por participante).
  2. Cada participante usa la interfaz (celda anterior) con al menos 2 archivos y
     responde el cuestionario; se registran sus respuestas en el CSV (P1..P10) junto
     con su perfil (sin datos personales: solo un código, edad opcional y si tiene o
     no formación técnica).
  3. Al volver a correr esta celda con el CSV diligenciado, calcula el puntaje SUS,
     el intervalo de confianza y la figura para el documento.
LOS PUNTAJES NO SE INVENTAN: sin respuestas reales, la celda solo deja la plantilla.
"""
ITEMS_SUS = [
    "Creo que me gustaría usar este sistema con frecuencia.",
    "Encontré el sistema innecesariamente complejo.",
    "Pensé que el sistema era fácil de usar.",
    "Creo que necesitaría el apoyo de una persona técnica para poder usar este sistema.",
    "Encontré que las diversas funciones del sistema estaban bien integradas.",
    "Pensé que había demasiada inconsistencia en este sistema.",
    "Imagino que la mayoría de las personas aprenderían a usar este sistema muy rápidamente.",
    "Encontré el sistema muy engorroso de usar.",
    "Me sentí muy seguro(a) usando el sistema.",
    "Necesité aprender muchas cosas antes de poder empezar a usar el sistema.",
]
RUTA_SUS = os.path.join(RUTAS["fase4"], "sus_respuestas.csv")
columnas = ["participante", "perfil_tecnico", "edad", "entendio_explicacion"] + [f"P{i}" for i in range(1, 11)] + ["comentario"]

if not os.path.exists(RUTA_SUS):
    pd.DataFrame(columns=columnas).to_csv(RUTA_SUS, index=False)
    with open(os.path.join(RUTAS["fase4"], "cuestionario_SUS.txt"), "w", encoding="utf-8") as f:
        f.write("CUESTIONARIO SUS (1 = totalmente en desacuerdo ... 5 = totalmente de acuerdo)\n\n")
        for i, it in enumerate(ITEMS_SUS, 1):
            f.write(f"P{i}. {it}\n")
        f.write("\nPregunta adicional: ¿Entendió por qué el sistema clasificó el archivo como lo hizo? (Sí / Parcialmente / No)\n")
    print("Plantilla creada:", RUTA_SUS, "\nCuestionario:", os.path.join(RUTAS["fase4"], "cuestionario_SUS.txt"))


def calcular_sus(df):
    p = df[[f"P{i}" for i in range(1, 11)]].astype(float)
    impares = p[[f"P{i}" for i in range(1, 11, 2)]].sub(1).sum(axis=1)
    pares = (5 - p[[f"P{i}" for i in range(2, 11, 2)]]).sum(axis=1)
    return (impares + pares) * 2.5

df_sus = pd.read_csv(RUTA_SUS)
df_sus = df_sus.dropna(subset=[f"P{i}" for i in range(1, 11)])
if len(df_sus) == 0:
    print("\nAún no hay respuestas registradas. Diligencia el CSV con los participantes reales y vuelve a correr esta celda.")
    resumen_sus = {"n_participantes": 0, "estado": "pendiente: sin respuestas reales"}
else:
    df_sus["sus"] = calcular_sus(df_sus)
    n = len(df_sus); media = float(df_sus["sus"].mean()); de = float(df_sus["sus"].std(ddof=1)) if n > 1 else 0.0
    ic = 1.96 * de / np.sqrt(n) if n > 1 else 0.0
    por_perfil = df_sus.groupby("perfil_tecnico")["sus"].agg(["count", "mean"]).round(1) if "perfil_tecnico" in df_sus else None
    adjetivo = "Excelente" if media >= 85 else "Bueno" if media >= 72 else "Aceptable" if media >= 52 else "Deficiente"
    resumen_sus = {"n_participantes": n, "sus_promedio": round(media, 1), "sus_desviacion": round(de, 1), "ic95": round(ic, 1),
                   "sus_minimo": float(df_sus["sus"].min()), "sus_maximo": float(df_sus["sus"].max()), "adjetivo": adjetivo,
                   "cumple_meta": bool(media >= CFG["METAS"]["sus_minimo"]),
                   "por_perfil": por_perfil.reset_index().to_dict(orient="records") if por_perfil is not None else None,
                   "promedio_por_item": df_sus[[f"P{i}" for i in range(1, 11)]].mean().round(2).to_dict(),
                   "entendio_explicacion": df_sus["entendio_explicacion"].value_counts().to_dict() if "entendio_explicacion" in df_sus else None}
    print(f"\nSUS promedio = {media:.1f} ± {ic:.1f} (IC 95 %), n = {n}, rango {resumen_sus['sus_minimo']:.0f}-{resumen_sus['sus_maximo']:.0f}, "
          f"calificación '{adjetivo}', meta {CFG['METAS']['sus_minimo']:.0f}: {'CUMPLE' if resumen_sus['cumple_meta'] else 'NO CUMPLE'}")
    # Evidencia: puntaje SUS calculado por participante (a partir de las respuestas reales del CSV)
    guardar_csv(df_sus[["participante", "perfil_tecnico", "entendio_explicacion"] + [f"P{i}" for i in range(1, 11)] + ["sus"]],
                os.path.join(RUTAS["fase4"], "sus_puntajes_por_participante.csv"))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(df_sus["participante"].astype(str), df_sus["sus"], color=["#2E7D32" if s >= CFG["METAS"]["sus_minimo"] else "#C00000" for s in df_sus["sus"]])
    axes[0].axhline(media, color="#1F3864", linestyle="--", label=f"promedio = {media:.1f}"); axes[0].axhline(68, color="gray", linestyle=":", label="referencia SUS = 68")
    axes[0].set_ylim(0, 100); axes[0].set_ylabel("Puntaje SUS"); axes[0].set_title("Puntaje SUS por participante"); axes[0].legend(fontsize=8); axes[0].tick_params(axis="x", rotation=90, labelsize=7)
    prom = df_sus[[f"P{i}" for i in range(1, 11)]].mean()
    axes[1].barh([f"P{i}" for i in range(10, 0, -1)], prom[::-1], color="#2E75B6"); axes[1].set_xlim(1, 5); axes[1].set_title("Respuesta promedio por ítem (1-5)")
    guardar_figura(fig, "fig_sus_resultados.png")
guardar_json(resumen_sus, os.path.join(RUTAS["fase4"], "fase4_sus.json"))


## Exportación de resultados y paquete de evidencias
Consolida los resúmenes de todas las fases en `resultados_para_documento.json`, imprime el cuadro de cumplimiento de las metas del anteproyecto y genera dos paquetes:

* `figuras_documento/` → `entregables_documento.zip`: paquete **ligero** (figuras PNG, tablas principales y JSON) para actualizar el documento.
* `evidencias/` → `evidencias_proyecto.zip`: paquete **completo**, organizado por fase (`01_dataset`, `02_seleccion_caracteristicas`, `03_modelos`, `04_explicabilidad`, `05_interfaz_y_pruebas`, `06_figuras`) con un **manifiesto** (`00_MANIFIESTO_EVIDENCIAS.md` / `00_indice_evidencias.csv`) que describe cada archivo, su tamaño y su estado (presente / pendiente). Incluye las bases de datos reducidas (parquet + muestra CSV de 5.000 filas), matrices de confusión, curvas, predicciones por archivo, valores SHAP, pruebas y el registro de memoria. Los modelos serializados entran al zip solo si pesan menos de `LIMITE_MB_MODELO_EN_ZIP` (150 MB); si no, quedan en `Drive/.../modelos`.

En Colab la celda descarga los dos zip al computador (si el navegador bloquea la descarga, se toman desde Google Drive).

In [ ]:
"""
EXPORTACIÓN DE RESULTADOS Y PAQUETE DE EVIDENCIAS

1. Consolida los resúmenes de todas las fases en resultados_para_documento.json e
   imprime el cuadro de cumplimiento de las metas del anteproyecto.
2. figuras_documento/ + entregables_documento.zip: paquete LIGERO (figuras PNG,
   tablas CSV principales y JSON) para actualizar el documento de trabajo de grado.
3. evidencias/ + evidencias_proyecto.zip: paquete COMPLETO con la evidencia de cada
   paso, organizada por fase y con un MANIFIESTO que describe cada archivo:
     01_dataset                 composición del dataset construido y diccionario de datos
     02_seleccion_caracteristicas  ranking, curva K vs F1, características seleccionadas,
                                datasets reducidos (parquet + muestra CSV), sha256 de la muestra
     03_modelos                 búsqueda de hiperparámetros, comparación, matrices de confusión,
                                reportes, curvas ROC/PR, métricas por umbral/tipo/semana,
                                predicciones por archivo, modelos (si pesan menos del límite)
     04_explicabilidad          importancias SHAP, valores SHAP por archivo, acuerdo SHAP-LIME
     05_interfaz_y_pruebas      código de la app, pruebas funcionales y de rendimiento, SUS
     06_figuras                 todas las figuras generadas
   Los .parquet se abren con pandas: pd.read_parquet("archivo.parquet").
4. En Colab, descarga los dos zip al computador (también quedan en Google Drive).

Control de memoria: los datasets reducidos se copian como archivos (sin cargarlos)
y las muestras CSV se escriben leyendo por bloques.
"""
import glob
import shutil
import pyarrow.parquet as pq

def cargar_si_existe(ruta):
    return cargar_json(ruta) if os.path.exists(ruta) else {"estado": "pendiente (no se ha ejecutado esta fase)"}

resultados_documento = {
    "generado": time.strftime("%Y-%m-%d %H:%M:%S"),
    "dataset": cargar_si_existe(os.path.join(RUTAS["datos"], "resumen_datos.json")),
    "fase1": cargar_si_existe(os.path.join(RUTAS["fase1"], "fase1_resumen.json")),
    "fase2": cargar_si_existe(os.path.join(RUTAS["fase2"], "fase2_resumen.json")),
    "fase3": cargar_si_existe(os.path.join(RUTAS["fase3"], "fase3_resumen.json")),
    "fase4_pruebas": cargar_si_existe(os.path.join(RUTAS["fase4"], "fase4_pruebas.json")),
    "fase4_sus": cargar_si_existe(os.path.join(RUTAS["fase4"], "fase4_sus.json")),
    "configuracion": {k: v for k, v in CFG.items() if k != "HF_TOKEN"},
}
guardar_json(resultados_documento, os.path.join(BASE_DIR, "resultados_para_documento.json"))

# ----------------------------------------------------------------------------
# 1) Cuadro de cumplimiento de las metas del anteproyecto
# ----------------------------------------------------------------------------
f1r = resultados_documento["fase1"]; f2r = resultados_documento["fase2"]; f3r = resultados_documento["fase3"]
f4p = resultados_documento["fase4_pruebas"]; f4s = resultados_documento["fase4_sus"]
sel = f2r.get("modelo_seleccionado"); r_sel = f2r.get("resultados", {}).get(sel, {}) if sel else {}
cuadro = pd.DataFrame([
    ["Reducción de dimensionalidad", f">= {CFG['METAS']['reduccion_dimensionalidad_minima_pct']} %", f"{f1r.get('pct_reduccion', 'pendiente')} %", f1r.get("cumple_meta_reduccion")],
    ["F1 conservado tras la selección", f">= {CFG['META_F1_CONSERVADO']} %", f"{f1r.get('pct_f1_conservado', 'pendiente')} %", f1r.get("cumple_meta_f1_conservado")],
    ["F1-score (prueba temporal)", f">= {CFG['METAS']['f1_minimo']}", r_sel.get("f1", "pendiente"), f2r.get("cumplimiento_metas_modelo_seleccionado", {}).get("f1")],
    ["Tasa de falsos positivos", f"<= {100 * CFG['METAS']['fpr_maximo']:.0f} %", f"{100 * r_sel['fpr']:.2f} %" if r_sel else "pendiente", f2r.get("cumplimiento_metas_modelo_seleccionado", {}).get("fpr")],
    ["Cobertura de explicaciones", "100 %", f"{f3r.get('cobertura_explicacion_pct', 'pendiente')} %", f3r.get("cobertura_explicacion_pct") == 100.0 if isinstance(f3r.get("cobertura_explicacion_pct"), float) else None],
    ["Latencia por archivo (p95, 20 MB)", f"< {CFG['METAS']['latencia_maxima_s']:.0f} s", (f"{f4p['rendimiento'][-1]['total_ms_p95'] / 1000:.2f} s" if f4p.get("rendimiento") else "pendiente"), (bool(f4p["rendimiento"][-1]["cumple_meta"]) if f4p.get("rendimiento") else None)],
    ["Usabilidad SUS", f">= {CFG['METAS']['sus_minimo']:.0f}", f4s.get("sus_promedio", "pendiente"), f4s.get("cumple_meta")],
], columns=["Indicador", "Meta", "Resultado", "Cumple"])
cuadro.to_csv(os.path.join(BASE_DIR, "cuadro_cumplimiento_metas.csv"), index=False)
print("=== Cumplimiento de metas ===")
print(cuadro.to_string(index=False))

# ----------------------------------------------------------------------------
# 2) Paquete ligero para el documento (figuras + tablas principales + JSON)
# ----------------------------------------------------------------------------
carpeta_fig = os.path.join(BASE_DIR, "figuras_documento")
os.makedirs(carpeta_fig, exist_ok=True)
for f in sorted(os.listdir(RUTAS["figuras"])):
    if f.endswith(".png"):
        shutil.copy(os.path.join(RUTAS["figuras"], f), os.path.join(carpeta_fig, f))
tablas = [os.path.join(RUTAS["fase1"], "curva_k_vs_f1.csv"), os.path.join(RUTAS["fase1"], "caracteristicas_seleccionadas.csv"),
          os.path.join(RUTAS["fase2"], "fase2_comparacion_modelos.csv"), os.path.join(RUTAS["fase2"], "matrices_confusion.csv"),
          os.path.join(RUTAS["fase3"], "shap_importancia_global.csv"), os.path.join(RUTAS["fase4"], "pruebas_funcionales.csv"),
          os.path.join(RUTAS["fase4"], "pruebas_rendimiento.csv"), os.path.join(RUTAS["datos"], "resumen_datos.csv"),
          os.path.join(RUTAS["datos"], "diccionario_grupos.csv"), os.path.join(BASE_DIR, "cuadro_cumplimiento_metas.csv"),
          os.path.join(BASE_DIR, "resultados_para_documento.json")]
for t in tablas:
    if os.path.exists(t):
        shutil.copy(t, carpeta_fig)
zip_documento = shutil.make_archive(os.path.join(BASE_DIR, "entregables_documento"), "zip", carpeta_fig)
print("\nPaquete para el documento:", zip_documento, f"({os.path.getsize(zip_documento) / 2 ** 20:.1f} MB)")

# ----------------------------------------------------------------------------
# 3) Paquete completo de evidencias, organizado por fase y descrito en un manifiesto
# ----------------------------------------------------------------------------
EVID = RUTAS["evidencias"]
if os.path.isdir(EVID):
    shutil.rmtree(EVID)   # se reconstruye completo en cada exportación (siempre refleja el estado actual de Drive)
os.makedirs(EVID)
D, F1, F2, F3, F4, M = RUTAS["datos"], RUTAS["fase1"], RUTAS["fase2"], RUTAS["fase3"], RUTAS["fase4"], RUTAS["modelos"]

# (carpeta destino, archivo origen, descripción). Los archivos que aún no existen se listan como PENDIENTES en el manifiesto.
CATALOGO = [
    ("01_dataset", os.path.join(D, "resumen_datos.csv"), "Composición del dataset construido: archivos por split (train/test/challenge), tipo PE y clase, con rango de semanas."),
    ("01_dataset", os.path.join(D, "resumen_datos.json"), "Igual que el CSV, con totales, fracciones de muestreo y número de características."),
    ("01_dataset", os.path.join(D, "distribucion_por_semana.csv"), "Número de archivos por semana y clase en cada split (cobertura temporal del muestreo)."),
    ("01_dataset", os.path.join(D, "diccionario_grupos.csv"), "Diccionario de datos por grupo: los 12 grupos del vector EMBER2024, n.º de variables y descripción."),
    ("01_dataset", os.path.join(D, "diccionario_caracteristicas.csv"), "Diccionario completo: nombre legible y grupo de cada una de las 2.568 posiciones del vector."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "fase1_resumen.json"), "KPIs de la Fase 1: tamaño de la muestra, limpieza, F1 de referencia, K elegido, % de reducción, cumplimiento de metas."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "muestra_seleccion_sha256.csv.gz"), "SHA-256, clase y partición (entrenamiento/validación) de cada archivo de la muestra usada para seleccionar características."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "columnas_constantes.csv"), "Variables eliminadas por ser constantes en la muestra."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "importancia_extratrees_completa.csv"), "Ranking completo de importancia (Extra Trees, Gini) de todas las variables limpias, con nombre y grupo."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "ranking_caracteristicas.json"), "Ranking del top preliminar, candidatos tras el filtro de correlación y F1 de referencia."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "redundantes_descartadas.csv"), "Variables descartadas por redundancia (|r| > umbral): con cuál estaban correlacionadas y el valor de |r|."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "curva_k_vs_f1.csv"), "Curva K vs. F1: F1, % de reducción y % del F1 de referencia conservado para cada K probado."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "seleccion_final.json"), "Regla aplicada y lista de las K columnas seleccionadas (con nombres y grupos)."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "caracteristicas_seleccionadas.csv"), "Tabla de las K características seleccionadas: posición, columna, nombre legible, grupo, importancia y puesto en el ranking."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "grupos_en_seleccion.csv"), "Cuántas características aporta cada grupo del vector al subconjunto seleccionado."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "correlacion_seleccionadas.csv"), "Matriz de correlación de Pearson entre las K características seleccionadas."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "train_reducido.parquet"), "BASE DE DATOS REDUCIDA de entrenamiento: K características seleccionadas + metadatos (sha256, label, tipo, fecha, semana, familia, comportamiento)."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "test_reducido.parquet"), "BASE DE DATOS REDUCIDA de prueba temporal (semanas 53-64): mismas columnas."),
    ("02_seleccion_caracteristicas", os.path.join(F1, "challenge_reducido.parquet"), "BASE DE DATOS REDUCIDA del conjunto de desafío (malware evasivo): mismas columnas."),
    ("03_modelos", os.path.join(F2, "fase2_resumen.json"), "Resumen completo de la Fase 2: métricas de cada modelo, hiperparámetros, modelo seleccionado, criterio y cumplimiento de metas."),
    ("03_modelos", os.path.join(F2, "busqueda_hiperparametros_random_forest.csv"), "Búsqueda aleatoria de hiperparámetros de Random Forest: combinaciones probadas, F1 medio/desv. en validación cruzada y tiempo."),
    ("03_modelos", os.path.join(F2, "busqueda_hiperparametros_xgboost.csv"), "Búsqueda aleatoria de hiperparámetros de XGBoost: combinaciones probadas, F1 medio/desv. en validación cruzada y tiempo."),
    ("03_modelos", os.path.join(F2, "historia_entrenamiento_dnn.csv"), "Historia de entrenamiento de la red neuronal por época (pérdida, AUC y accuracy en entrenamiento y validación)."),
    ("03_modelos", os.path.join(F2, "fase2_comparacion_modelos.csv"), "Tabla comparativa de modelos (métricas principales sobre la prueba temporal y el desafío)."),
    ("03_modelos", os.path.join(F2, "fase2_comparacion_modelos_completa.csv"), "Tabla comparativa con TODAS las métricas escalares (incluye conteos de la matriz de confusión, PR-AUC, latencias, tamaño del modelo)."),
    ("03_modelos", os.path.join(F2, "matrices_confusion.csv"), "MATRIZ DE CONFUSIÓN (validación sobre la prueba temporal) de cada modelo: TN, FP, FN, TP, FPR, FNR y accuracy."),
    ("03_modelos", os.path.join(F2, "reportes_clasificacion.csv"), "Reporte de clasificación por clase (precision, recall, F1, soporte) y promedios macro/ponderado de cada modelo."),
    ("03_modelos", os.path.join(F2, "curvas_roc.csv"), "Puntos (FPR, TPR, umbral) de la curva ROC de cada modelo."),
    ("03_modelos", os.path.join(F2, "curvas_precision_recall.csv"), "Puntos (recall, precisión) de la curva precisión-recall de cada modelo."),
    ("03_modelos", os.path.join(F2, "metricas_vs_umbral.csv"), "Precision, recall, F1, FPR, FNR y recall en el desafío para umbrales de decisión de 0,1 a 0,9."),
    ("03_modelos", os.path.join(F2, "metricas_por_tipo.csv"), "Métricas por tipo de archivo (Win32, Win64, .NET) para cada modelo, con el número de archivos de cada grupo."),
    ("03_modelos", os.path.join(F2, "metricas_por_semana.csv"), "F1, recall y FPR por semana del conjunto de prueba (deriva temporal), con el número de archivos por semana."),
    ("03_modelos", os.path.join(F2, "predicciones_test.parquet"), "Predicción por archivo de la prueba temporal: sha256, tipo, semana, clase real y probabilidad de malware de cada modelo."),
    ("03_modelos", os.path.join(F2, "predicciones_challenge.csv"), "Predicción por archivo del conjunto de desafío: sha256, tipo y probabilidad de malware de cada modelo."),
    ("03_modelos", os.path.join(M, "modelo_seleccionado.json"), "Modelo seleccionado, criterio de selección, umbral y columnas usadas."),
    ("03_modelos", os.path.join(M, "random_forest_info.json"), "Random Forest: mejores hiperparámetros, F1 de validación cruzada, tiempo de entrenamiento, tamaño y n.º de nodos."),
    ("03_modelos", os.path.join(M, "xgboost_info.json"), "XGBoost: mejores hiperparámetros, F1 de validación cruzada, tiempo de entrenamiento y tamaño."),
    ("03_modelos", os.path.join(M, "dnn_info.json"), "Red neuronal: arquitectura, épocas efectivas, tiempo de entrenamiento y tamaño."),
    ("03_modelos", os.path.join(M, "scaler.joblib"), "StandardScaler ajustado con el entrenamiento (necesario para reproducir las predicciones)."),
    ("04_explicabilidad", os.path.join(F3, "fase3_resumen.json"), "Indicadores de explicabilidad: cobertura, fidelidad de SHAP, acuerdo SHAP-LIME, tiempos, top-10 global y ejemplos explicados."),
    ("04_explicabilidad", os.path.join(F3, "shap_importancia_global.csv"), "Importancia global SHAP (media de |SHAP| y SHAP medio) de cada característica seleccionada, con nombre y grupo."),
    ("04_explicabilidad", os.path.join(F3, "shap_importancia_por_grupo.csv"), "Aporte de cada grupo de características a las decisiones del modelo (suma de |SHAP| medio)."),
    ("04_explicabilidad", os.path.join(F3, "shap_valores_muestra.parquet"), "Valores SHAP de cada archivo de la muestra explicada (una columna por característica) + sha256, clase y probabilidad."),
    ("04_explicabilidad", os.path.join(F3, "muestra_explicada.csv"), "Muestra explicada archivo por archivo: clase real, probabilidad, veredicto, nivel de riesgo y explicación en lenguaje natural."),
    ("04_explicabilidad", os.path.join(F3, "acuerdo_shap_lime_por_instancia.csv"), "Acuerdo SHAP-LIME por archivo: los 5 factores de cada técnica, coincidencias, índice de Jaccard y tiempo de LIME."),
    ("04_explicabilidad", os.path.join(F3, "explicaciones_ejemplos.txt"), "Explicaciones en lenguaje natural de los ejemplos malicioso y benigno usados en las figuras."),
    ("04_explicabilidad", os.path.join(F3, "lime_ejemplo_malicioso.html"), "Explicación LIME interactiva (HTML) del ejemplo malicioso."),
    ("04_explicabilidad", os.path.join(M, "xai_config.joblib"), "Configuración de explicabilidad usada por la interfaz (plantillas por grupo, valor base, escala)."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "app_malware.py"), "Código fuente de la aplicación (motor AnalizadorMalware + interfaz Gradio)."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "ejemplos_demo.parquet"), "Archivos de ejemplo cargados en la interfaz (vectores reducidos, etiqueta real, tipo y sha256)."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "ejemplo_resultado_analisis.json"), "Respuesta completa del motor de análisis para un ejemplo (veredicto, probabilidad, riesgo, factores y frases)."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "pruebas_funcionales.csv"), "Pruebas funcionales PF-01 a PF-09: descripción, resultado PASA/FALLA y detalle."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "pruebas_funcionales_detalle.json"), "Respuesta completa del analizador en cada prueba funcional."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "archivos_de_prueba.csv"), "Identificación (tamaño y SHA-256) de los ejecutables sintéticos usados en las pruebas."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "pruebas_rendimiento.csv"), "Latencia de extremo a extremo por tamaño de archivo: p50, p95, máximo, desglose extracción/predicción y cumplimiento de la meta."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "pruebas_rendimiento_detalle.csv"), "Todas las mediciones individuales de las pruebas de rendimiento (una fila por repetición)."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "fase4_pruebas.json"), "Resumen JSON de las pruebas funcionales y de rendimiento."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "cuestionario_SUS.txt"), "Cuestionario SUS aplicado a los participantes."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "sus_respuestas.csv"), "Respuestas SUS registradas (plantilla; se diligencia con participantes reales)."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "sus_puntajes_por_participante.csv"), "Puntaje SUS calculado por participante (solo existe cuando hay respuestas reales)."),
    ("05_interfaz_y_pruebas", os.path.join(F4, "fase4_sus.json"), "Resumen SUS: promedio, desviación, IC 95 %, calificación y cumplimiento de la meta (o estado pendiente)."),
    ("05_interfaz_y_pruebas", os.path.join(RUTAS["figuras"], "fig_interfaz_gradio.png"), "Captura de pantalla de la interfaz (la toma el usuario al lanzar la app)."),
    ("00_resumen", os.path.join(BASE_DIR, "resultados_para_documento.json"), "Consolidado de los resúmenes de todas las fases + configuración usada."),
    ("00_resumen", os.path.join(BASE_DIR, "cuadro_cumplimiento_metas.csv"), "Cuadro de cumplimiento de las metas del anteproyecto."),
    ("00_resumen", os.path.join(BASE_DIR, "registro_memoria.csv"), "Registro del uso de RAM del proceso en cada punto de control (evidencia de que la corrida cabe en Colab)."),
]
# Modelos serializados: entran al zip solo si pesan menos del límite (si no, quedan en Drive/modelos y se anota en el manifiesto)
for archivo_m, desc_m in [("random_forest.joblib", "Modelo Random Forest entrenado (joblib comprimido)."),
                          ("xgboost.joblib", "Modelo XGBoost entrenado (joblib comprimido)."), ("dnn.keras", "Red neuronal densa entrenada (Keras).")]:
    CATALOGO.append(("03_modelos", os.path.join(M, archivo_m), desc_m))

filas_manifiesto, omitidos_por_tamano = [], []
for carpeta, origen, descripcion in CATALOGO:
    destino_dir = os.path.join(EVID, carpeta)
    os.makedirs(destino_dir, exist_ok=True)
    nombre = os.path.basename(origen)
    if os.path.exists(origen):
        mb = os.path.getsize(origen) / 2 ** 20
        if nombre.endswith((".joblib", ".keras")) and nombre != "scaler.joblib" and nombre != "xai_config.joblib" and mb > CFG["LIMITE_MB_MODELO_EN_ZIP"]:
            omitidos_por_tamano.append((carpeta, nombre, mb))
            filas_manifiesto.append({"carpeta": carpeta, "archivo": nombre, "estado": f"EN DRIVE (no se copia: {mb:.0f} MB > límite)",
                                     "tamano_mb": round(mb, 2), "descripcion": descripcion, "origen": origen})
            continue
        shutil.copy(origen, os.path.join(destino_dir, nombre))
        filas_manifiesto.append({"carpeta": carpeta, "archivo": nombre, "estado": "presente", "tamano_mb": round(mb, 3),
                                 "descripcion": descripcion, "origen": origen})
    else:
        filas_manifiesto.append({"carpeta": carpeta, "archivo": nombre, "estado": "PENDIENTE (aún no generado)", "tamano_mb": None,
                                 "descripcion": descripcion, "origen": origen})

# Muestras legibles (CSV) de los datasets reducidos, escritas por bloques; opcionalmente el CSV completo comprimido
for split in ["train", "test", "challenge"]:
    origen = os.path.join(F1, f"{split}_reducido.parquet")
    if not os.path.exists(origen):
        continue
    pf = pq.ParquetFile(origen)
    ruta_muestra = os.path.join(EVID, "02_seleccion_caracteristicas", f"{split}_reducido_muestra_{CFG['FILAS_MUESTRA_CSV']}filas.csv")
    escritas = 0
    with open(ruta_muestra, "w", encoding="utf-8") as f_csv:
        for lote in pf.iter_batches(batch_size=min(CFG["FILAS_MUESTRA_CSV"], int(CFG["LOTE_FILAS_LECTURA"]))):
            df_l = lote.to_pandas().head(CFG["FILAS_MUESTRA_CSV"] - escritas)
            df_l.to_csv(f_csv, index=False, header=(escritas == 0), lineterminator="\n")
            escritas += len(df_l)
            if escritas >= CFG["FILAS_MUESTRA_CSV"]:
                break
    filas_manifiesto.append({"carpeta": "02_seleccion_caracteristicas", "archivo": os.path.basename(ruta_muestra), "estado": "presente",
                             "tamano_mb": round(os.path.getsize(ruta_muestra) / 2 ** 20, 3),
                             "descripcion": f"Primeras {escritas} filas de {split}_reducido.parquet en CSV (para abrir en Excel); el parquet tiene {pf.metadata.num_rows:,} filas.",
                             "origen": origen})
    if CFG.get("EXPORTAR_CSV_COMPLETO"):
        ruta_gz = os.path.join(EVID, "02_seleccion_caracteristicas", f"{split}_reducido_completo.csv.gz")
        import gzip
        with gzip.open(ruta_gz, "wt", encoding="utf-8") as f_gz:
            for i, lote in enumerate(pf.iter_batches(batch_size=int(CFG["LOTE_FILAS_LECTURA"]))):
                lote.to_pandas().to_csv(f_gz, index=False, header=(i == 0), lineterminator="\n")
        filas_manifiesto.append({"carpeta": "02_seleccion_caracteristicas", "archivo": os.path.basename(ruta_gz), "estado": "presente",
                                 "tamano_mb": round(os.path.getsize(ruta_gz) / 2 ** 20, 3),
                                 "descripcion": f"{split}_reducido completo en CSV comprimido ({pf.metadata.num_rows:,} filas).", "origen": origen})
    del pf

# Todas las figuras
os.makedirs(os.path.join(EVID, "06_figuras"), exist_ok=True)
DESCRIPCION_FIGURAS = {
    "fig_composicion_dataset.png": "Composición del subconjunto PE de EMBER2024 por split, tipo y clase.",
    "fig_curva_k_vs_f1.png": "Curva K vs. % del F1 de referencia conservado, con la meta y el K seleccionado.",
    "fig_importancia_topk.png": "Importancia (Extra Trees) de las K características seleccionadas, coloreadas por grupo.",
    "fig_correlacion_seleccionadas.png": "Mapa de calor de la correlación entre las K características seleccionadas.",
    "fig_comparacion_modelos.png": "Comparación de F1, recall, precisión, AUC-ROC y recall en el desafío de los tres modelos.",
    "fig_roc_modelos.png": "Curvas ROC (escala lineal y logarítmica) de los tres modelos.",
    "fig_pr_modelos.png": "Curvas precisión-recall de los tres modelos.",
    "fig_matrices_confusion.png": "Matrices de confusión de los tres modelos sobre la prueba temporal.",
    "fig_f1_por_semana.png": "Deriva temporal: F1 por semana del conjunto de prueba.",
    "fig_desempeno_por_tipo.png": "F1 por tipo de archivo y recall por tipo en el conjunto de desafío.",
    "fig_tiempos_modelos.png": "Tiempo de entrenamiento y latencia de predicción por modelo.",
    "fig_busqueda_hiperparametros.png": "F1 de validación cruzada de cada combinación probada en la búsqueda de hiperparámetros.",
    "fig_dnn_historia.png": "Pérdida y AUC por época de la red neuronal (entrenamiento vs. validación).",
    "fig_shap_global_barras.png": "Importancia global SHAP (media de |SHAP|) del modelo seleccionado.",
    "fig_shap_por_grupo.png": "Aporte de cada grupo de características a las decisiones (SHAP).",
    "fig_shap_beeswarm.png": "Resumen SHAP (beeswarm): dirección y magnitud del efecto de cada característica.",
    "fig_shap_local_malicioso.png": "Explicación local SHAP (waterfall) de un archivo malicioso.",
    "fig_shap_local_benigno.png": "Explicación local SHAP (waterfall) de un archivo benigno.",
    "fig_lime_ejemplo.png": "Explicación LIME del mismo archivo malicioso (contraste con SHAP).",
    "fig_acuerdo_shap_lime.png": "Distribución del índice de Jaccard entre los top-5 de SHAP y LIME.",
    "fig_latencia_por_tamano.png": "Latencia de análisis por tamaño de archivo (desglose y p95 frente a la meta).",
    "fig_sus_resultados.png": "Puntajes SUS por participante y promedio por ítem (con respuestas reales).",
    "fig_interfaz_gradio.png": "Captura de pantalla de la interfaz Gradio.",
}
for f in sorted(os.listdir(RUTAS["figuras"])):
    if f.endswith(".png"):
        shutil.copy(os.path.join(RUTAS["figuras"], f), os.path.join(EVID, "06_figuras", f))
        filas_manifiesto.append({"carpeta": "06_figuras", "archivo": f, "estado": "presente",
                                 "tamano_mb": round(os.path.getsize(os.path.join(RUTAS["figuras"], f)) / 2 ** 20, 3),
                                 "descripcion": DESCRIPCION_FIGURAS.get(f, "Figura generada por el notebook."), "origen": os.path.join(RUTAS["figuras"], f)})
for f, desc in DESCRIPCION_FIGURAS.items():
    if not os.path.exists(os.path.join(RUTAS["figuras"], f)):
        filas_manifiesto.append({"carpeta": "06_figuras", "archivo": f, "estado": "PENDIENTE (aún no generada)", "tamano_mb": None,
                                 "descripcion": desc, "origen": os.path.join(RUTAS["figuras"], f)})

# Manifiesto (CSV + Markdown legible)
manifiesto = pd.DataFrame(filas_manifiesto).sort_values(["carpeta", "archivo"]).reset_index(drop=True)
manifiesto.to_csv(os.path.join(EVID, "00_indice_evidencias.csv"), index=False)
n_presentes = int((manifiesto["estado"] == "presente").sum())
pendientes = manifiesto[manifiesto["estado"].str.startswith("PENDIENTE")]
with open(os.path.join(EVID, "00_MANIFIESTO_EVIDENCIAS.md"), "w", encoding="utf-8") as f_md:
    f_md.write("# Manifiesto de evidencias — Trabajo de grado: detección de malware con ML y XAI (EMBER2024)\n\n")
    f_md.write(f"Generado: {time.strftime('%Y-%m-%d %H:%M:%S')} · Entorno: {'Google Colab' if EN_COLAB else 'local'} · Carpeta base: `{BASE_DIR}`\n\n")
    f_md.write(f"Archivos presentes: **{n_presentes}** · pendientes: **{len(pendientes)}** · omitidos del zip por tamaño: **{len(omitidos_por_tamano)}**\n\n")
    f_md.write("Resumen de la corrida: " + json.dumps({
        "K_seleccionado": f1r.get("K"), "pct_reduccion": f1r.get("pct_reduccion"), "modelo_seleccionado": sel,
        "f1_prueba_temporal": r_sel.get("f1"), "fpr": r_sel.get("fpr"), "recall_challenge": r_sel.get("recall_challenge"),
        "n_train": f2r.get("n_train"), "n_test": f2r.get("n_test"), "n_challenge": f2r.get("n_challenge")}, ensure_ascii=False, default=str) + "\n\n")
    f_md.write("Cómo abrir los `.parquet`: `import pandas as pd; df = pd.read_parquet('archivo.parquet')` (o `pip install pyarrow`). "
               "Las columnas `f###` son posiciones del vector EMBER2024; su nombre legible está en "
               "`01_dataset/diccionario_caracteristicas.csv` y, para las seleccionadas, en `02_seleccion_caracteristicas/caracteristicas_seleccionadas.csv`.\n\n")
    for carpeta, grupo in manifiesto.groupby("carpeta", sort=True):
        f_md.write(f"## {carpeta}\n\n| Archivo | Estado | Tamaño (MB) | Descripción |\n|---|---|---|---|\n")
        for _, fila in grupo.iterrows():
            tam = "" if pd.isna(fila["tamano_mb"]) else f"{fila['tamano_mb']:.3f}"
            f_md.write(f"| `{fila['archivo']}` | {fila['estado']} | {tam} | {fila['descripcion']} |\n")
        f_md.write("\n")
    if len(pendientes):
        f_md.write("## Pendientes\n\n" + "\n".join(f"- `{c}/{a}`: {d}" for c, a, d in zip(pendientes["carpeta"], pendientes["archivo"], pendientes["descripcion"])) + "\n")
    if omitidos_por_tamano:
        f_md.write("\n## Modelos que quedaron solo en Google Drive (superan el límite de tamaño del zip)\n\n" +
                   "\n".join(f"- `{n}` ({mb:.0f} MB) en `{M}`" for _, n, mb in omitidos_por_tamano) + "\n")
shutil.copy(os.path.join(EVID, "00_MANIFIESTO_EVIDENCIAS.md"), carpeta_fig)

zip_evidencias = shutil.make_archive(os.path.join(BASE_DIR, "evidencias_proyecto"), "zip", EVID)
print(f"\nPaquete completo de evidencias: {zip_evidencias} ({os.path.getsize(zip_evidencias) / 2 ** 20:.1f} MB) | "
      f"{n_presentes} archivos presentes, {len(pendientes)} pendientes")
print(manifiesto.groupby("carpeta")["estado"].value_counts().to_string())
if len(pendientes):
    print("\nEvidencias pendientes (se generan al ejecutar la fase correspondiente o las aporta el usuario):")
    for c, a in zip(pendientes["carpeta"], pendientes["archivo"]):
        print(f"  - {c}/{a}")

# ----------------------------------------------------------------------------
# 4) Descarga al computador (solo en Colab; los archivos también quedan en Google Drive)
# ----------------------------------------------------------------------------
if EN_COLAB:
    try:
        from google.colab import files as _files
        for _z in [zip_documento, zip_evidencias]:
            print("Descargando", os.path.basename(_z), "...")
            _files.download(_z)
    except Exception as e:  # noqa: BLE001
        print("No se pudo iniciar la descarga automática (", e, "). Descarga los zip desde Google Drive:", BASE_DIR)
else:
    print("Fuera de Colab: los paquetes quedan en", BASE_DIR)
memoria_ram("exportación fin")


## Sincronización con GitHub
Copia código, resúmenes, tablas y figuras (no los datasets ni los modelos pesados) al repositorio y hace commit + push pidiendo el token en el momento con `getpass` (nunca queda escrito en el notebook).

In [ ]:
"""
SINCRONIZACIÓN CON EL REPOSITORIO DE GITHUB

Copia al repositorio el código de la aplicación, los resúmenes, tablas y figuras
(NO los datasets ni los modelos pesados) y hace commit + push.

SEGURIDAD: el token de GitHub se pide en el momento con getpass y nunca queda
escrito en el notebook ni en el repositorio. (La versión anterior del notebook
tenía un token en texto plano: ese token debe revocarse en GitHub -> Settings ->
Developer settings -> Personal access tokens.)
"""
import shutil
import getpass
import subprocess

REPO_URL = "https://github.com/jhgar059/repo_proyecto.git"
REPO_DIR = os.path.join(LOCAL_DIR, "repo_proyecto")

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

# Se suben las evidencias ligeras (código, JSON, CSV, HTML y figuras). Quedan fuera los parquet y los modelos serializados
# (pesan cientos de MB o GB): esos se conservan en Google Drive y en evidencias_proyecto.zip.
def ligeros(carpeta):
    return [os.path.join(carpeta, f) for f in sorted(os.listdir(carpeta))
            if f.endswith((".json", ".csv", ".csv.gz", ".txt", ".html", ".py", ".md")) and not f.endswith(".ok")]

destinos = {
    "datos": ligeros(RUTAS["datos"]),
    "fase1_seleccion_caracteristicas": ligeros(RUTAS["fase1"]),
    "fase2_entrenamiento_modelos": ligeros(RUTAS["fase2"]) + [os.path.join(RUTAS["modelos"], f) for f in ["modelo_seleccionado.json", "random_forest_info.json", "xgboost_info.json", "dnn_info.json"]],
    "fase3_explicabilidad": ligeros(RUTAS["fase3"]),
    "fase4_interfaz": [a for a in ligeros(RUTAS["fase4"]) if os.path.basename(a) != "sus_respuestas.csv"],  # respuestas de participantes: no se publican
    "figuras": [os.path.join(RUTAS["figuras"], f) for f in os.listdir(RUTAS["figuras"]) if f.endswith(".png")],
    "evidencias": [os.path.join(RUTAS["evidencias"], f) for f in ["00_MANIFIESTO_EVIDENCIAS.md", "00_indice_evidencias.csv"]],
}
for carpeta, archivos in destinos.items():
    os.makedirs(os.path.join(REPO_DIR, carpeta), exist_ok=True)
    for a in archivos:
        if os.path.exists(a):
            shutil.copy(a, os.path.join(REPO_DIR, carpeta, os.path.basename(a)))
for a in ["resultados_para_documento.json", "cuadro_cumplimiento_metas.csv", "registro_memoria.csv"]:
    if os.path.exists(os.path.join(BASE_DIR, a)):
        shutil.copy(os.path.join(BASE_DIR, a), REPO_DIR)
print("Archivos copiados al repositorio local.")

if input("¿Hacer commit y push ahora? (s/n): ").strip().lower() == "s":
    usuario = input("Usuario de GitHub: ").strip()
    correo = input("Correo para el commit: ").strip()
    token = getpass.getpass("Token de acceso personal (no se muestra ni se guarda): ").strip()
    subprocess.run(["git", "-C", REPO_DIR, "config", "user.name", usuario], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "config", "user.email", correo], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "add", "."], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "commit", "-m", "Resultados EMBER2024: fases 1-4 (código, resúmenes y figuras)"], check=False)
    url_token = REPO_URL.replace("https://", f"https://{usuario}:{token}@")
    r = subprocess.run(["git", "-C", REPO_DIR, "push", url_token, "HEAD"], capture_output=True, text=True)
    print("Push OK" if r.returncode == 0 else "Error en push:\n" + r.stderr.replace(token, "***"))
    del token, url_token


## Guía para trasladar los resultados al documento
| Elemento del documento | Archivo generado (en `evidencias/` o en la carpeta de la fase) |
|---|---|
| Tabla de composición del dataset (cap. 1) | `datos/resumen_datos.csv` · `datos/distribucion_por_semana.csv` · `fig_composicion_dataset.png` |
| Diccionario de datos (cap. 1) | `datos/diccionario_grupos.csv` · `datos/diccionario_caracteristicas.csv` |
| KPIs de la Fase 1, curva K vs. F1, características seleccionadas, base de datos reducida | `fase1/fase1_resumen.json` · `fase1/curva_k_vs_f1.csv` · `fase1/caracteristicas_seleccionadas.csv` · `fase1/importancia_extratrees_completa.csv` · `fase1/redundantes_descartadas.csv` · `fase1/train/test/challenge_reducido.parquet` (+ muestras CSV) · `fig_curva_k_vs_f1.png` · `fig_importancia_topk.png` · `fig_correlacion_seleccionadas.png` |
| Búsqueda de hiperparámetros y entrenamiento | `fase2/busqueda_hiperparametros_*.csv` · `fase2/historia_entrenamiento_dnn.csv` · `modelos/*_info.json` · `fig_busqueda_hiperparametros.png` · `fig_dnn_historia.png` |
| Tabla comparativa de modelos, matrices de confusión, ROC/PR, deriva temporal | `fase2/fase2_comparacion_modelos(_completa).csv` · `fase2/matrices_confusion.csv` · `fase2/reportes_clasificacion.csv` · `fase2/curvas_roc.csv` · `fase2/curvas_precision_recall.csv` · `fase2/metricas_vs_umbral.csv` · `fase2/metricas_por_tipo.csv` · `fase2/metricas_por_semana.csv` · `fase2/predicciones_test.parquet` · `fig_comparacion_modelos.png` · `fig_roc_modelos.png` · `fig_pr_modelos.png` · `fig_matrices_confusion.png` · `fig_f1_por_semana.png` · `fig_desempeno_por_tipo.png` · `fig_tiempos_modelos.png` |
| Explicabilidad (cap. 3) | `fase3/fase3_resumen.json` · `fase3/shap_importancia_global.csv` · `fase3/shap_importancia_por_grupo.csv` · `fase3/shap_valores_muestra.parquet` · `fase3/muestra_explicada.csv` · `fase3/acuerdo_shap_lime_por_instancia.csv` · `fig_shap_global_barras.png` · `fig_shap_por_grupo.png` · `fig_shap_beeswarm.png` · `fig_shap_local_*.png` · `fig_lime_ejemplo.png` · `fig_acuerdo_shap_lime.png` |
| Pruebas funcionales, rendimiento y SUS (cap. 3) | `fase4/pruebas_funcionales.csv` · `fase4/pruebas_funcionales_detalle.json` · `fase4/pruebas_rendimiento(_detalle).csv` · `fase4/fase4_pruebas.json` · `fig_latencia_por_tamano.png` · `fase4/fase4_sus.json` · `fig_sus_resultados.png` · captura `fig_interfaz_gradio.png` |
| Cuadro de cumplimiento de metas (conclusiones) | `cuadro_cumplimiento_metas.csv` (también en `figuras_documento/`) |
| Uso de memoria de la corrida (anexo) | `registro_memoria.csv` |
